# V-Energy Grid 360 — Camada Gold (DE-04)

## 1. Visão Geral e Arquitetura da Camada Gold
A camada **Gold** representa o topo da arquitetura Medalhão. Sua responsabilidade é consolidar, agregar e enriquecer os dados limpos vindos da camada **Silver**, transformando tabelas relacionais em **visões analíticas de negócio**.

### Objetivos do Negócio Atendidos:
1. **Diagnóstico de Receita e Tarifa:** Responder à diretoria sobre a variação do volume faturado (-4%) versus o reajuste tarifário (+10,5%), identificando o impacto do consumo, bandeiras e Geração Distribuída (GD).
2. **Qualidade do Serviço (DEC/FEC):** Calcular indicadores internos simplificados de continuidade por alimentador e mês, sem alegar apuração ou conformidade regulatória oficial.
3. **Fila de Inspeção e Risco:** Identificar potenciais perdas não técnicas (fraudes/gatos) cruzando dados de consumo, inspeções anteriores e chamados do call center.
4. **Visão 360° da Unidade Consumidora:** Fonte única de verdade para alimentar o backend do Portal Web e o Agente de IA.

---

### Histórias de Usuário Atendidas neste Notebook:
* **Modelagem de Métricas:** *"Como engenheiro de dados, quero modelar a camada Gold a partir das perguntas de negócio das lideranças..."*
* **Indicadores ANEEL:** *"Como engenheiro de dados, quero calcular DEC e FEC por alimentador conforme a definição interna simplificada documentada..."*
* **Testes de Qualidade:** *"Como engenheiro de dados, quero testes de qualidade automatizados no pipeline..."*
Correções locais de 30/10/2026: vistas Silver publicadas materializadas e conferidas por run/lote; preparo isolado; flags e ausências preservadas; reconciliações e cronologia corrigidas. Aceite Gold e exportação das quinze projeções analíticas e um cadastro restrito dependem de execução autorizada e integração da carga v2.

Complementos da UC 360: `gd_uc`, `tratativas_uc` e `gold_cadastro.cadastro_titular_uc`. Nome somente no cadastro restrito. Antes de executar, provisionar os schemas cadastrais de publicação/preparo e habilitar `CADASTRO_RESTRITO_VALIDADO=sim` após conferir permissões. Ausência dessa configuração bloqueia o processamento antes das projeções.


###  Leitura de Controle e Rastreabilidade 

**Objetivo:** Capturar os metadados de execução da camada Silver antes do processamento.  
**Impacto:** Permite associar cada carga da Gold ao `run_id` e carimbo de data/hora (`data_processamento`) exato do lote de origem, garantindo auditoria completa entre camadas.

###  Governança de Ambiente e Parâmetros de Execução (`AMBIENTE`)

**Objetivo:** Garantir a integridade das travas de qualidade e a rastreabilidade entre as camadas Silver e Gold, sincronizando a execução do notebook entre ambientes de desenvolvimento (`dev`) e produção (`prod`).

---

####  Orientações de Execução para a Equipe:

1. **Execução Automatizada em Produção (Databricks Workflows / Job Oficial):**
   * O orquestrador injeta automaticamente o parâmetro `AMBIENTE = prod` no runtime do Job.
   * **Garantia de Produção:** O pipeline valida se a camada Silver possui o status `'CONCLUIDO'`, bloqueia execuções com dados sintéticos e registra a execução como oficial de auditoria na tabela `gold.controle_gold`.

2. **Execução Interativa Manual (Desenvolvimento e Validação no Notebook):**
   * Devido à persistência do estado dos widgets na interface visual do Databricks, verifique o widget **`Ambiente de Execução (AMBIENTE)`** no canto superior esquerdo do notebook.
   * **Para homologação manual com comportamento de Produção:** Certifique-se de que a caixinha do widget exibe `prod` antes de rodar todas as células.
   * **Caso a tela exiba `dev`:** Altere manualmente o valor da caixa para `prod`, aperte *Enter* e reexecute a **Seção 0**.

3. **Registro Oficial de Governança:**
   * Execuções manuais geram histórico de desenvolvimento. O carimbo oficial de homologação dos dados para consumo corporativo (Aplicações / BI) é o registrado pelas execuções do **Job Automatizado de Produção**.

In [ ]:
# SEÇÃO 0 — CONFIGURAÇÃO, GOVERNANÇA E FUNÇÕES AUXILIARES

from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F


CATALOG = "workspace"
SCHEMA_SILVER_DADOS = "silver_publicada"
SCHEMA_SILVER_CONTROLE = "silver"
SCHEMA_GOLD = "gold"

# AMBIENTE deve vir dos parâmetros do Databricks Job/Workflow.
# Não atribuir "dev" ou "prod" manualmente em outra parte do notebook.
AMBIENTE = dbutils.widgets.get("AMBIENTE").strip().lower()

if AMBIENTE not in {"dev", "test", "prod"}:
    raise ValueError(
        f"AMBIENTE inválido: {AMBIENTE!r}. Use dev, test ou prod."
    )

PERMITIR_RUN_SINTETICO = AMBIENTE in {"dev", "test"}

# Alias para células anteriores que usam SCHEMA_SILVER.
SCHEMA_SILVER = SCHEMA_SILVER_DADOS

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")

print(f"[GOVERNANÇA] Ambiente recebido do Job: {AMBIENTE}")
print(f"[GOVERNANÇA] Permite Silver sintética: {PERMITIR_RUN_SINTETICO}")
print(f"Silver Dados:    {CATALOG}.{SCHEMA_SILVER_DADOS}")
print(f"Silver Controle: {CATALOG}.{SCHEMA_SILVER_CONTROLE}")
print(f"Gold:            {CATALOG}.{SCHEMA_GOLD}")


def assert_columns_exist(df, columns, name):
    """Falha com mensagem clara quando faltam colunas obrigatórias."""
    missing = sorted(set(columns) - set(df.columns))

    if missing:
        raise ValueError(f"{name}: colunas ausentes: {missing}")


def assert_no_nulls(df, columns, name):
    """Falha se houver nulos nas colunas informadas."""
    condition = None

    for column in columns:
        part = F.col(column).isNull()
        condition = part if condition is None else (condition | part)

    if condition is not None and df.filter(condition).limit(1).count() > 0:
        raise ValueError(
            f"{name}: há valores nulos nas colunas obrigatórias {columns}."
        )


def assert_unique(df, columns, name):
    """Falha se a chave informada não for única."""
    duplicates = (
        df.groupBy(*columns)
          .count()
          .filter(F.col("count") > 1)
          .limit(1)
          .count()
    )

    if duplicates > 0:
        raise ValueError(
            f"{name}: a chave {columns} não é única. "
            "Revise a granularidade da fonte antes de continuar."
        )


# A1 — Validação cruzada entre controle_publicacao e controle_silver


TB_PUBLICACAO = (
    f"{CATALOG}.{SCHEMA_SILVER_CONTROLE}.controle_publicacao"
)
TB_SILVER = (
    f"{CATALOG}.{SCHEMA_SILVER_CONTROLE}.controle_silver"
)

# Inicializar as variáveis para que seus nomes existam em ambos os caminhos.
CONTROL_TABLE_PUBLICACAO = TB_PUBLICACAO
CONTROL_TABLE_SILVER = TB_SILVER
status_exec = None
silver_lineage_status = None

tabelas_controle_existentes = (
    spark.catalog.tableExists(TB_PUBLICACAO)
    and spark.catalog.tableExists(TB_SILVER)
)

if not tabelas_controle_existentes:
    if not PERMITIR_RUN_SINTETICO:
        raise RuntimeError(
            "Falha crítica de governança: são necessárias as tabelas "
            f"{TB_PUBLICACAO} e {TB_SILVER}. "
            "A execução Gold foi interrompida."
        )

    agora_utc = datetime.now(timezone.utc)

    run_id_silver = (
        f"DEV_SYNTHETIC_{agora_utc.strftime('%Y%m%d_%H%M%S')}_"
        f"{uuid4().hex[:8]}"
    )
    lote_bronze_silver = "DEV_LOTE"
    data_processamento_silver = agora_utc

    status_exec = "SEM_CONTROLE"
    silver_lineage_status = "SINTETICO_SEM_CONTROLE_SILVER"

    print("[AVISO DE GOVERNANÇA] Tabelas físicas de controle não encontradas.")
    print(f"   • Ambiente: {AMBIENTE}")
    print(f"   • Run ID sintético: {run_id_silver}")
    print(f"   • Lote sintético: {lote_bronze_silver}")
    print(f"   • Status Silver: {status_exec}")
    print(f"   • Linhagem: {silver_lineage_status}")
    print("   • O ID sintético não comprova a versão da Silver consumida.")

else:
    df_publicacao = spark.table(TB_PUBLICACAO)
    df_controle_silver = spark.table(TB_SILVER)

    # A publicação precisa informar o run e o lote ativos.
    assert_columns_exist(
        df_publicacao,
        ["chave", "run_ativo", "lote_ativo"],
        TB_PUBLICACAO
    )

    col_data_publicacao = next(
        (
            coluna
            for coluna in [
                "publicado_em",
                "data_processamento",
                "concluido_em"
            ]
            if coluna in df_publicacao.columns
        ),
        None
    )

    if col_data_publicacao is None:
        raise ValueError(
            f"{TB_PUBLICACAO}: não foi encontrada uma coluna de data "
            "entre publicado_em, data_processamento e concluido_em."
        )

    registros_publicacao = (
        df_publicacao.filter(F.col("chave") == "silver")
        .withColumn("_data_publicacao_ts", F.to_timestamp(F.col(col_data_publicacao)))
        .collect()
    )
    if len(registros_publicacao) != 1 or any(
        registros_publicacao[0][c] is None
        for c in ["run_ativo", "lote_ativo", "_data_publicacao_ts"]
    ):
        raise ValueError("A publicação Silver exige uma única referência válida")

    registro_publicacao = registros_publicacao[0]

    run_id_silver = registro_publicacao["run_ativo"]
    lote_bronze_silver = registro_publicacao["lote_ativo"]
    data_processamento_silver = registro_publicacao["_data_publicacao_ts"]

    # Reconhecer os nomes de coluna usados pela tabela de histórico.
    colunas_silver = set(df_controle_silver.columns)

    col_run_silver = next(
        (
            coluna
            for coluna in ["_run_id", "run_id", "run_ativo"]
            if coluna in colunas_silver
        ),
        None
    )

    col_lote_silver = next(
        (
            coluna
            for coluna in ["_lote_bronze", "lote_bronze", "lote_ativo"]
            if coluna in colunas_silver
        ),
        None
    )

    col_status_silver = next(
        (
            coluna
            for coluna in ["status", "situacao"]
            if coluna in colunas_silver
        ),
        None
    )

    colunas_obrigatorias_ausentes = [
        descricao
        for descricao, coluna in [
            ("coluna de run_id", col_run_silver),
            ("coluna de lote Bronze", col_lote_silver),
            ("coluna de status", col_status_silver)
        ]
        if coluna is None
    ]

    if colunas_obrigatorias_ausentes:
        raise ValueError(
            f"{TB_SILVER}: schema inválido; faltam "
            f"{', '.join(colunas_obrigatorias_ausentes)}."
        )

    # Procurar todos os registros que correspondem ao run e lote publicados.
    df_status_run_lote = (
        df_controle_silver
        .filter(
            (F.col(col_run_silver).cast("string") == str(run_id_silver))
            & (F.col(col_lote_silver).cast("string") == str(lote_bronze_silver))
        )
        .withColumn(
            "_status_norm",
            F.upper(F.trim(F.col(col_status_silver).cast("string")))
        )
    )

    resumo_status = (
        df_status_run_lote
        .agg(
            F.count(F.lit(1)).alias("qtd_registros"),
            F.sum(
                F.when(F.col("_status_norm").isNull(), 1).otherwise(0)
            ).alias("qtd_status_nulos"),
            F.collect_set("_status_norm").alias("status_encontrados")
        )
        .first()
    )

    qtd_registros = resumo_status["qtd_registros"] or 0
    qtd_status_nulos = resumo_status["qtd_status_nulos"] or 0
    status_encontrados = resumo_status["status_encontrados"] or []

    status_concluido = (
        qtd_registros == 1
        and qtd_status_nulos == 0
        and all(
            status == "CONCLUIDO"
            for status in status_encontrados
        )
    )

    # Esta variável é consumida pela célula final controle_gold.
    if status_concluido:
        status_exec = "CONCLUIDO"
    elif qtd_registros == 0:
        status_exec = "SEM_REGISTRO"
    elif qtd_status_nulos > 0:
        status_exec = "STATUS_NULO"
    else:
        status_exec = ", ".join(sorted(status_encontrados))

    if not status_concluido and not PERMITIR_RUN_SINTETICO:
        raise RuntimeError(
            "Falha de execução Silver: "
            f"run={run_id_silver}, "
            f"lote={lote_bronze_silver}, "
            f"status={status_exec!r}, "
            f"registros={qtd_registros}, "
            f"status nulos={qtd_status_nulos}. "
            "A Gold exige status CONCLUIDO."
        )

    if status_concluido:
        silver_lineage_status = "CONTROLE_FISICO"
        print("[CONTROLE SILVER] Validação cruzada concluída.")
    else:
        silver_lineage_status = "CONTROLE_FISICO_NAO_CONCLUIDO"
        print(
            "[AVISO DE GOVERNANÇA] Controle físico encontrado, "
            "mas o status não foi aprovado."
        )

    print(f"   • Publicação: {TB_PUBLICACAO}")
    print(f"   • Histórico: {TB_SILVER}")
    print(f"   • Run ID: {run_id_silver}")
    print(f"   • Lote: {lote_bronze_silver}")
    print(f"   • Processada em: {data_processamento_silver}")
    print(f"   • Status Silver: {status_exec}")
    print(f"   • Linhagem: {silver_lineage_status}")


# Função auxiliar para flag de Geração Distribuída

def get_distinct_uc_with_gd(df_gd):
    """
    Retorna uma linha por UC com GD.

    Regra atual: qualquer registro em geracao_distribuida indica GD.
    Se houver status ou vigência na fonte, aplique aqui a regra aprovada.
    """
    assert_columns_exist(
        df_gd,
        ["id_uc"],
        "geracao_distribuida"
    )

    return (
        df_gd
        .filter(F.col("id_uc").isNotNull())
        .select("id_uc")
        .dropDuplicates(["id_uc"])
        .withColumn("possui_gd", F.lit("Sim"))
    )

# Uma referência temporal única evita resultados diferentes entre células/reexecuções lazy.
from decimal import Decimal
import re
REFERENCIA_GOLD = datetime.now(timezone.utc)
spark.conf.set("spark.sql.session.timeZone", "UTC")

def instante_referencia():
    return F.lit(REFERENCIA_GOLD)

def data_referencia():
    return instante_referencia().cast("date")

def uc_real():
    return F.col("id_uc").isNotNull() & ~F.col("id_uc").startswith("TESTE")

def nota_valida():
    return F.when(
        ~F.coalesce(F.col("_nota_invalida"), F.lit(False))
        & (F.col("nota_satisfacao") >= 0), F.col("nota_satisfacao")
    )

def duracao_valida():
    return F.when(
        ~F.coalesce(F.col("_duracao_incompativel"), F.lit(False))
        & (F.col("duracao_min") >= 0), F.col("duracao_min").cast("double")
    )

_fontes_silver = {}
def read_silver(tabela):
    """Materializa somente a vista publicada, conferindo a identidade antes/depois.

    O checkpoint local corta a leitura lazy da vista mutável. Perda dos blocos
    interrompe o Job: refazer a execução inteira, sem reler outro lote silenciosamente.
    """
    if not re.fullmatch(r"[a-z_]+", tabela):
        raise ValueError("Nome de fonte Silver inválido")
    if tabela not in _fontes_silver:
        conferir_origem()
        nome = f"{CATALOG}.{SCHEMA_SILVER_DADOS}.{tabela}"
        df = spark.table(nome).localCheckpoint(eager=True)
        if silver_lineage_status == "CONTROLE_FISICO":
            assert_columns_exist(df, ["_run_id", "_lote_id"], nome)
            divergente = (~F.col("_run_id").eqNullSafe(F.lit(run_id_silver))
                          | ~F.col("_lote_id").eqNullSafe(F.lit(lote_bronze_silver)))
            if df.filter(divergente).limit(1).count():
                df.unpersist()
                raise ValueError("Vista Silver contém execução/lote incompatível")
        conferir_origem()
        _fontes_silver[tabela] = df
    return _fontes_silver[tabela]

def conferir_origem():
    if silver_lineage_status != "CONTROLE_FISICO":
        return
    rows = spark.table(TB_PUBLICACAO).filter(F.col("chave") == "silver").collect()
    if len(rows) != 1 or rows[0]["run_ativo"] != run_id_silver or rows[0]["lote_ativo"] != lote_bronze_silver:
        raise RuntimeError("A publicação Silver mudou durante o preparo Gold")
    rows = (spark.table(TB_SILVER)
            .filter((F.col(col_run_silver) == run_id_silver) & (F.col(col_lote_silver) == lote_bronze_silver))
            .collect())
    if len(rows) != 1 or str(rows[0][col_status_silver]).strip().upper() != "CONCLUIDO":
        raise RuntimeError("A execução Silver deixou de estar validada")


def promover_tabelas(tabelas, registrar_controle, controle):
    """Promove somente candidatos validados; recupera versões anteriores em falha capturada.

    Exige Job sem execuções concorrentes. Delta não oferece transação entre estas
    tabelas publicadas: leitores/exportador devem aguardar o fim do Job. Uma interrupção
    abrupta exige recuperação operacional antes de retomar o consumo.
    """
    conferir_origem()
    anteriores = {}
    tentadas = []
    versao_controle = (spark.sql(f"DESCRIBE HISTORY {controle} LIMIT 1").first()["version"]
                       if spark.catalog.tableExists(controle) else None)
    for nome in tabelas:
        destino = destino_gold(nome)
        anteriores[nome] = (spark.sql(f"DESCRIBE HISTORY {destino} LIMIT 1").first()["version"]
                            if spark.catalog.tableExists(destino) else None)
    try:
        for nome in tabelas:
            destino = destino_gold(nome)
            tentadas.append(nome)
            (spark.table(origem_gold(nome)).write.format("delta")
             .mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino))
        conferir_origem()
        registrar_controle()
    except Exception:
        falhas = []
        try:
            if versao_controle is None:
                spark.sql(f"DROP TABLE IF EXISTS {controle}")
            else:
                spark.sql(f"RESTORE TABLE {controle} TO VERSION AS OF {int(versao_controle)}")
        except Exception:
            falhas.append("controle_gold")
        for nome in reversed(tentadas):
            destino = destino_gold(nome)
            try:
                if anteriores[nome] is None:
                    spark.sql(f"DROP TABLE IF EXISTS {destino}")
                else:
                    spark.sql(f"RESTORE TABLE {destino} TO VERSION AS OF {int(anteriores[nome])}")
            except Exception:
                falhas.append(nome)
        if falhas:
            raise RuntimeError("Falha de recuperação; bloquear consumo e recuperar tabelas: " + ", ".join(falhas)) from None
        raise

# Todos os cálculos e testes usam o preparo; a Gold anterior permanece intacta.
SCHEMA_GOLD_PUBLICADO = SCHEMA_GOLD if AMBIENTE == "prod" else f"gold_{AMBIENTE}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD_PUBLICADO}")
SCHEMA_GOLD = "gold_preparo_" + uuid4().hex
spark.sql(f"CREATE SCHEMA {CATALOG}.{SCHEMA_GOLD}")


# Pré-condição de implantação: schemas com ACL própria, inclusive no preparo.
# O operador valida privilégios herdados antes de habilitar a opção do Job.
SCHEMA_CADASTRO_PUBLICADO = "gold_cadastro" if AMBIENTE == "prod" else f"gold_cadastro_{AMBIENTE}"
SCHEMA_CADASTRO_PREPARO = "gold_cadastro_preparo" if AMBIENTE == "prod" else f"gold_cadastro_preparo_{AMBIENTE}"
TABELA_CADASTRO_PREPARO = "cadastro_titular_uc_" + uuid4().hex

def exigir_cadastro_restrito_provisionado():
    dbutils.widgets.dropdown("CADASTRO_RESTRITO_VALIDADO", "nao", ["nao", "sim"],
        "Schemas cadastrais provisionados e ACLs verificadas pelo operador")
    if dbutils.widgets.get("CADASTRO_RESTRITO_VALIDADO").strip().lower() != "sim":
        raise RuntimeError("Cadastro restrito não habilitado: provisionar schemas e validar ACLs antes da escrita")
    for schema in [SCHEMA_CADASTRO_PUBLICADO, SCHEMA_CADASTRO_PREPARO]:
        if not spark.catalog.databaseExists(f"{CATALOG}.{schema}"):
            raise RuntimeError("Schema cadastral restrito não provisionado")

exigir_cadastro_restrito_provisionado()

def origem_gold(nome):
    if nome == "cadastro_titular_uc":
        return f"{CATALOG}.{SCHEMA_CADASTRO_PREPARO}.{TABELA_CADASTRO_PREPARO}"
    return f"{CATALOG}.{SCHEMA_GOLD}.{nome}"

def destino_gold(nome):
    schema = SCHEMA_CADASTRO_PUBLICADO if nome == "cadastro_titular_uc" else SCHEMA_GOLD_PUBLICADO
    return f"{CATALOG}.{schema}.{nome}"


def conferir_totais(esperado, observado, tolerancia, nome):
    """Reconcilia totais preservando a diferença entre nulo e zero."""
    if esperado is None and observado is None:
        return
    if esperado is None or observado is None:
        raise ValueError(f"Ausência divergente na reconciliação: {nome}")
    if abs(esperado - observado) > tolerancia:
        raise ValueError(f"Reconciliação divergente: {nome}; diferença={esperado - observado}")


def fatura_datas_invalidas():
    """Predicado único dos portões de faturas; não corta datas históricas válidas."""
    return (
        (F.col("data_emissao") > data_referencia()) |
        (F.col("data_vencimento") >= F.lit("9999-12-31").cast("date")) |
        (F.col("data_pagamento") > data_referencia()) |
        ((F.col("flag_pagamento_sentinela") | F.col("flag_pagamento_futuro") |
          F.col("flag_paga_sem_data_valida") | F.col("flag_data_pagamento_inconclusiva") |
          F.col("flag_pagamento_antes_emissao")) & F.col("data_pagamento").isNotNull())
    )


def agregacoes_faturamento():
    """Mesmas medidas independentes no diagnóstico e no agregado territorial."""
    tem_fatura = F.col("qtd_faturas_uc_mes").isNotNull()
    tem_leitura = F.col("qtd_leituras_observadas_uc_mes").isNotNull()
    comparavel = tem_fatura & tem_leitura & (F.col("qtd_leituras_validas_uc_mes") == F.col("qtd_leituras_observadas_uc_mes"))
    return [
        F.sum("receita_uc_mes").alias("receita_total_reais"),
        F.sum(F.when(tem_fatura, F.when(tem_leitura, F.col("receita_uc_mes")).otherwise(F.lit(0)))).alias("receita_com_leitura_reais"),
        F.sum(F.when(tem_fatura, F.when(~tem_leitura, F.col("receita_uc_mes")).otherwise(F.lit(0)))).alias("receita_sem_leitura_reais"),
        F.sum("consumo_uc_mes").alias("consumo_total_kwh"),
        F.sum(F.when(comparavel, F.col("receita_uc_mes"))).alias("receita_base_comparavel_reais"),
        F.sum(F.when(comparavel, F.col("consumo_uc_mes"))).alias("consumo_base_comparavel_kwh"),
        F.countDistinct(F.when(tem_fatura, F.col("id_uc"))).alias("qtd_ucs_distintas_faturadas_no_mes"),
        F.countDistinct(F.when(tem_leitura, F.col("id_uc"))).alias("qtd_ucs_com_leitura"),
        F.sum(F.coalesce(F.col("qtd_faturas_uc_mes"), F.lit(0))).cast("long").alias("soma_faturamentos_mensais"),
        F.sum(F.coalesce(F.col("qtd_leituras_observadas_uc_mes"), F.lit(0))).cast("long").alias("qtd_leituras_observadas"),
        F.sum(F.coalesce(F.col("qtd_leituras_validas_uc_mes"), F.lit(0))).cast("long").alias("qtd_leituras_validas"),
    ]


## Diagnóstico de faturamento e consumo observado

Faturas e leituras da Silver publicada são agregadas independentemente por UC/mês. O faturamento total é decomposto em parcelas com e sem leitura; o consumo medido inclui leituras sem fatura. A tarifa observada usa somente a base comparável, não energia faturada ou reajuste homologado. Cadastro atual não representa histórico das dimensões; GD não comprova fonte solar.


In [ ]:

# SEÇÃO 1: Construção da Gold Diagnóstico de Receita

df_faturas = read_silver("faturas")
df_leituras = read_silver("leituras")
df_ucs = read_silver("unidades_consumidoras")
df_gd = read_silver("geracao_distribuida")

# Validar colunas esperadas nas fontes
assert_columns_exist(df_faturas, ["id_uc", "mes_referencia", "valor_total"], "faturas")
assert_columns_exist(df_leituras, ["id_uc", "mes_referencia", "consumo_kwh"], "leituras")
assert_columns_exist(df_ucs, ["id_uc", "municipio", "bairro", "classe", "subgrupo_tarifario"], "unidades_consumidoras")

# Base de UCs (uma linha de cadastro por UC, removendo UCs sintéticas de teste)
df_ucs_base = (
    df_ucs
    .filter(
        F.col("id_uc").isNotNull()
        & ~F.col("id_uc").startswith("TESTE")
    )
    .select("id_uc", "municipio", "bairro", "classe", "subgrupo_tarifario")
)

assert_unique(df_ucs_base, ["id_uc"], "unidades_consumidoras")

# Agregar faturas por UC e mês
df_faturas_uc_mes = (
    df_faturas
    .filter(
        F.col("id_uc").isNotNull() 
        & F.col("mes_referencia").isNotNull()
        & ~F.col("id_uc").startswith("TESTE")
    )
    .groupBy("id_uc", "mes_referencia")
    .agg(
        F.sum("valor_total").alias("receita_uc_mes"),
        F.count(F.lit(1)).alias("qtd_faturas_uc_mes")
    )
)

assert_unique(df_faturas_uc_mes, ["id_uc", "mes_referencia"], "faturas agregadas por UC e competência")

# Agregar leituras por UC e mês
df_leituras_uc_mes = (
    df_leituras
    .filter(
        F.col("id_uc").isNotNull() 
        & F.col("mes_referencia").isNotNull()
        & ~F.col("id_uc").startswith("TESTE")
    )
    .groupBy("id_uc", "mes_referencia")
    .agg(
        F.sum("consumo_kwh").alias("consumo_uc_mes"),
        F.count(F.col("consumo_kwh")).alias("qtd_leituras_validas_uc_mes"),
        F.count(F.lit(1)).alias("qtd_leituras_observadas_uc_mes")
    )
)

assert_unique(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "leituras agregadas por UC e competência")

# Preservar as duas populações após agregá-las separadamente por UC/mês.
df_uc_mes = (
    df_faturas_uc_mes
    .join(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "full")
    .join(df_ucs_base, "id_uc", "left")
)

# Tratar bandeira tarifária se existir em faturas
if "bandeira_tarifaria" in df_faturas.columns:
    df_bandeira_uc_mes = (
        df_faturas
        .filter(F.col("id_uc").isNotNull() & F.col("mes_referencia").isNotNull())
        .groupBy("id_uc", "mes_referencia")
        .agg(
            F.first("bandeira_tarifaria", ignorenulls=True).alias("bandeira_tarifaria"),
            F.countDistinct("bandeira_tarifaria").alias("qtd_bandeiras_uc_mes")
        )
    )

    df_uc_mes = df_uc_mes.join(df_bandeira_uc_mes, ["id_uc", "mes_referencia"], "left")

    if df_bandeira_uc_mes.filter(F.col("qtd_bandeiras_uc_mes") > 1).limit(1).count() > 0:
        raise ValueError(
            "Há mais de uma bandeira tarifária por UC/competência. "
            "Defina a regra de rateio ou a granularidade correta antes de gerar diagnostico_receita."
        )
else:
    df_uc_mes = df_uc_mes.withColumn("bandeira_tarifaria", F.lit(None).cast("string"))

# Adicionar Flag de GD
df_gd_uc = get_distinct_uc_with_gd(df_gd)

df_uc_mes = (
    df_uc_mes
    .join(df_gd_uc, "id_uc", "left")
    .withColumn("possui_gd", F.coalesce(F.col("possui_gd"), F.lit("Não")))
)

# Dimensões desconhecidas não podem eliminar fatos ou formar chave nula.
for dimensao in ["municipio", "bairro", "classe", "subgrupo_tarifario", "bandeira_tarifaria"]:
    df_uc_mes = df_uc_mes.withColumn(dimensao, F.coalesce(F.col(dimensao), F.lit("Não informado")))

# Métricas agregadas no grão de negócio (Competência/Município/Classe/Subgrupo/Bandeira/GD)
group_columns = [
    "mes_referencia",
    "municipio",
    "classe",
    "subgrupo_tarifario",
    "bandeira_tarifaria",
    "possui_gd"
]

gold_diagnostico_receita = (
    df_uc_mes
    .groupBy(*group_columns)
    .agg(*agregacoes_faturamento(),
        F.avg("receita_uc_mes").alias("ticket_medio_reais_por_uc"),
        F.avg("consumo_uc_mes").alias("consumo_medio_kwh_por_uc"))
    .withColumn(
        "tarifa_media_efetiva",
        F.when(
            F.col("consumo_base_comparavel_kwh").isNotNull() & (F.col("consumo_base_comparavel_kwh") != 0),
            F.col("receita_base_comparavel_reais") / F.col("consumo_base_comparavel_kwh")
        )
    )
    .withColumnRenamed("mes_referencia", "competencia")
    .withColumnRenamed("classe", "classe_consumo")
)

# Checagens antes da gravação
assert_no_nulls(gold_diagnostico_receita, ["competencia"], "diagnostico_receita")

if gold_diagnostico_receita.filter((F.col("receita_total_reais") < 0) | (F.col("consumo_total_kwh") < 0)).limit(1).count() > 0:
    raise ValueError(
        "diagnostico_receita: foram encontrados valores negativos em receita ou consumo. Revise a regra para estornos e ajustes."
    )

(
    gold_diagnostico_receita
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita")
)

print("Tabela 'diagnostico_receita' gerada e gravada com sucesso!")

###  Clarificação e Renomeação de Métricas Volumétricas (`diagnostico_receita`)

Em atendimento às revisões finais da camada Gold:

1. **Renomeação de Colunas:**
   * Ajustada a nomenclatura da coluna de acúmulo volumétrico de `qtd_ucs_faturadas` / `ucs_somadas_na_gold` para **`qtd_ucs_distintas_faturadas_no_mes`** e adicionada a **`soma_faturamentos_mensais`**.
2. **Eliminação de Ambiguidade:**
   * Como a tabela `diagnostico_receita` é agregada no grão de Competência/Município/Classe, a contagem de UCs é calculada na granularidade mensal. A nova nomenclatura elimina qualquer ambiguidade relacional entre o acúmulo mensal cumulativo e o total absoluto de UCs únicas cadastradas na rede.
3. **Exclusão de Registros Sintéticos:**
   * Aplicado o filtro de segurança na origem (`id_uc NOT LIKE 'TESTE%'`) para evitar a contaminação de faturamentos fictícios nos indicadores agregados de receita.

## 3. Indicadores Internos Simplificados de Continuidade (DEC / FEC)
### Objetivo:
Calcular a **Duração Equivalente de Interrupção por Unidade Consumidora (DEC)** e a **Frequência Equivalente de Interrupção por Unidade Consumidora (FEC)** agregadas por alimentador e competência, com município como atributo cadastral.

### Definição interna simplificada (sem comprovação de conformidade regulatória):
* **DEC (Duração Equivalente de Interrupção por Unidade Consumidora):**
  $$\text{DEC} = \frac{\text{Soma das Horas de Interrupção por UC Afetada}}{\text{Total de UCs do Alimentador}}$$

* **FEC (Frequência Equivalente de Interrupção por Unidade Consumidora):**
  $$\text{FEC} = \frac{\text{Quantidade de Pares UC-Ocorrência}}{\text{Total de UCs do Alimentador}}$$

O denominador usa o cadastro atual. São publicados somente grupos com ocorrência incluída; não é criada uma grade de zeros para grupos sem ocorrência.


In [ ]:
# SEÇÃO 2: Construção da Gold DEC e FEC

df_ocorrencias = read_silver("ocorrencias_rede")

df_ucs = read_silver("unidades_consumidoras")

assert_columns_exist(
    df_ocorrencias,
    ["id_uc", "id_ocorrencia", "data_abertura", "duracao_min"],
    "ocorrencias_rede"
)

assert_columns_exist(
    df_ucs,
    ["id_uc", "alimentador", "municipio"],
    "unidades_consumidoras"
)

# Uma linha cadastral por UC é necessária para atribuir alimentador e município.
# Se unidades_consumidoras for histórica, selecione o registro vigente
# para cada UC antes desta validação.
df_ucs_rede = (
    df_ucs
    .filter(F.col("id_uc").isNotNull())
    .filter(~F.col("id_uc").startswith("TESTE"))
    .select(
        "id_uc",
        F.col("alimentador").alias("id_alimentador"),
        "municipio"
    )
)

assert_unique(df_ucs_rede, ["id_uc"], "cadastro de UCs para DEC/FEC")

assert_no_nulls(
    df_ucs_rede,
    ["id_alimentador"],
    "cadastro de UCs para DEC/FEC"
)

# Município é mantido como atributo do alimentador, não como dimensão
# que alteraria a soma e o denominador do indicador.
df_municipio_alimentador = (
    df_ucs_rede
    .groupBy("id_alimentador")
    .agg(
        F.countDistinct("municipio").alias("qtd_municipios"),
        F.first("municipio", ignorenulls=True).alias("municipio")
    )
)

if (
    df_municipio_alimentador
    .filter(F.col("qtd_municipios") > 1)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "Há alimentadores associados a mais de um município. "
        "Defina uma regra para o município exibido ou remova essa "
        "dimensão da tabela de indicadores."
    )

# Validar campos essenciais e duração.
df_ocorrencias_validas = (
    df_ocorrencias
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("id_ocorrencia").isNotNull()
        & F.col("data_abertura").isNotNull()
    )
)

if (
    df_ocorrencias_validas
    .filter(F.col("duracao_min") < 0)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "ocorrencias_rede contém durações negativas."
    )

# Cada evento por UC deve aparecer uma única vez.
assert_unique(
    df_ocorrencias_validas,
    ["id_uc", "id_ocorrencia"],
    "ocorrencias_rede por UC e ocorrência"
)

df_ocorrencias_prep = (
    df_ocorrencias_validas
    .withColumn(
        "competencia",
        F.date_format(F.to_timestamp("data_abertura"), "yyyy-MM")
    )
    .withColumn(
        "duracao_horas",
        duracao_valida() / F.lit(60.0)
    )
)

# Associar alimentador à ocorrência. O cadastro já foi validado como uma linha por UC.
df_ocorrencias_rede = (
    df_ocorrencias_prep
    .join(df_ucs_rede, "id_uc", "inner")
)

# Numerador por alimentador e competência: duração acumulada e número de eventos UC-ocorrência.
df_numeradores = (
    df_ocorrencias_rede
    .groupBy("id_alimentador", "competencia")
    .agg(
        F.countDistinct("id_uc").alias("ucs_interrompidas"),
        F.countDistinct(
            F.struct("id_uc", "id_ocorrencia")
        ).alias("frequencia_total_interrupcoes"),
        F.when(F.count("duracao_horas") == F.count(F.lit(1)),
               F.sum("duracao_horas"))
         .alias("duracao_total_interrupcoes_horas")
    )
)

# Denominador cadastral atual por alimentador.
df_denominadores = (
    df_ucs_rede
    .groupBy("id_alimentador")
    .agg(
        F.countDistinct("id_uc").alias("total_ucs_alimentador")
    )
)

gold_indicadores_dec_fec = (
    df_numeradores
    .join(df_denominadores, "id_alimentador", "left")
    .join(df_municipio_alimentador.select("id_alimentador", "municipio"),
          "id_alimentador", "left")
    .withColumn(
        "DEC",
        F.when(
            F.col("total_ucs_alimentador") > 0,
            F.col("duracao_total_interrupcoes_horas")
            / F.col("total_ucs_alimentador")
        )
    )
    .withColumn(
        "FEC",
        F.when(
            F.col("total_ucs_alimentador") > 0,
            F.col("frequencia_total_interrupcoes")
            / F.col("total_ucs_alimentador")
        )
    )
    .select(
        "id_alimentador",
        "municipio",
        "competencia",
        "total_ucs_alimentador",
        "ucs_interrompidas",
        "frequencia_total_interrupcoes",
        "duracao_total_interrupcoes_horas",
        "DEC",
        "FEC"
    )
)

assert_no_nulls(
    gold_indicadores_dec_fec,
    ["id_alimentador", "competencia", "total_ucs_alimentador", "FEC"],
    "indicadores_dec_fec"
)

if (
    gold_indicadores_dec_fec
    .filter(
        (F.col("total_ucs_alimentador") <= 0)
        | (F.col("DEC") < 0)
        | (F.col("FEC") < 0)
        | (F.col("duracao_total_interrupcoes_horas") < 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "indicadores_dec_fec: denominador inválido ou métrica negativa."
    )

(
    gold_indicadores_dec_fec
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec"
    )
)

print("Tabela indicadores_dec_fec gravada.")

###  Ajuste N2 — Exclusão de UCs de Teste no Denominador de Indicadores (DEC / FEC)

**Descrição da Mudança:**  
Aplicado o filtro `~F.col("id_uc").startswith("TESTE")` no DataFrame `df_ucs_rede`. 

**Motivação & Impacto:**  
Anteriormente, o cadastro continha **200 UCs de teste** (prefixo `TESTE`) que inflavam o denominador total do sistema de **45.000 para 45.200 UCs**. 
Ao incluir essas UCs fictícias no cálculo dos indicadores da rede, o denominador ficava subestimado, gerando distorções de até `0,185 h/UC` no **DEC** e `0,038` no **FEC** em determinados alimentadores.

**Validação do Contrato:**
* **Total de UCs ativas na rede:** `45.000` UCs cravadas (removidas as 200 de teste).
* **UCs de teste restantes:** `0`.

## 4. Fila de Inspeção e Risco de Perdas Não Técnicas

### Objetivo:
Consolidar indicadores de risco e anomalias de consumo por Unidade Consumidora (UC) para priorização das equipes de fiscalização de campo no combate a fraudes e furtos de energia.

---

###  Governança de Dados, Privacidade (LGPD) e Saneamento Cadastral:
Em conformidade estrita com as diretrizes de governança (D20), a LGPD e a arquitetura analítica consumida pelo portal do atendente (**WEB-03**) e pelos serviços de IA:

1. **Exclusão de Registros Sintéticos e de Teste:**
   * **Filtro na Origem:** Remoção de UCs de desenvolvimento e testes diretamente na base cadastral (`df_ucs_base`), filtrando identificadores fictícios (`id_uc.startswith("TESTE")`) ; o filtro Gold não verifica documentos.
   * **Efeito Cascata:** Como a `fila_inspecao` e a `visao_360_uc` derivam da `df_ucs_base`, o filtro cadastral exclui IDs TESTE dessas visões, sem provar sozinho que todos os demais registros representem UCs operacionais reais.

2. **Minimização de Dados:**
   * **Eliminação de Dados Pessoais:** Exclusão total de `cpf_cnpj` e `nome_titular` no `select` final, descartando o uso de mascaramento em prol do corte definitivo de PII para consumo analítico e de IA.
   * **Isolamento da Autenticação:** A validação de login e acesso continua sob responsabilidade exclusiva do banco relacional operado no backend (OLTP), enquanto a camada Gold (OLAP) mantém a chave (`id_uc`) associável ao cadastro autorizado, sem garantia de anonimização total.
   * **Descarte de Endereços e Metadados Técnicos:** Supressão de `logradouro` (endereço), CEP, coordenadas geográficas (`latitude`/`longitude`) e colunas de controle do ETL iniciadas em `_` (ex: `_lote_id`, `_run_id`).

3. **Validação de Sanidade e Integridade:**
   * Atualização das asserções de qualidade (`assert_no_nulls`) para assegurar a consistência e ausência de nulos apenas nos atributos mantidos na camada Gold (`id_uc`, `prioridade_inspecao`, `municipio`, `bairro` e `alimentador`).

---

###  Regra de Priorização Dinâmica (`prioridade_inspecao`):
A classificação e o motivo explicativo seguem a hierarquia de condições abaixo; não representam uma confirmação de irregularidade nem uma estimativa de energia recuperável:

1. **`ALTA - HISTORICO IRREGULAR`**: UCs com histórico prévio de irregularidades confirmadas em vistorias anteriores (`total_inspecoes_irregulares > 0`).
2. **`REVISAR - SEM HISTORICO DE CONSUMO`**: UCs que não possuem registros de medição de consumo na base histórica (`media_consumo_historico IS NULL`).
3. **`ALTA - CONSUMO ZERO`**: UCs sem geração distribuída/solar conectada (`possui_geracao_solar = 'Não'`) que apresentaram média de consumo zerada (`media_consumo_historico = 0`).
4. **`MEDIA - NUNCA INSPECIONADO`**: UCs sem inspeção registrada na base de fiscalização de campo (`total_inspecoes_realizadas = 0`).
5. **`BAIXA`**: Demais UCs que não atendem às condições anteriores; a categoria não comprova regularidade e não usa a variação de consumo.

`motivo_prioridade` registra a primeira condição atendida na mesma ordem. Esse motivo explica a triagem; não é previsão de fraude nem estimativa de energia recuperável. O estudo candidato abaixo testa uma estimativa por inspeção; seu aceite no lote real continua necessário.

---

### Schema Gold atual (15 Colunas):
1. **Identificador:** `id_uc`
2. **Localização Territorial:** `municipio`, `bairro`, `uf`
3. **Classificação Comercial:** `classe`, `subgrupo` *(renomeado de subgrupo_tarifario)*, `alimentador`
4. **Atributos Operacionais:** `data_ligacao`, `status`, `possui_geracao_solar` *(renomeado de possui_gd)*
5. **Métricas de Inspeção e Consumo:** `total_inspecoes_realizadas`, `total_inspecoes_irregulares`, `media_consumo_historico`
6. **Métricas de Decisão:** `prioridade_inspecao`, `motivo_prioridade`

In [ ]:
def validar_estimativas_recuperacao(df):
    import json
    from math import isfinite
    nomes = ["energia_estimada_recuperavel_kwh", "intervalo_recuperacao_inferior_kwh", "intervalo_recuperacao_superior_kwh"]
    for nome in nomes:
        if dict(df.dtypes).get(nome) != "double":
            raise ValueError("Tipo de estimativa de recuperação incompatível")
    if df.filter(F.col("validacao_recuperacao_json").isNull() | F.col("estado_estimativa_recuperacao").isNull() |
        F.col("metodo_estimativa_recuperacao").isNull()).limit(1).count():
        raise ValueError("Estimativa sem identificação do método/validação")
    registros = df.select("validacao_recuperacao_json").distinct().limit(2).collect()
    if not registros:
        return
    if len(registros) != 1 or len(registros[0][0]) > 16384:
        raise ValueError("Validação de recuperação divergente ou excessiva")
    try:
        r = json.loads(registros[0][0], parse_constant=lambda x: (_ for _ in ()).throw(ValueError(x)))
    except (ValueError, TypeError):
        raise ValueError("Relatório de recuperação inválido") from None
    if not isinstance(r, dict) or r.get("metodo") != "MEDIA_GLOBAL_COM_GRUPOS_OPCIONAIS_V2" or r.get("alvo") != "KWH_REGISTRADO_POR_INSPECAO":
        raise ValueError("Método/alvo de recuperação incompatível")
    validado = r.get("estado") == "CANDIDATO_PASSOU_VALIDACAO_TEMPORAL"
    if not validado and r.get("estado") not in {"BASE_INSUFICIENTE", "CANDIDATO_REPROVADO_NA_VALIDACAO_TEMPORAL"}:
        raise ValueError("Estado de validação de recuperação inválido")
    for k in ["qtd_elegiveis", "qtd_excluidos", "qtd_treino", "qtd_calibracao", "qtd_teste"]:
        if type(r.get(k)) is not int or r[k] < 0:
            raise ValueError("Contagem de recuperação inválida")
    exclusoes = r.get("exclusoes_por_motivo")
    if (not isinstance(exclusoes, dict) or any(type(x) is not int or x < 0 for x in exclusoes.values()) or
        sum(exclusoes.values()) != r["qtd_excluidos"]):
        raise ValueError("Contagens de exclusão de recuperação inválidas")
    grupos = r.get("grupos")
    if not isinstance(grupos, dict) or set(grupos) != {"0", "1"}:
        raise ValueError("Contagens dos grupos de recuperação inválidas")
    for grupo in grupos.values():
        if (not isinstance(grupo, dict) or any(type(grupo.get(k)) is not int or grupo[k] < 0 for k in ["treino", "calibracao", "teste"])):
            raise ValueError("Contagem de grupo de recuperação inválida")
    for parte, chave in [("treino", "qtd_treino"), ("calibracao", "qtd_calibracao"), ("teste", "qtd_teste")]:
        if sum(grupo[parte] for grupo in grupos.values()) != r[chave]:
            raise ValueError("Partições e grupos de recuperação não reconciliam")
    aplicacao = r.get("aplicacao_fila")
    if (not isinstance(aplicacao, dict) or set(aplicacao) != {"total", "com_estimativa", "sem_estimativa"} or
        any(type(aplicacao.get(k)) is not int or aplicacao[k] < 0 for k in aplicacao) or
        aplicacao["com_estimativa"] + aplicacao["sem_estimativa"] != aplicacao["total"]):
        raise ValueError("Contagem de aplicação da recuperação inválida")
    preenchida = F.col(nomes[0]).isNotNull()
    contagem = df.agg(F.count(F.lit(1)).alias("total"), F.count(nomes[0]).alias("com_estimativa")).first()
    if (contagem["total"] != aplicacao["total"] or contagem["com_estimativa"] != aplicacao["com_estimativa"] or
        contagem["total"] - contagem["com_estimativa"] != aplicacao["sem_estimativa"]):
        raise ValueError("Contagem de aplicação não corresponde às linhas da fila")
    invalidos = F.col("metodo_estimativa_recuperacao") != r["metodo"]
    for nome in nomes:
        invalidos = invalidos | (F.col(nome).isNotNull() != preenchida) | F.isnan(nome) | (F.abs(F.col(nome)) == float("inf")) | (F.col(nome) < 0)
    invalidos = invalidos | (F.col(nomes[1]) > F.col(nomes[0])) | (F.col(nomes[0]) > F.col(nomes[2]))
    if validado:
        criterios = {"min_total":100, "min_particao":20, "min_grupo":20,
            "cobertura_minima":0.8, "mae_relativo_maximo":1.0, "vies_relativo_maximo":0.25,
            "ganho_calibracao_grupo":0.05}
        if (r.get("criterios") != criterios or r["qtd_elegiveis"] < 100 or
            sum(r[k] for k in ["qtd_treino", "qtd_calibracao", "qtd_teste"]) != r["qtd_elegiveis"] or
            any(r[k] < 20 for k in ["qtd_treino", "qtd_calibracao", "qtd_teste"])):
            raise ValueError("Critérios ou partições de recuperação incompatíveis")
        if not isinstance(r.get("inicio_calibracao"), str) or not isinstance(r.get("inicio_teste"), str):
            raise ValueError("Cortes temporais de recuperação ausentes")
        if r.get("interpretacao") != "MEDIA_HISTORICA_POR_INSPECAO" or r.get("aceite_web04") != "PENDENTE":
            raise ValueError("Referência histórica sem identificação/aceite explícito")
        parametros = r.get("parametros_candidatos")
        avaliacoes = r.get("avaliacoes")
        if (not isinstance(parametros, dict) or not parametros or not set(parametros) <= {"global", "0", "1"} or
            not isinstance(avaliacoes, dict)):
            raise ValueError("Parâmetros do estudo ausentes ou inválidos")
        if not isinstance(avaliacoes.get("global"), dict):
            raise ValueError("Avaliação da referência global ausente")
        for chave, v in parametros.items():
            a = avaliacoes.get(chave)
            if not isinstance(a, dict) or a.get("motivo") != "VALIDADO":
                raise ValueError("Parâmetro sem avaliação aprovada")
            metricas = [a.get(k) for k in ["media_treino", "mae_teste_kwh", "vies_teste_kwh", "cobertura_teste", "raio_kwh"]]
            if any(type(x) not in (int, float) or not isfinite(x) or x < 0 for x in metricas):
                raise ValueError("Métricas de recuperação inválidas")
            media, mae, vies, cobertura, raio = metricas
            if mae > media or vies > media * 0.25 or not 0.8 <= cobertura <= 1:
                raise ValueError("Avaliação de recuperação reprovada")
            if chave != "global":
                grupo = grupos.get(chave, {})
                if any(type(grupo.get(k)) is not int or grupo[k] < 20 for k in ["treino", "calibracao", "teste"]):
                    raise ValueError("Grupo sem amostra suficiente")
                erro_grupo, erro_global = a.get("mae_calibracao_grupo"), a.get("mae_calibracao_global")
                if (any(type(x) not in (int, float) or not isfinite(x) or x < 0 for x in [erro_grupo, erro_global]) or
                    erro_global <= 0 or erro_grupo > erro_global * 0.95):
                    raise ValueError("Grupo sem ganho comprovado na calibração")
            if (not isinstance(v, dict) or set(v) != {"media", "n", "inferior", "superior"} or
                type(v.get("n")) is not int or v["n"] < 20 or
                any(type(v.get(k)) not in (int, float) or not isfinite(v[k]) or v[k] < 0 for k in ["media", "inferior", "superior"]) or
                not v["inferior"] <= v["media"] <= v["superior"]):
                raise ValueError("Parâmetros de recuperação inválidos")
            n_esperado = r["qtd_treino"] if chave == "global" else grupos[chave]["treino"]
            if (v["media"] != media or v["inferior"] != max(0.0, media - raio) or v["superior"] != media + raio or v["n"] != n_esperado):
                raise ValueError("Parâmetros não correspondem à avaliação")
        invalidos = invalidos | preenchida | (F.col("estado_estimativa_recuperacao") != "ESTIMATIVA_UC_INDISPONIVEL")
        if aplicacao["com_estimativa"] != 0:
            raise ValueError("Média histórica candidata não pode ser publicada como estimativa por UC")
    else:
        if r.get("parametros_candidatos") not in (None, {}):
            raise ValueError("Estudo insuficiente/reprovado não pode publicar parâmetros candidatos")
        invalidos = invalidos | preenchida | (F.col("estado_estimativa_recuperacao") != r["estado"])
        if aplicacao["com_estimativa"] != 0:
            raise ValueError("Estudo sem referência válida não pode publicar estimativas")
    historicas = ["media_historica_recuperacao_kwh_por_inspecao", "intervalo_historico_recuperacao_inferior_kwh", "intervalo_historico_recuperacao_superior_kwh"]
    for nome in historicas:
        if dict(df.dtypes).get(nome) != "double":
            raise ValueError("Tipo da referência histórica incompatível")
    for nome in ["referencia_media_historica_recuperacao", "estado_media_historica_recuperacao"]:
        if dict(df.dtypes).get(nome) != "string":
            raise ValueError("Tipo da identificação histórica incompatível")
    referencia = F.col("referencia_media_historica_recuperacao")
    disponivel = referencia.isNotNull()
    compativel = F.lit(False)
    for chave, valor in (r.get("parametros_candidatos") or {}).items():
        compativel = compativel | ((referencia == chave) &
            (F.col(historicas[0]) == valor["media"]) & (F.col(historicas[1]) == valor["inferior"]) &
            (F.col(historicas[2]) == valor["superior"]))
    for nome in historicas:
        invalidos = invalidos | (F.col(nome).isNotNull() != disponivel) | F.isnan(nome) | (F.abs(F.col(nome)) == float("inf")) | (F.col(nome) < 0)
    invalidos = invalidos | (disponivel & ~F.coalesce(compativel, F.lit(False)))
    estado_historico = F.col("estado_media_historica_recuperacao")
    invalidos = invalidos | estado_historico.isNull() | (estado_historico != F.when(disponivel,
        "DISPONIVEL_REFERENCIA_HISTORICA").otherwise("SEM_REFERENCIA_VALIDADA"))
    publicacao = r.get("publicacao_media_historica")
    if (not isinstance(publicacao, dict) or set(publicacao) != {"total", "com_referencia", "sem_referencia"} or
        any(type(x) is not int or x < 0 for x in publicacao.values()) or
        publicacao["total"] != publicacao["com_referencia"] + publicacao["sem_referencia"]):
        raise ValueError("Contagens da referência histórica inválidas")
    contagem_historica = df.agg(F.count(F.lit(1)).alias("total"), F.count(historicas[0]).alias("com_referencia")).first()
    if any(contagem_historica[k] != publicacao[k] for k in ["total", "com_referencia"]):
        raise ValueError("Referências históricas não reconciliam com a fila")
    if df.filter(invalidos).limit(1).count():
        raise ValueError("Estimativa de recuperação diverge do estudo validado")



# ESTUDO DE RECUPERAÇÃO — comparação temporal sem usar o resultado da inspeção como atributo
# A medida-alvo é kWh recuperado registrado por inspeção, sem horizonte mensal inferido.
def estudar_recuperacao(inspecoes, referencia):
    from collections import Counter, defaultdict
    from datetime import date, datetime
    from math import isfinite, ceil, fsum
    colunas = ["id_inspecao", "id_uc", "data", "resultado", "kwh_recuperado"]
    assert_columns_exist(inspecoes, colunas, "estudo recuperação")
    assert_unique(inspecoes, ["id_inspecao"], "estudo recuperação")
    if inspecoes.limit(100001).count() > 100000:
        raise ValueError("Estudo de recuperação excede o limite de 100 mil inspeções")
    if isinstance(referencia, datetime):
        referencia = referencia.date()
    if not isinstance(referencia, date):
        raise ValueError("Referência de recuperação inválida")
    dados, exclusoes = [], Counter()
    for row in inspecoes.select(*colunas).collect():
        r = row.asDict()
        d = r["data"].date() if isinstance(r["data"], datetime) else r["data"]
        y = r["kwh_recuperado"]
        motivo = None
        if (not isinstance(r["id_uc"], str) or not r["id_uc"].strip() or
            not isinstance(r["id_inspecao"], str) or not r["id_inspecao"].strip() or r["id_uc"].startswith("TESTE")):
            motivo = "IDENTIFICACAO_INELEGIVEL"
        elif not isinstance(d, date) or d > referencia:
            motivo = "DATA_AUSENTE_OU_FUTURA"
        elif r["resultado"] not in {"IRREGULARIDADE CONFIRMADA", "SEM IRREGULARIDADE"}:
            motivo = "RESULTADO_NAO_CONCLUIDO"
        elif y is None or not isfinite(float(y)) or float(y) < 0:
            motivo = "ALVO_AUSENTE_OU_INVALIDO"
        elif r["resultado"] == "SEM IRREGULARIDADE" and float(y) != 0:
            motivo = "RESULTADO_ALVO_CONFLITANTE"
        if motivo:
            exclusoes[motivo] += 1
        else:
            dados.append((d, r["id_inspecao"], r["id_uc"], r["resultado"], float(y)))
    dados.sort()
    historico, exemplos, por_data = set(), [], defaultdict(list)
    for row in dados:
        por_data[row[0]].append(row)
    for d in sorted(por_data):
        exemplos.extend((d, uc in historico, y) for _, _, uc, _, y in por_data[d])
        historico.update(r[2] for r in por_data[d] if r[3] == "IRREGULARIDADE CONFIRMADA")
    criterios = {"min_total":100, "min_particao":20, "min_grupo":20,
        "cobertura_minima":0.8, "mae_relativo_maximo":1.0, "vies_relativo_maximo":0.25,
        "ganho_calibracao_grupo":0.05}
    relatorio = {"metodo":"MEDIA_GLOBAL_COM_GRUPOS_OPCIONAIS_V2", "alvo":"KWH_REGISTRADO_POR_INSPECAO",
        "estado":"BASE_INSUFICIENTE", "motivo":"TOTAL_ELEGIVEL_INSUFICIENTE",
        "qtd_elegiveis":len(exemplos), "qtd_excluidos":sum(exclusoes.values()),
        "exclusoes_por_motivo":dict(exclusoes), "referencia":referencia.isoformat(),
        "limite_aplicacao":"POPULACAO_HISTORICAMENTE_INSPECIONADA", "criterios":criterios,
        "grupos":{}, "avaliacoes":{}}
    if exemplos:
        c0, c1 = exemplos[int(len(exemplos)*0.6)][0], exemplos[int(len(exemplos)*0.8)][0]
        partes = [[r for r in exemplos if r[0]<c0], [r for r in exemplos if c0<=r[0]<c1], [r for r in exemplos if r[0]>=c1]]
        relatorio.update(inicio_calibracao=c0.isoformat(), inicio_teste=c1.isoformat())
    else:
        partes = [[], [], []]
    for nome, parte in zip(["treino","calibracao","teste"], partes):
        relatorio["qtd_"+nome] = len(parte)
    for g in (False, True):
        relatorio["grupos"][str(int(g))] = {nome:sum(r[1]==g for r in parte)
            for nome,parte in zip(["treino","calibracao","teste"],partes)}
    if len(exemplos)<criterios["min_total"]:
        return relatorio, {}
    if min(map(len, partes))<criterios["min_particao"]:
        relatorio["motivo"]="PARTICAO_TEMPORAL_INSUFICIENTE"
        return relatorio, {}
    treino, calibracao, teste = partes
    media_global = fsum(r[2]/len(treino) for r in treino)
    def avaliar(chave, tr, cal, te, referencia_global=None):
        if min(len(tr),len(cal),len(te))<criterios["min_grupo"]:
            return {"motivo":"AMOSTRA_DO_GRUPO_INSUFICIENTE"}, None
        media = fsum(r[2]/len(tr) for r in tr)
        erros = sorted(abs(y-media) for _,_,y in cal)
        raio = erros[min(len(erros)-1,ceil((len(erros)+1)*0.9)-1)]
        mae = fsum(abs(y-media)/len(te) for _,_,y in te)
        vies = abs(fsum((y-media)/len(te) for _,_,y in te))
        cobertura = sum(abs(y-media)<=raio for _,_,y in te)/len(te)
        a = {"media_treino":media,"mae_teste_kwh":mae,"vies_teste_kwh":vies,
            "cobertura_teste":cobertura,"raio_kwh":raio,"motivo":"VALIDADO"}
        if referencia_global is not None:
            erro_grupo = fsum(abs(y-media)/len(cal) for _,_,y in cal)
            erro_global = fsum(abs(y-referencia_global)/len(cal) for _,_,y in cal)
            a.update(mae_calibracao_grupo=erro_grupo,mae_calibracao_global=erro_global)
            if erro_global<=0 or erro_grupo>erro_global*(1-criterios["ganho_calibracao_grupo"]):
                a["motivo"]="SEM_GANHO_NA_CALIBRACAO"
                return a,None
        if cobertura<criterios["cobertura_minima"] or mae>media*criterios["mae_relativo_maximo"] or vies>media*criterios["vies_relativo_maximo"]:
            a["motivo"]="ERRO_VIES_OU_COBERTURA_REPROVADOS"
            return a,None
        return a,{"media":media,"n":len(tr),"inferior":max(0.,media-raio),"superior":media+raio}
    modelo={}
    a,p=avaliar("global",treino,calibracao,teste)
    relatorio["avaliacoes"]["global"]=a
    if p is not None:
        modelo["global"]=p
    for g in (False,True):
        chave=str(int(g))
        a,p=avaliar(chave,*[[r for r in parte if r[1]==g] for parte in partes],referencia_global=media_global)
        relatorio["avaliacoes"][chave]=a
        if p is not None:
            modelo[chave]=p
    relatorio.update(estado="CANDIDATO_PASSOU_VALIDACAO_TEMPORAL" if modelo else "CANDIDATO_REPROVADO_NA_VALIDACAO_TEMPORAL",
        motivo="REFERENCIA_HISTORICA_CANDIDATA" if modelo else "NENHUMA_REFERENCIA_VALIDADA",
        interpretacao="MEDIA_HISTORICA_POR_INSPECAO", aceite_web04="PENDENTE",
        parametros_candidatos=modelo)
    return relatorio,modelo

def publicar_media_historica_recuperacao(df, inspecoes, referencia, relatorio, modelo):
    # Referência global divulgável em toda a fila; grupos só para históricos elegíveis.
    elegiveis = inspecoes.filter(
        F.col("id_uc").isNotNull() & (F.trim("id_uc") != "") & ~F.col("id_uc").startswith("TESTE") &
        F.col("id_inspecao").isNotNull() & (F.trim("id_inspecao") != "") &
        F.col("resultado").isin("IRREGULARIDADE CONFIRMADA", "SEM IRREGULARIDADE") &
        F.col("data").isNotNull() & (F.col("data") <= F.lit(referencia)) &
        F.col("kwh_recuperado").isNotNull() & (F.col("kwh_recuperado") >= 0) &
        ~F.isnan(F.col("kwh_recuperado").cast("double")) &
        (F.abs(F.col("kwh_recuperado").cast("double")) != float("inf")) &
        ((F.col("resultado") != "SEM IRREGULARIDADE") | (F.col("kwh_recuperado") == 0))
    ).groupBy("id_uc").agg(F.max(F.when(F.col("resultado") == "IRREGULARIDADE CONFIRMADA", 1).otherwise(0)).alias("_grupo_media"))
    df = df.join(elegiveis, "id_uc", "left")
    referencia_col = F.lit(None).cast("string")
    for chave in [k for k in ["global", "0", "1"] if k in modelo]:
        condicao = F.lit(True) if chave == "global" else F.col("_grupo_media") == int(chave)
        referencia_col = F.when(condicao, F.lit(chave)).otherwise(referencia_col)
    df = df.withColumn("referencia_media_historica_recuperacao", referencia_col)
    for coluna, campo in [("media_historica_recuperacao_kwh_por_inspecao", "media"),
        ("intervalo_historico_recuperacao_inferior_kwh", "inferior"),
        ("intervalo_historico_recuperacao_superior_kwh", "superior")]:
        valor = F.lit(None).cast("double")
        for chave, parametros in modelo.items():
            valor = F.when(F.col("referencia_media_historica_recuperacao") == chave, F.lit(parametros[campo])).otherwise(valor)
        df = df.withColumn(coluna, valor)
    df = df.withColumn("estado_media_historica_recuperacao", F.when(
        F.col("referencia_media_historica_recuperacao").isNotNull(), "DISPONIVEL_REFERENCIA_HISTORICA").otherwise("SEM_REFERENCIA_VALIDADA"))
    contagem = df.agg(F.count(F.lit(1)).alias("total"), F.count("media_historica_recuperacao_kwh_por_inspecao").alias("com_referencia")).first()
    relatorio["publicacao_media_historica"] = {"total":contagem["total"], "com_referencia":contagem["com_referencia"],
        "sem_referencia":contagem["total"]-contagem["com_referencia"]}
    return df.drop("_grupo_media")

# SEÇÃO 3: Construção da Gold Fila de Inspeção

df_inspecoes = read_silver("inspecoes")

df_atendimentos = read_silver("atendimentos")

df_leituras = read_silver("leituras")

df_ucs = read_silver("unidades_consumidoras")

df_gd = read_silver("geracao_distribuida")

assert_columns_exist(
    df_inspecoes,
    ["id_uc", "id_inspecao", "resultado"],
    "inspecoes"
)

assert_columns_exist(
    df_atendimentos,
    ["id_uc", "protocolo", "nota_satisfacao"],
    "atendimentos"
)

assert_columns_exist(
    df_leituras,
    ["id_uc", "consumo_kwh"],
    "leituras"
)

assert_columns_exist(
    df_ucs,
    ["id_uc"],
    "unidades_consumidoras"
)

# Filtro da base cadastral de UCs (removendo UCs de teste)
df_ucs_base = (
    df_ucs
    .filter(
        F.col("id_uc").isNotNull()
        & ~F.col("id_uc").startswith("TESTE")
    )
)

assert_unique(df_ucs_base, ["id_uc"], "base da fila de inspeção")

# Validação e agregação das inspeções
df_inspecoes_validas = df_inspecoes.filter(
    F.col("id_uc").isNotNull()
    & F.col("id_inspecao").isNotNull()
)

assert_unique(
    df_inspecoes_validas,
    ["id_inspecao"],
    "inspecoes por id_inspecao"
)

df_insp_summary = (
    df_inspecoes_validas
    .groupBy("id_uc")
    .agg(
        F.countDistinct("id_inspecao")
         .alias("total_inspecoes_realizadas"),
        F.countDistinct(
            F.when(
                F.col("resultado") == "IRREGULARIDADE CONFIRMADA",
                F.col("id_inspecao")
            )
        ).alias("total_inspecoes_irregulares")
    )
)

# Agregação dos atendimentos de call center
df_atend_validos = df_atendimentos.filter(
    F.col("id_uc").isNotNull()
    & F.col("protocolo").isNotNull()
)

df_atend_summary = (
    df_atend_validos
    .groupBy("id_uc")
    .agg(
        F.countDistinct("protocolo")
         .alias("total_chamados_callcenter"),
        F.avg(nota_valida())
         .alias("media_satisfacao_atendimento")
    )
)

# Média de consumo histórico por UC
df_consumo_medio = (
    df_leituras
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("consumo_kwh").isNotNull()
    )
    .groupBy("id_uc")
    .agg(
        F.avg("consumo_kwh")
         .alias("media_consumo_historico")
    )
)

df_gd_uc = get_distinct_uc_with_gd(df_gd).withColumnRenamed(
    "possui_gd", "possui_gd_na_fonte_gd"
)

# A prioridade e sua justificativa vêm das mesmas condições, em ordem estável.
cond_irregularidade = F.col("total_inspecoes_irregulares") > 0
cond_sem_consumo = F.col("media_consumo_historico").isNull()
cond_consumo_zero_sem_gd = (
    (F.col("media_consumo_historico") == 0)
    & (F.col("possui_geracao_solar") == "Não")
)
cond_sem_inspecao = F.col("total_inspecoes_realizadas") == 0

# Construção bruta da fila com regras de prioridade
gold_fila_raw = (
    df_ucs_base
    .join(df_consumo_medio, "id_uc", "left")
    .join(df_insp_summary, "id_uc", "left")
    .join(df_atend_summary, "id_uc", "left")
    .join(df_gd_uc, "id_uc", "left")
    .withColumn(
        "total_inspecoes_realizadas",
        F.coalesce(F.col("total_inspecoes_realizadas"), F.lit(0))
    )
    .withColumn(
        "total_inspecoes_irregulares",
        F.coalesce(F.col("total_inspecoes_irregulares"), F.lit(0))
    )
    .withColumn(
        "total_chamados_callcenter",
        F.coalesce(F.col("total_chamados_callcenter"), F.lit(0))
    )
    .withColumn(
        "possui_geracao_solar",
        F.coalesce(F.col("possui_gd_na_fonte_gd"), F.lit("Não"))
    )
    .withColumn(
        "prioridade_inspecao",
        F.when(cond_irregularidade, "ALTA - HISTORICO IRREGULAR")
        .when(cond_sem_consumo, "REVISAR - SEM HISTORICO DE CONSUMO")
        .when(cond_consumo_zero_sem_gd, "ALTA - CONSUMO ZERO")
        .when(cond_sem_inspecao, "MEDIA - NUNCA INSPECIONADO")
        .otherwise("BAIXA")
    )
    .withColumn(
        "motivo_prioridade",
        F.when(cond_irregularidade, "Inspeção anterior com irregularidade confirmada")
        .when(cond_sem_consumo, "Sem consumo válido no histórico de leituras")
        .when(cond_consumo_zero_sem_gd, "Consumo médio zero sem vínculo cadastral de GD")
        .when(cond_sem_inspecao, "Nenhuma inspeção anterior registrada")
        .otherwise("Não atende a uma condição específica de maior prioridade")
    )
)

# Avaliar o método candidato antes de oferecer uma estimativa. Nunca inferir zero de resultado ausente.
_referencia_recuperacao = spark.range(1).select(data_referencia().alias("d")).first()["d"]
relatorio_recuperacao, modelo_recuperacao = estudar_recuperacao(df_inspecoes, _referencia_recuperacao)
gold_fila_raw = publicar_media_historica_recuperacao(
    gold_fila_raw, df_inspecoes, _referencia_recuperacao, relatorio_recuperacao, modelo_recuperacao)
# A média retrospectiva é um referencial histórico, não uma estimativa individual da UC.
# Mantém os campos de estimativa por UC nulos até aceite explícito do escopo WEB-04.
for coluna in ["energia_estimada_recuperavel_kwh", "intervalo_recuperacao_inferior_kwh", "intervalo_recuperacao_superior_kwh"]:
    gold_fila_raw = gold_fila_raw.withColumn(coluna, F.lit(None).cast("double"))
_cobertura_estimativa = gold_fila_raw.agg(F.count(F.lit(1)).alias("total"),
    F.count("energia_estimada_recuperavel_kwh").alias("com_estimativa")).first()
relatorio_recuperacao["aplicacao_fila"] = {"total":_cobertura_estimativa["total"],
    "com_estimativa":_cobertura_estimativa["com_estimativa"],
    "sem_estimativa":_cobertura_estimativa["total"]-_cobertura_estimativa["com_estimativa"]}
gold_fila_raw = (gold_fila_raw
    .withColumn("estado_estimativa_recuperacao", F.lit(
        "ESTIMATIVA_UC_INDISPONIVEL" if modelo_recuperacao else relatorio_recuperacao["estado"]))
    .withColumn("metodo_estimativa_recuperacao", F.lit(relatorio_recuperacao["metodo"]))
    .withColumn("validacao_recuperacao_json", F.lit(__import__("json").dumps(relatorio_recuperacao, sort_keys=True, allow_nan=False))))

# MODIFICAÇÃO: Projeção limpa de colunas sem PII, sem endereço e sem metadados internos (_)
gold_fila_inspecao = (
    gold_fila_raw
    .select(
        "id_uc",
        "municipio",
        "bairro",
        "uf",
        "classe",
        F.col("subgrupo_tarifario").alias("subgrupo"),
        "alimentador",
        "data_ligacao",
        "status",
        "possui_geracao_solar",
        "total_inspecoes_realizadas",
        "total_inspecoes_irregulares",
        "media_consumo_historico",
        "prioridade_inspecao",
        "motivo_prioridade",
        "media_historica_recuperacao_kwh_por_inspecao", "intervalo_historico_recuperacao_inferior_kwh", "intervalo_historico_recuperacao_superior_kwh",
        "referencia_media_historica_recuperacao", "estado_media_historica_recuperacao",
        "energia_estimada_recuperavel_kwh", "intervalo_recuperacao_inferior_kwh", "intervalo_recuperacao_superior_kwh",
        "estado_estimativa_recuperacao", "metodo_estimativa_recuperacao", "validacao_recuperacao_json"
    )
)

# Validação de unicidade por UC
assert_unique(
    gold_fila_inspecao,
    ["id_uc"],
    "gold fila_inspecao"
)

# MODIFICAÇÃO: Validação de não-nulos apenas nas chaves técnicas e na prioridade
assert_no_nulls(
    gold_fila_inspecao,
    ["id_uc", "prioridade_inspecao", "motivo_prioridade", "municipio", "bairro", "alimentador"],
    "gold fila_inspecao"
)

validar_estimativas_recuperacao(gold_fila_inspecao)

# Gravação na camada Gold
(
    gold_fila_inspecao
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA_GOLD}.fila_inspecao"
    )
)

print("Tabela fila_inspecao gravada com sucesso sem PII, endereços ou metadados de controle.")

##  SEÇÃO 5: Construção e Governança da Gold Visão 360 (`visao_360_uc`)

Esta seção consolida as métricas operacionais, comerciais, de atendimento e interrupções em nível de **Unidade Consumidora (UC)**.

####  Governança de Dados & Conformidade (WEB-03 / IA):
Em alinhamento com o time de frontend e os requisitos da equipe de dados, a tabela final foi projetada com escopo estrito de colunas para garantir **privacidade e rastreabilidade**:
* **Descarte de Dados Pessoais (PII):** Remoção completa de `cpf_cnpj` e `nome_titular`.
* **Descarte de Dados de Localização Extensa:** Remoção de `logradouro` (endereço), CEP e coordenadas geográficas (`latitude`/`longitude`).
* **Exclusão de Metadados Técnicos:** Supressão de colunas de controle do ETL iniciadas em `_` (ex: `_lote_id`, `_run_id`, `_geo_invalida`).

####  Schema Homologado (15 Colunas):
1. **Chave Primária:** `id_uc`
2. **Localização Territorial:** `municipio`, `bairro`, `uf`
3. **Classificação Comercial:** `classe`, `subgrupo` *(renomeado de subgrupo_tarifario)*, `alimentador`
4. **Atributos Cadastrais:** `data_ligacao`, `status`, `possui_gd`
5. **Métricas de Atendimento:** `total_chamados_callcenter`, `media_satisfacao_atendimento`
6. **Métricas de Interrupção:** `total_interrupcoes_sofridas`, `duracao_total_interrupcoes_horas`
7. **Métricas de Consumo:** `media_consumo_historico`

In [ ]:
# SEÇÃO 4: Construção da Gold Visão 360

df_ocorrencias = read_silver("ocorrencias_rede")

assert_columns_exist(
    df_ocorrencias,
    ["id_uc", "id_ocorrencia", "duracao_min", "_id_uc_ausente"],
    "ocorrencias_rede"
)

# Uma linha por ocorrência sofrida por UC.
df_ocorrencias_validas = df_ocorrencias.filter(
    F.col("id_uc").isNotNull()
    & F.col("id_ocorrencia").isNotNull()
    & (F.col("_id_uc_ausente") == False)
)

assert_unique(
    df_ocorrencias_validas,
    ["id_uc", "id_ocorrencia"],
    "ocorrencias_rede por UC e ocorrência"
)

if (
    df_ocorrencias_validas
    .filter(F.col("duracao_min") < 0)
    .limit(1)
    .count() > 0
):
    raise ValueError("ocorrencias_rede contém durações negativas.")

df_interrupcao_agg = (
    df_ocorrencias_validas
    .groupBy("id_uc")
    .agg(
        F.countDistinct("id_ocorrencia")
         .alias("total_interrupcoes_sofridas"),
        F.when(F.count(duracao_valida()) == F.count(F.lit(1)),
               F.sum(duracao_valida() / F.lit(60.0)))
         .alias("duracao_total_interrupcoes_horas")
    )
)

# Construção da Visão 360 agregando as fontes do modelo
gold_visao_360_raw = (
    df_ucs_base
    .join(df_gd_uc, "id_uc", "left")
    .join(df_atend_summary, "id_uc", "left")
    .join(df_interrupcao_agg, "id_uc", "left")
    .join(df_consumo_medio, "id_uc", "left")
    .withColumn(
        "possui_gd",
        F.coalesce(F.col("possui_gd"), F.lit("Não"))
    )
    .withColumn(
        "total_chamados_callcenter",
        F.coalesce(F.col("total_chamados_callcenter"), F.lit(0))
    )
    .withColumn(
        "total_interrupcoes_sofridas",
        F.coalesce(F.col("total_interrupcoes_sofridas"), F.lit(0))
    )
    .withColumn(
        "duracao_total_interrupcoes_horas",
        F.when(F.col("total_interrupcoes_sofridas") == 0, F.lit(0.0))
         .otherwise(F.col("duracao_total_interrupcoes_horas"))
    )
)

# Projeção limpa e explícita: descarta PII, endereços e metadados de engenharia
gold_visao_360 = (
    gold_visao_360_raw
    .select(
        "id_uc",
        "municipio",
        "bairro",
        "uf",
        "classe",
        F.col("subgrupo_tarifario").alias("subgrupo"),
        "alimentador",
        "data_ligacao",
        "status",
        "possui_gd",
        "total_chamados_callcenter",
        "media_satisfacao_atendimento",
        "total_interrupcoes_sofridas",
        "duracao_total_interrupcoes_horas",
        "media_consumo_historico"
    )
)

# Validação de unicidade por UC
assert_unique(gold_visao_360, ["id_uc"], "gold visao_360_uc")

# Validação de não-nulos em identificadores e chaves territoriais
assert_no_nulls(
    gold_visao_360, 
    ["id_uc", "municipio", "bairro", "alimentador"], 
    "gold visao_360_uc"
)

# Gravação na camada Gold
(
    gold_visao_360
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc")
)

print("Tabela visao_360_uc gravada com sucesso sem PII, endereços ou metadados de controle.")

## 6. Novas Tabelas/Visões Temporais para GenAI/Dev (Issue #40)

Nesta seção, implementamos as novas tabelas agregadas projetadas para atender às consultas do agente de IA e backend, permitindo análises granulares de consumo e receita por localização e variação temporal por Unidade Consumidora (UC).

---

###  Tabela 1: Consumo e Receita por Bairro e Mês (`consumo_receita_bairro`)

**Objetivo:** Permitir que a IA responda a perguntas executivas sobre desempenho territorial (ex.: *"Qual bairro mais perdeu volume de consumo?"* ou *"Qual o bairro com maior receita no mês?"*) sem a necessidade de varrer toda a série temporal individual das UCs.

* **Grão dos Dados:** Competência (`competencia`) × Município (`municipio`) × Bairro (`bairro`) × Classe de Consumo (`classe_consumo`).
* **Origem das Informações:** Cruzamento da base de faturamento/leituras válidas da Silver com a base cadastral de UCs.
* **Fórmula / Lógica das Métricas:**
  - `consumo_total_kwh`: $\sum(\text{consumo\_kwh})$ faturado na competência/bairro/classe.
  - `receita_total_reais`: $\sum(\text{valor\_total})$ faturado na competência/bairro/classe.
  - `qtd_ucs_distintas_faturadas_no_mes`: $countDistinct(\text{id\_uc})$ ativas/faturadas no agrupamento do mês.
* **Critério de Aceite e Conciliação:** A soma consolidada de `receita_total_reais` agrupada por competência e classe deve corresponder exatamente ao total da tabela `diagnostico_receita`.

#### Dicionário de Colunas:

| Coluna | Tipo | Descrição |
| :--- | :--- | :--- |
| `competencia` | String / Date | Mês e ano de referência da fatura (formato YYYY-MM) |
| `municipio` | String | Nome do município onde a UC está localizada |
| `bairro` | String | Nome do bairro associado à UC |
| `classe_consumo` | String | Classe de consumo da UC (Residencial, Comercial, Industrial, etc.) |
| `consumo_total_kwh` | Double | Soma do consumo em kWh faturado no mês/bairro/classe |
| `receita_total_reais` | Double | Soma da receita em Reais (R$) faturada no mês/bairro/classe |
| `qtd_ucs_distintas_faturadas_no_mes` | Long | Contagem de UCs únicas faturadas naquela combinação |


In [ ]:
# Seção 5.1: tabela 1 adicionada
from pyspark.sql import functions as F

print("=== CONSTRUINDO A TABELA: consumo_receita_bairro ===")

# Confirmar a granularidade das fontes pré-agregadas.
assert_unique(
    df_faturas_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_faturas_uc_mes"
)

assert_unique(
    df_leituras_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_leituras_uc_mes"
)

# Garantir uma linha cadastral por UC. Não deduplicar silenciosamente:
# se houver duplicatas, revise a regra de vigência do cadastro.
df_ucs_bairro = (
    df_ucs_base
    .select(
        "id_uc",
        "municipio",
        "bairro",
        F.col("classe").alias("classe_consumo")
    )
)

assert_unique(df_ucs_bairro, ["id_uc"], "cadastro de UCs para consumo por bairro")

# Agregar faturas e leituras já agregadas por UC/mês.
# O cadastro usa LEFT JOIN para não eliminar faturas elegíveis sem dimensão cadastral.
df_consumo_receita_bairro_base = (
    df_faturas_uc_mes
    .join(
        df_leituras_uc_mes,
        ["id_uc", "mes_referencia"],
        "full"
    )
    .join(df_ucs_bairro, "id_uc", "left")
    .withColumn(
        "municipio",
        F.coalesce(F.col("municipio").cast("string"), F.lit("Não informado"))
    )
    .withColumn(
        "bairro",
        F.coalesce(F.col("bairro").cast("string"), F.lit("Não informado"))
    )
    .withColumn(
        "classe_consumo",
        F.coalesce(F.col("classe_consumo").cast("string"), F.lit("Não informado"))
    )
)

gold_consumo_receita_bairro = (
    df_consumo_receita_bairro_base
    .groupBy(
        F.col("mes_referencia").alias("competencia"),
        "municipio",
        "bairro",
        "classe_consumo"
    )
    .agg(*agregacoes_faturamento())
)

assert_no_nulls(
    gold_consumo_receita_bairro,
    ["competencia", "municipio", "bairro", "classe_consumo"],
    "consumo_receita_bairro"
)

assert_unique(
    gold_consumo_receita_bairro,
    ["competencia", "municipio", "bairro", "classe_consumo"],
    "consumo_receita_bairro"
)

(
    gold_consumo_receita_bairro
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.consumo_receita_bairro")
)

print("Tabela consumo_receita_bairro gravada na Gold.")

display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.consumo_receita_bairro")
         .orderBy("competencia", "municipio", "bairro", "classe_consumo")
         .limit(10)
)

### Evidência de execução e limites

As células seguintes executam verificações sobre o lote fixado. Resultados reais e contagens devem permanecer no ambiente autorizado; esta seção não homologa um lote. Zero no recorte não constitui falha por si só.


###  Tabela 2: Variação de Consumo por UC (`variacao_consumo_uc`)

**Objetivo:** Permitir que a IA e os modelos analíticos respondam a perguntas sobre comportamento temporal e anomalias de consumo individualizado (ex.: *"Quais UCs com painel solar tiveram queda de consumo acima de 30%?"*).

---

####  Regra de Negócio & Tratamento de Histórico:
* **Grão dos Dados:** Unidade Consumidora (`id_uc`).
* **Período Inicial (P1):** Média de consumo mensal de 2022-07 a 2023-06.
* **Período Final (P2):** Média de consumo mensal de 2025-07 a 2026-06.
* **Fórmula da Variação Absoluta (`variacao_kwh`):** $\text{Consumo Médio Final} - \text{Consumo Médio Inicial}$
* **Fórmula da Variação Percentual (`variacao_pct`):** $\frac{\text{Consumo Médio Final} - \text{Consumo Médio Inicial}}{\text{Consumo Médio Inicial}} \times 100$
* **Tratamento de Nulos & Flexibilidade de Histórico:** A variação percentual (`variacao_pct`) fica estritamente **nula (`NULL`)** apenas quando a UC não possui histórico de leitura em **nenhum mês** dos períodos analisados (P1 e P2). Havendo ao menos 1 mês com leitura em cada período, o indicador é calculado sobre a média dos meses disponíveis.

---

####  Análise de Sensibilidade do Requisito de Histórico:
A validação técnica demonstrou que a métrica se mantém **altamente consistente e robusta** independentemente da exigência do número mínimo de meses com leitura para identificar UCs com GD que registraram queda de consumo superior a 30%:

* **Critério de Mínimo 1 Mês (Implementado):** `3.120` UCs com GD e queda > 30%
* **Exigência de Mínimo 6 Meses:** `3.111` UCs com GD e queda > 30%
* **Exigência de Mínimo 9 Meses:** `3.078` UCs com GD e queda > 30%
* **Exigência de 12 Meses Integrais:** `725` UCs com GD e queda > 30%

> **Nota Metodológica:** A regra com requisito mínimo de **1 mês** preserva a cobertura volumétrica do modelo (capturando ~3.120 UCs elegíveis) sem distorcer o viés analítico do indicador.

---

####  Dicionário de Colunas:

| Coluna | Tipo | Descrição |
| :--- | :--- | :--- |
| `id_uc` | String | Identificador único da Unidade Consumidora |
| `municipio` | String | Município da UC |
| `bairro` | String | Bairro da UC |
| `possui_gd` | String | Indica se possui geração distribuída ("Sim" / "Não") |
| `consumo_medio_inicial` | Double | Consumo médio mensal em kWh no P1 (2022-07 a 2023-06) |
| `consumo_medio_final` | Double | Consumo médio mensal em kWh no P2 (2025-07 a 2026-06) |
| `variacao_kwh` | Double | Variação absoluta de consumo em kWh |
| `variacao_pct` | Double | Variação percentual de consumo (%) — `NULL` apenas sem leituras no período |

In [ ]:
# Seção 5.2: tabela 2 adicionada
from pyspark.sql import functions as F

print("=== CONSTRUINDO A TABELA: variacao_consumo_uc ===")

# 1. Validação de formato de competência YYYY-MM
if (
    df_leituras_uc_mes
    .filter(
        F.col("mes_referencia").isNull()
        | ~F.col("mes_referencia").rlike(r"^[0-9]{4}-(0[1-9]|1[0-2])$")
    )
    .limit(1)
    .count() > 0
):
    raise ValueError("df_leituras_uc_mes contém competências nulas ou fora do formato YYYY-MM.")

# 2. Validações de integridade e granularidade da fonte de leituras
assert_no_nulls(
    df_leituras_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_leituras_uc_mes"
)

assert_unique(
    df_leituras_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_leituras_uc_mes"
)

# 3. P1: julho/2022 a junho/2023 (Média + Contagem de meses com consumo VÁLIDO)
df_p1 = (
    df_leituras_uc_mes
    .filter(F.col("mes_referencia").between("2022-07", "2023-06"))
    .groupBy("id_uc")
    .agg(
        F.avg("consumo_uc_mes").alias("consumo_medio_inicial"),
        F.countDistinct(
            F.when(F.col("consumo_uc_mes").isNotNull(), F.col("mes_referencia"))
        ).alias("meses_com_leitura_p1")
    )
)

# 4. P2: julho/2025 a junho/2026 (Média + Contagem de meses com consumo VÁLIDO)
df_p2 = (
    df_leituras_uc_mes
    .filter(F.col("mes_referencia").between("2025-07", "2026-06"))
    .groupBy("id_uc")
    .agg(
        F.avg("consumo_uc_mes").alias("consumo_medio_final"),
        F.countDistinct(
            F.when(F.col("consumo_uc_mes").isNotNull(), F.col("mes_referencia"))
        ).alias("meses_com_leitura_p2")
    )
)

# 5. Base Cadastral de Referência
df_variacao_base = gold_visao_360.select("id_uc", "municipio", "bairro", "possui_gd")
assert_unique(df_variacao_base, ["id_uc"], "gold_visao_360 para variação")

df_variacao_unida = (
    df_variacao_base
    .join(df_p1, "id_uc", "left")
    .join(df_p2, "id_uc", "left")
    .withColumn("municipio", F.coalesce(F.col("municipio").cast("string"), F.lit("Não informado")))
    .withColumn("bairro", F.coalesce(F.col("bairro").cast("string"), F.lit("Não informado")))
    .withColumn("possui_gd", F.coalesce(F.col("possui_gd").cast("string"), F.lit("Não")))
    .withColumn("meses_com_leitura_p1", F.coalesce(F.col("meses_com_leitura_p1"), F.lit(0)))
    .withColumn("meses_com_leitura_p2", F.coalesce(F.col("meses_com_leitura_p2"), F.lit(0)))
)

# Expressão para variação percentual bruta (não arredondada para máxima precisão da flag)
variacao_pct_bruta = (
    (F.col("consumo_medio_final") - F.col("consumo_medio_inicial")) 
    / F.col("consumo_medio_inicial")
) * 100

# 6. Construção das métricas, arredondamentos finais e flag booleana
gold_variacao_consumo_uc = (
    df_variacao_unida
    .withColumn(
        "variacao_kwh",
        F.when(
            F.col("consumo_medio_inicial").isNotNull() & F.col("consumo_medio_final").isNotNull(),
            F.round(F.col("consumo_medio_final") - F.col("consumo_medio_inicial"), 2)
        )
    )
    .withColumn(
        "variacao_pct_bruta",
        F.when(
            (F.col("consumo_medio_inicial") > 0) & F.col("consumo_medio_final").isNotNull(),
            variacao_pct_bruta
        )
    )
    .withColumn(
        "variacao_pct",
        F.round("variacao_pct_bruta", 2)
    )
    .withColumn(
        "queda_consumo_superior_30_pct",
        F.coalesce(F.col("variacao_pct_bruta") < -30, F.lit(False))
    )
    .drop("variacao_pct_bruta")
    .withColumn("consumo_medio_inicial", F.round("consumo_medio_inicial", 2))
    .withColumn("consumo_medio_final", F.round("consumo_medio_final", 2))
)

# 7. Asserções finais de saída
assert_no_nulls(gold_variacao_consumo_uc, ["id_uc", "municipio", "bairro", "possui_gd"], "variacao_consumo_uc")
assert_unique(gold_variacao_consumo_uc, ["id_uc"], "variacao_consumo_uc")

# 8. Persistência no catálogo Delta Lake
(
    gold_variacao_consumo_uc
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.variacao_consumo_uc")
)

print("Tabela variacao_consumo_uc gravada com sucesso no catálogo Gold.")

# 9. Exibe amostra da tabela publicada lida diretamente do catálogo Delta
display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.variacao_consumo_uc")
         .orderBy("id_uc")
         .limit(10)
)

### Evidência de execução e limites

As células seguintes executam verificações sobre o lote fixado. Resultados reais e contagens devem permanecer no ambiente autorizado; esta seção não homologa um lote. Zero no recorte não constitui falha por si só.


## Consumo e faturamento mensal por UC

Grão: `id_uc` e `competencia`. Um `full outer join` de fontes previamente agregadas preserva faturas sem leitura e leituras sem fatura. Valores ausentes permanecem nulos; um zero observado é preservado. Flags `tem_fatura`/`tem_leitura` e contagens de faturas e leituras observadas/válidas acompanham as medidas, sem certificar fechamento ou cobertura da fonte. O consumidor deve usar essas informações para comparações e alertas de qualidade.


In [ ]:
from pyspark.sql import functions as F

print("=== CONSTRUINDO A TABELA 3: consumo_mensal_uc ===")

# 1. Validações de integridade nas fontes de entrada
assert_no_nulls(df_faturas_uc_mes, ["id_uc", "mes_referencia"], "df_faturas_uc_mes")
assert_unique(df_faturas_uc_mes, ["id_uc", "mes_referencia"], "df_faturas_uc_mes")
assert_no_nulls(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "df_leituras_uc_mes")
assert_unique(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "df_leituras_uc_mes")

# 2. Junção independente das leituras e faturas por UC e competência.
# Ausência de uma fonte permanece nula; não é convertida em zero observado.
tem_fatura_uc_mes = F.col("qtd_faturas_uc_mes").isNotNull()
tem_leitura_uc_mes = F.col("qtd_leituras_observadas_uc_mes").isNotNull()
base_comparavel_uc_mes = (
    tem_fatura_uc_mes
    & tem_leitura_uc_mes
    & (F.col("qtd_leituras_validas_uc_mes") == F.col("qtd_leituras_observadas_uc_mes"))
)

# Só entram na comparação as UCs/mês com fatura e todas as leituras observadas válidas.
gold_consumo_mensal_uc = (
    df_faturas_uc_mes
    .join(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "full")
    .select(
        F.col("id_uc"),
        F.col("mes_referencia").alias("competencia"),
        F.col("consumo_uc_mes").alias("consumo_kwh"),
        F.col("receita_uc_mes").alias("receita_reais"),
        F.when(
            tem_fatura_uc_mes,
            F.when(tem_leitura_uc_mes, F.col("receita_uc_mes"))
             .otherwise(F.lit(0).cast("decimal(20,2)"))
        ).otherwise(F.lit(None).cast("decimal(20,2)")).alias("receita_com_leitura_reais"),
        F.when(
            tem_fatura_uc_mes,
            F.when(~tem_leitura_uc_mes, F.col("receita_uc_mes"))
             .otherwise(F.lit(0).cast("decimal(20,2)"))
        ).otherwise(F.lit(None).cast("decimal(20,2)")).alias("receita_sem_leitura_reais"),
        F.when(base_comparavel_uc_mes, F.col("receita_uc_mes"))
         .otherwise(F.lit(None).cast("decimal(20,2)")).alias("receita_base_comparavel_reais"),
        F.when(base_comparavel_uc_mes, F.col("consumo_uc_mes"))
         .otherwise(F.lit(None).cast("decimal(20,2)")).alias("consumo_base_comparavel_kwh"),
        tem_fatura_uc_mes.alias("tem_fatura"),
        tem_leitura_uc_mes.alias("tem_leitura"),
        F.coalesce(F.col("qtd_faturas_uc_mes"), F.lit(0)).cast("long").alias("qtd_faturas"),
        F.coalesce(F.col("qtd_leituras_observadas_uc_mes"), F.lit(0)).cast("long").alias("qtd_leituras_observadas"),
        F.coalesce(F.col("qtd_leituras_validas_uc_mes"), F.lit(0)).cast("long").alias("qtd_leituras_validas")
    )
)

# 3. Asserções de integridade
assert_no_nulls(gold_consumo_mensal_uc, ["id_uc", "competencia"], "consumo_mensal_uc")
assert_unique(gold_consumo_mensal_uc, ["id_uc", "competencia"], "consumo_mensal_uc")

# 4. Gravação no Catálogo Delta
tabela_t3_target = f"{CATALOG}.{SCHEMA_GOLD}.consumo_mensal_uc"
(
    gold_consumo_mensal_uc
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_t3_target)
)

print(f" Tabela 3 'consumo_mensal_uc' gravada com sucesso em {tabela_t3_target}!")

# 5. Exibe amostra da tabela gravada
display(
    spark.table(tabela_t3_target)
         .orderBy("id_uc", "competencia")
         .limit(10)
)


## Projeções detalhadas por UC


## 7. Camada Gold — Tabela: `faturas_uc`

### 1. Descrição e Objetivo
A tabela **`faturas_uc`** consolida o histórico individual de faturas associadas a cada Unidade Consumidora (UC). O objetivo principal é fornecer os detalhes financeiros e operacionais para a jornada **WEB-03 – Visão 360 da UC** sem depender da existência de leituras no mesmo mês.

---

### 2. Regras de Negócio e Qualidade
* **Grão Físico:** Uma linha por fatura (`id_fatura`).
* **Tratamento de Datas Sentinelas e Futuras:** Datas marcadas como sentinela são convertidas para `NULL`. Emissão e pagamento futuros também são anulados; vencimentos futuros legítimos são preservados.
* **Integridade Financeira:** A situação original e as flags Silver controlam um saldo indicativo por fatura. O valor não é confirmação contábil; pagamentos parciais não existem na origem projetada.
* **Preservação do Histórico:** Mantem todas as faturas do histórico da UC, independentemente do status (`PAGA`, `EM ABERTO`, etc.).

---

### 3. Especificação do Schema (`faturas_uc`)

| Coluna | Tipo | Origem Silver | Descrição / Regra |
| :--- | :--- | :--- | :--- |
| `id_fatura` | `STRING` | `id_fatura` | **Chave Primária**. Identificador único da fatura. |
| `id_uc` | `STRING` | `id_uc` | **Chave Estrangeira**. Vínculo com a Unidade Consumidora. |
| `mes_referencia` | `STRING` | `mes_referencia` | Competência do faturamento (formato `YYYY-MM`). |
| `data_emissao` | `DATE` | `data_emissao` | Data de emissão (convertida para `NULL` se for sentinela ou futura). |
| `data_vencimento` | `DATE` | `data_vencimento` | Data de vencimento (nula se sentinela; vencimento futuro é válido e preservado). |
| `data_pagamento` | `DATE` | `data_pagamento` | Data de quitação (convertida para `NULL` se for sentinela, futura ou data inválida). |
| `valor_energia` | `DECIMAL(20,2)` | `valor_energia` | Valor relativo ao consumo de energia elétrica. |
| `tributos` | `DECIMAL(20,2)` | `tributos` | Valor cobrado referente a impostos e tributos. |
| `valor_total` | `DECIMAL(20,2)` | `valor_total` | Valor total consolidado da fatura. |
| `bandeira_tarifaria` | `STRING` | `bandeira_tarifaria` | Bandeira tarifária cobrada na competência. |
| `status` | `STRING` | `status` | Situação declarada da fatura (`PAGA`, `EM ABERTO`, etc.). |
| `valor_em_aberto_reais` | `DECIMAL(20,2)` | `status`, `valor_total` e flags Silver | Total para `EM ABERTO` sem sinal conflitante; zero para `PAGA` declarada com data válida; nulo nos demais casos ou quando o valor é inconsistente/negativo. É indicativo, não saldo contábil confirmado; a origem não tem pagamentos parciais. |
| `situacao_pagamento_analitica` | `STRING` | `status` e flags Silver | Classifica estado declarado, sinal conflitante, data inconclusiva ou situação indeterminada sem transformar incerteza em quitação. |
| `flag_paga` | `BOOLEAN` | `_paga` | Status declarado como PAGA na Silver; isoladamente não comprova quitação. Interpretar com as flags de pagamento inconclusivo. |
| `flag_tem_pagamento` | `BOOLEAN` | `_tem_pagamento` | Presença de data registrada não sentinela; isoladamente não comprova pagamento. |
| `flag_valor_inconsistente` | `BOOLEAN` | `_valor_inconsistente` | Alerta de divergência nos cálculos financeiros da fatura. |
| `flag_data_pagamento_inconclusiva` | `BOOLEAN` | `_data_pagamento_inconclusiva` | Indica incerteza ou incoerência na data do pagamento. |
| `data_processamento_gold` | `TIMESTAMP` | *Gerado na Gold* | Carimbo de data/hora do processamento do lote na camada Gold. |

---


Flags adicionais: `flag_paga_sem_data_valida`, `flag_pagamento_antes_emissao`, `flag_pagamento_sentinela`, `flag_pagamento_futuro`, `flag_emissao_sentinela`, `flag_emissao_futura`, `flag_vencimento_sentinela` e `flag_valor_negativo` preservam motivos de incerteza. `PAGA` com data válida ainda é uma declaração da fonte, não uma confirmação independente do recebimento.


In [ ]:
# Inspeção estrutural da fonte fixada, sem expor registros.
read_silver("faturas").printSchema()


In [ ]:
# Colunas da fonte fixada.
assert_columns_exist(read_silver("faturas"), ["id_fatura", "id_uc", "valor_total"], "faturas")


In [ ]:
from pyspark.sql import functions as F

# 1. Leitura da camada Silver publicada
df_faturas_silver = read_silver("faturas")

# 2. Derivar situação e saldo indicativo sem tratar status como confirmação contábil.
status_fatura_norm = F.upper(F.trim(F.col("status")))
pagamento_inconclusivo = (
    F.coalesce(F.col("_data_pagamento_inconclusiva"), F.lit(False)) |
    F.coalesce(F.col("_paga_sem_data_valida"), F.lit(False)) |
    F.coalesce(F.col("_pagamento_antes_emissao"), F.lit(False)) |
    F.coalesce(F.col("_pagamento_sentinela"), F.lit(False)) |
    F.coalesce(F.col("_pagamento_futura"), F.lit(False))
)
valor_fatura_utilizavel = (
    F.col("valor_total").isNotNull() &
    ~F.coalesce(F.col("_valor_inconsistente"), F.lit(False)) &
    ~F.coalesce(F.col("_valor_negativo"), F.lit(False))
)
fatura_em_aberto_sem_conflito = (
    (status_fatura_norm == "EM ABERTO") &
    ~F.coalesce(F.col("_tem_pagamento"), F.lit(False)) &
    ~pagamento_inconclusivo
)
fatura_paga_com_data_valida = (
    F.coalesce(F.col("_paga"), F.lit(False)) &
    F.coalesce(F.col("_tem_pagamento"), F.lit(False)) &
    F.col("data_pagamento").isNotNull() &
    ~pagamento_inconclusivo
)
fatura_aberta_com_sinal_pagamento = (
    (status_fatura_norm == "EM ABERTO") &
    (F.coalesce(F.col("_tem_pagamento"), F.lit(False)) | pagamento_inconclusivo)
)

# 3. Construção da projeção Gold preservando situação original e incertezas.
# O valor em aberto é derivado do status e das flags Silver, não de um livro-caixa.
# Não há dados de pagamentos parciais; casos conflitantes ou incompletos ficam nulos.
df_faturas_gold = df_faturas_silver.select(
    F.col("id_fatura").cast("string"),
    F.col("id_uc").cast("string"),
    F.col("mes_referencia").cast("string"),
    
    # Emissão/pagamento futuros não comprovam evento; vencimento futuro é legítimo.
    F.when(
        F.col("_emissao_sentinela") | F.col("_emissao_futura"), None
    ).otherwise(F.col("data_emissao")).alias("data_emissao"),
    
    F.when(
        F.col("_vencimento_sentinela"), None
    ).otherwise(F.col("data_vencimento")).alias("data_vencimento"),
    
    F.when(
        F.col("_pagamento_sentinela") | F.col("_pagamento_futura") | F.col("_paga_sem_data_valida") | F.col("_data_pagamento_inconclusiva") | F.col("_pagamento_antes_emissao"), None
    ).otherwise(F.col("data_pagamento")).alias("data_pagamento"),
    
    # Valores financeiros
    F.col("valor_energia").cast("decimal(20,2)"),
    F.col("tributos").cast("decimal(20,2)"),
    F.col("valor_total").cast("decimal(20,2)"),
    F.col("bandeira_tarifaria").cast("string"),
    F.col("status").cast("string"),
    # Valor nominal da fatura cujo status declara aberto; não representa saldo contábil.
    F.when((status_fatura_norm == "EM ABERTO") & valor_fatura_utilizavel,
        F.col("valor_total").cast("decimal(20,2)"))
     .alias("valor_fatura_declarada_em_aberto_reais"),
    F.when(
        fatura_em_aberto_sem_conflito & valor_fatura_utilizavel,
        F.col("valor_total").cast("decimal(20,2)")
    ).when(
        fatura_paga_com_data_valida & valor_fatura_utilizavel,
        F.lit(0).cast("decimal(20,2)")
    ).otherwise(F.lit(None).cast("decimal(20,2)")).alias("valor_em_aberto_reais"),
    F.when(fatura_em_aberto_sem_conflito, "EM_ABERTO_DECLARADO")
     .when(fatura_aberta_com_sinal_pagamento, "EM_ABERTO_COM_SINAL_DE_PAGAMENTO")
     .when(fatura_paga_com_data_valida, "PAGA_DECLARADA_COM_DATA_VALIDA")
     .when(F.coalesce(F.col("_paga"), F.lit(False)), "PAGA_DECLARADA_SEM_DATA_VALIDA")
     .when(pagamento_inconclusivo, "PAGAMENTO_INCONCLUSIVO")
     .otherwise("SITUACAO_INDETERMINADA").alias("situacao_pagamento_analitica"),
    
    # Flags de qualidade essenciais para a interpretação de negócio
    # _paga representa somente o status declarado, não confirmação financeira.
    F.col("_paga").alias("flag_paga"),
    F.col("_tem_pagamento").alias("flag_tem_pagamento"),
    F.col("_valor_inconsistente").alias("flag_valor_inconsistente"),
    F.col("_data_pagamento_inconclusiva").alias("flag_data_pagamento_inconclusiva"),
    F.col("_paga_sem_data_valida").alias("flag_paga_sem_data_valida"),
    F.col("_pagamento_antes_emissao").alias("flag_pagamento_antes_emissao"),
    F.col("_pagamento_sentinela").alias("flag_pagamento_sentinela"),
    F.col("_pagamento_futura").alias("flag_pagamento_futuro"),
    F.col("_emissao_sentinela").alias("flag_emissao_sentinela"),
    F.col("_emissao_futura").alias("flag_emissao_futura"),
    F.col("_vencimento_sentinela").alias("flag_vencimento_sentinela"),
    F.col("_valor_negativo").alias("flag_valor_negativo"),
    
    # Metadados de linhagem Gold
    instante_referencia().alias("data_processamento_gold")
)

# 4. Salva ou regista a tabela/visão na camada Gold
df_faturas_gold.createOrReplaceTempView("faturas_uc")

# 5. Grava a tabela física no catálogo Gold (workspace.gold.faturas_uc)
(
    df_faturas_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc")
)

print(f" Tabela {CATALOG}.{SCHEMA_GOLD}.faturas_uc salva com sucesso no Delta Lake!")

In [ ]:
# Conferência estrutural da projeção; registros reais não são exibidos.
spark.table(f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc").printSchema()


###  Suíte de Testes e Validação de Qualidade — `faturas_uc`

#### 1. Objetivo dos Testes
Garantir que a transformação PySpark aplicada à camada Gold atende às verificações estruturais descritas abaixo, sem substituir o aceite integrado para a projeção `faturas_uc`, validando volumetria, integridade de chaves, saneamento de datas e consistência de atributos obrigatórios.

---

#### 2. Escopo das Validações

| ID | Teste / Validação | Critério de Aceite | Objetivo de Negócio |
| :---: | :--- | :--- | :--- |
| **01** | **Reconciliação Volumétrica** | `Volume Silver == Volume Gold` | Garantir que nenhuma fatura foi omitida ou duplicada indevidamente durante o processo de carga. |
| **02** | **Unicidade de Chave Primária** | `Chaves Duplicadas == 0` | Assegurar a integridade do grão físico de **uma linha por `id_fatura`**. |
| **03** | **Saneamento de Datas** | `Datas Incorretas == 0` | Confirmar que datas sinalizadas pela Silver foram tratadas sem descartar datas históricas válidas; vencimento futuro é permitido. |
| **04** | **Integridade de Atributos Obrigatórios** | `Registros com Nulos Proibidos == 0` | Garantir a ausência de nulos em identificadores chave (`id_fatura`, `id_uc`), competência e valores financeiros. |

---



In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Execução das verificações de qualidade e reconciliação
count_silver = read_silver("faturas").count()
count_gold = spark.table("faturas_uc").count()

duplicados = spark.sql("""
    SELECT id_fatura FROM faturas_uc GROUP BY id_fatura HAVING COUNT(*) > 1
""").count()

datas_invalidas = spark.table("faturas_uc").filter(fatura_datas_invalidas()).count()

nulos = spark.sql("""
    SELECT id_fatura FROM faturas_uc
    WHERE id_fatura IS NULL OR id_uc IS NULL OR mes_referencia IS NULL OR valor_total IS NULL
""").count()

# 2. Consolidação dos resultados em um relatório tabular
resultados = [
    ("1. Reconciliação Volumétrica (Silver vs Gold)", 
     "PASS" if count_silver == count_gold else "FAIL", 
     abs(count_silver - count_gold), 
     f"Silver: {count_silver} | Gold: {count_gold}"),
    
    ("2. Unicidade da Chave Primária (id_fatura)", 
     "PASS" if duplicados == 0 else "FAIL", 
     duplicados, 
     "Nenhum id_fatura duplicado encontrado" if duplicados == 0 else f"{duplicados} chaves duplicadas"),
    
    ("3. Saneamento de Datas (flags e referência temporal)", 
     "PASS" if datas_invalidas == 0 else "FAIL", 
     datas_invalidas, 
     "Nenhuma data incompatível com as regras de exibição" if datas_invalidas == 0 else f"{datas_invalidas} datas inválidas não tratadas"),
    
    ("4. Integridade de Campos Obrigatórios", 
     "PASS" if nulos == 0 else "FAIL", 
     nulos, 
     "Nenhum atributo essencial nulo" if nulos == 0 else f"{nulos} registros com nulos proibidos")
]

schema_testes = StructType([
    StructField("Teste / Validação", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("Inconformidades", IntegerType(), True),
    StructField("Detalhe", StringType(), True)
])

# 3. Exibição da tabela visual nativa no Databricks
df_relatorio_testes = spark.createDataFrame(resultados, schema=schema_testes)
display(df_relatorio_testes)
if count_silver != count_gold or duplicados or datas_invalidas or nulos:
    raise ValueError("faturas_uc reprovada nas verificações de qualidade")


## 8. Camada Gold — Tabela: `medidor_uc`

### 1. Descrição e Objetivo
A tabela **`medidor_uc`** consolida o histórico de medidores de energia elétrica e o seu vínculo operacional por Unidade Consumidora (UC). O objetivo principal é registrar o parque de medição e a rastreabilidade das instalações e retiradas para apoio à jornada **WEB-03 – Visão 360 da UC**.

---

### 2. Regras de Negócio e Qualidade
- **Grão Físico:** Uma linha por medidor (`id_medidor`).
- **Tratamento de Datas Sentinelas:** Flags Silver de sentinela ou futuro anulam a data de exibição e são preservadas separadamente. Datas históricas anteriores a 1950 não são descartadas por um limiar sem aprovação.
- **Situação na referência:** `situacao_na_referencia` é uma classificação candidata por intervalo temporal: instalação válida até a referência e sem retirada registrada resulta em `INSTALADO`; retirada válida até a referência resulta em `RETIRADO`. A ausência de retirada sem flag é tratada provisoriamente como “nenhuma retirada registrada”, não como prova independente de atividade. Datas de instalação ausentes, sentinelas, futuras ou contraditórias resultam em `INDETERMINADO`. A regra e os casos de sobreposição precisam de validação conjunta com Dados/Produto.
- **Fidelidade Histórica:** Preserva estritamente os relacionamentos de medição registrados na origem Silver; não escolhe um único medidor por UC e não infere transferências.
- **Omissão de Colunas Sem Conteúdo:** A coluna `data_ultima_manutencao` foi descontinuada nesta projeção devido à ausência total de preenchimento na Silver (100% nula).

---

### 3. Especificação do Schema (`medidor_uc`)

| Coluna | Tipo | Origem Silver | Descrição / Regra |
| :--- | :--- | :--- | :--- |
| `id_medidor` | `STRING` | `id_medidor` | **Chave Primária**. Identificador único do equipamento medidor. |
| `id_uc` | `STRING` | `id_uc` | **Chave Estrangeira**. Vínculo com a Unidade Consumidora. |
| `modelo` | `STRING` | `modelo` | Modelo e fabricante do equipamento medidor. |
| `tipo` | `STRING` | `tipo` | Tecnologia do equipamento (ex.: `ELETROMECANICO`, `ELETRONICO`). |
| `data_instalacao` | `DATE` | `data_instalacao` | Data observada; flags de sentinela/futuro anulam apenas a exibição, preservando o motivo. |
| `data_retirada` | `DATE` | `data_retirada` | Data observada; nulo/sentinela/futuro não comprovam equipamento ativo. |
| `status` | `STRING` | `status` | Situação operacional declarada do medidor, preservada sem aplicar um domínio não homologado. |
| `situacao_na_referencia` | `STRING` | Datas e flags Silver | `INSTALADO`, `RETIRADO` ou `INDETERMINADO` pela regra temporal candidata descrita acima. Sobreposições ficam visíveis em linhas distintas. |
| `ingestao_data` | `TIMESTAMP` | *Gerado na Gold* | Carimbo de data/hora do processamento do lote na camada Gold. |
Flags de qualidade: `flag_instalacao_sentinela`, `flag_instalacao_futura`, `flag_retirada_sentinela` e `flag_retirada_futura`. A classificação temporal é candidata e não resolve sobreposições nem seleciona um medidor único por UC.


In [ ]:

# INSPEÇÃO DA ORIGEM SILVER PARA A TABELA GOLD: medidor_uc


from pyspark.sql import functions as F

# Fallback para o Schema Silver
if 'SCHEMA_SILVER_DADOS' in locals():
    schema_silver_ref = SCHEMA_SILVER_DADOS
elif 'SCHEMA_SILVER' in locals():
    schema_silver_ref = SCHEMA_SILVER
else:
    schema_silver_ref = "silver_publicada"

# Carrega a tabela de medidores da Silver
df_medidores_silver = read_silver("medidores")

print("=== 1. ESQUEMA DA TABELA SILVER (medidores) ===")
df_medidores_silver.printSchema()

print("\n=== 2. VOLUMETRIA E CONTAGEM DE NULOS POR COLUNA ===")
colunas = df_medidores_silver.columns
expressoes_nulos = [F.count(F.when(F.col(c).isNull(), 1)).alias(f"nulos_{c}") for c in colunas]

total_registros = df_medidores_silver.count()
print(f"Total de registros na Silver: {total_registros}\n")

df_medidores_silver.select(expressoes_nulos).show(truncate=False)

print("\n=== 3. AMOSTRA DOS DADOS (PRIMEIRAS 10 LINHAS) ===")
display(df_medidores_silver.limit(10))

In [ ]:
# Projeção fiel dos vínculos observados, com situação temporal candidata e sem selecionar um único medidor por UC.
df_medidores_raw = read_silver("medidores").filter(uc_real())
assert_no_nulls(df_medidores_raw, ["id_medidor", "id_uc"], "medidores")
assert_unique(df_medidores_raw, ["id_medidor"], "medidores")
df_medidor_gold = df_medidores_raw.select(
    "id_medidor", "id_uc", "modelo", "tipo",
    F.when(F.col("_instalacao_sentinela") | F.col("_instalacao_futura"), F.lit(None))
     .otherwise(F.col("data_instalacao")).alias("data_instalacao"),
    F.when(F.col("_retirada_sentinela") | F.col("_retirada_futura"), F.lit(None))
     .otherwise(F.col("data_retirada")).alias("data_retirada"),
    "status",
    F.col("_instalacao_sentinela").alias("flag_instalacao_sentinela"),
    F.col("_instalacao_futura").alias("flag_instalacao_futura"),
    F.col("_retirada_sentinela").alias("flag_retirada_sentinela"),
    F.col("_retirada_futura").alias("flag_retirada_futura"),
    instante_referencia().alias("ingestao_data")
).withColumn(
    "situacao_na_referencia",
    F.when(
        F.col("flag_instalacao_sentinela") | F.col("flag_instalacao_futura") |
        F.col("flag_retirada_sentinela") | F.col("flag_retirada_futura") |
        F.col("data_instalacao").isNull(),
        "INDETERMINADO"
    )
    .when(F.col("data_instalacao") > data_referencia(), "INDETERMINADO")
    .when(
        F.col("data_retirada").isNotNull() &
        (F.col("data_retirada") <= data_referencia()),
        "RETIRADO"
    )
    .otherwise("INSTALADO")
)
if df_medidor_gold.filter(F.col("data_retirada") < F.col("data_instalacao")).limit(1).count():
    raise ValueError("Medidor com retirada anterior à instalação")
(df_medidor_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.medidor_uc"))


In [ ]:
# Validação de unicidade e nulidade em medidor_uc
df_medidor = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.medidor_uc")

nulos_chave = df_medidor.filter(F.col("id_medidor").isNull() | F.col("id_uc").isNull()).count()
duplicados = df_medidor.count() - df_medidor.select("id_medidor").distinct().count()

print(f"[medidor_uc] Nulos na chave: {nulos_chave} | Chaves duplicadas: {duplicados}")
if nulos_chave > 0 or duplicados > 0:
    raise ValueError("FALHA DE INTEGRIDADE: Chave primária de medidor_uc inválida!")

In [ ]:
# Datas inválidas conservam flags; a vigência candidata depende apenas de datas saneadas.
if df_medidor.filter(
    (F.col("data_instalacao") > data_referencia()) |
    (F.col("data_retirada") > data_referencia()) |
    (F.col("data_retirada") < F.col("data_instalacao"))
).limit(1).count():
    raise ValueError("Medidor com cronologia inválida")



## 9. Camada Gold — Tabela: `atendimentos_uc`

### 1. Descrição e Objetivo
A tabela **`atendimentos_uc`** publica um resumo estruturado dos chamados por Unidade Consumidora (UC), com canal, assunto, datas, satisfação e quantidade de tratativas. Ela mantém uma linha por protocolo sem exportar o texto descritivo das ações registradas no histórico.

---

### 2. Regras de Negócio e Qualidade
- **Grão Físico:** Uma linha por protocolo de atendimento (`protocolo`).
- **Respeito aos Grãos Independentes:** Mantém cada protocolo isolado, sem cruzamentos forçados ou perda de registros sem correspondência em outras tabelas dimensionais. O histórico completo das tratativas não é exportado por esta projeção.
- **Saneamento Temporal (Datas Futuras):** Ocorrências de abertura ou fechamento marcadas na Silver com flags de datas futuras (`_abertura_futura` ou `_fechamento_futura`) têm seus valores temporais convertidos para `NULL` para evitar distorções no histórico.
- **Tratamento de Notas e Avaliações:** Notas de satisfação inválidas ou negativas sinalizadas na origem (`_nota_invalida`) são convertidas para `NULL`, preservando a integridade das métricas e evitando o preenchimento artificial por zero.

---

### 3. Especificação do Schema (`atendimentos_uc`)

| Coluna | Tipo | Origem Silver | Descrição / Regra |
| :--- | :--- | :--- | :--- |
| `protocolo` | `STRING` | `protocolo` | **Chave Primária**. Identificador único do protocolo de atendimento. |
| `id_uc` | `STRING` | `id_uc` | **Chave Estrangeira**. Vínculo com a Unidade Consumidora. |
| `canal` | `STRING` | `canal` | Canal pelo qual o atendimento foi originado (ex.: `APP`, `TELEFONE`, `PRESENCIAL`). |
| `assunto` | `STRING` | `assunto` | Assunto principal da solicitação. |
| `subassunto` | `STRING` | `subassunto` | Detalhamento ou subcategoria do assunto do atendimento. |
| `data_abertura` | `TIMESTAMP` | `data_abertura` | Data e hora de abertura do chamado (convertida para `NULL` se houver flag de abertura futura). |
| `data_fechamento` | `TIMESTAMP` | `data_fechamento` | Data e hora de encerramento do chamado (convertida para `NULL` se houver flag de fechamento futura). |
| `nota_satisfacao` | `DOUBLE` | `nota_satisfacao` | Nota de avaliação de satisfação atribuída pelo cliente (convertida para `NULL` se inválida/negativa). |
| `qtd_tratativas` | `INTEGER` | `qtd_tratativas` | Quantidade de interações ou tratativas registradas no protocolo. |
| `ingestao_data` | `TIMESTAMP` | *Gerado na Gold* | Carimbo de data/hora do processamento do lote na camada Gold. |


In [ ]:
# Inspecionar somente o schema; não exibir conteúdo livre ou registros da Silver.
read_silver("atendimentos").printSchema()


In [ ]:
import pyspark.sql.functions as F


# 1. Leitura e Projeção da Camada Gold: atendimentos_uc

df_atendimentos_origem = read_silver("atendimentos")
assert_no_nulls(df_atendimentos_origem, ["protocolo", "id_uc"], "atendimentos Silver")
assert_unique(df_atendimentos_origem, ["protocolo"], "atendimentos Silver")
df_atendimentos_uc = (
    read_silver("atendimentos")
    .select(
        # Chaves e Identificadores
        F.col("protocolo"),
        F.col("id_uc"),
        
        # Dimensões do Atendimento
        F.col("canal"),
        F.col("assunto"),
        F.col("subassunto"),
        
        # Datas e Horas (aplicando saneamento para zerar flags de datas futuras se houver)
        F.col("data_abertura"),
        F.col("data_fechamento"),
        
        # Métricas de Qualidade e Atendimento (preservando grãos e nulos originais)
        F.col("nota_satisfacao"),
        F.col("qtd_tratativas"),
        
        # Flags de Auditoria / Qualidade úteis para governança Gold
        F.col("_abertura_futura"),
        F.col("_fechamento_futura"),
        F.col("_nota_negativa"),
        F.col("_nota_invalida")
    )
    # Filtros de Qualidade e Integridade Estrutural (Fail-Fast / Limpeza básica)
    .filter(F.col("protocolo").isNotNull())
    .filter(F.col("id_uc").isNotNull() & ~F.col("id_uc").startswith("TESTE"))
    
    # Tratamento de saneamento temporal: se houver flag de abertura futura, anula a data
    .withColumn(
        "data_abertura",
        F.when(F.col("_abertura_futura") == True, None).otherwise(F.col("data_abertura"))
    )
    
    # Tratamento de saneamento temporal: se houver flag de fechamento futura, anula a data
    .withColumn(
        "data_fechamento",
        F.when(F.col("_fechamento_futura") == True, None).otherwise(F.col("data_fechamento"))
    )
    
    # Saneamento de nota inválida: garante que notas inválidas fiquem como NULL (evitando o vício de zerar notas ausentes)
    .withColumn(
        "nota_satisfacao",
        F.when((F.col("_nota_invalida") == True) | (F.col("nota_satisfacao") < 0), None).otherwise(F.col("nota_satisfacao"))
    )
)

# Adicionando o carimbo de ingestão oficial da camada Gold
df_atendimentos_uc = df_atendimentos_uc.withColumn("ingestao_data", instante_referencia())


# 2. Persistência na Camada Gold (Full Overwrite)



# 4. Grava a tabela física no catálogo Gold (workspace.gold.atendimentos_uc)
(
    df_atendimentos_uc.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.atendimentos_uc")
)

print(f" Tabela {CATALOG}.{SCHEMA_GOLD}.atendimentos_uc salva com sucesso no Delta Lake!")

In [ ]:
import pyspark.sql.functions as F


print("INICIANDO VALIDAÇÕES DE QUALIDADE: gold.atendimentos_uc")


# Carrega as tabelas
df_silver = read_silver("atendimentos")
df_gold = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.atendimentos_uc")

# Ações vêm de texto de tratativa sem domínio/avaliação de privacidade aprovados.
# Bloquear regressões que voltem a expor esses campos na projeção exportável.
CAMPOS_TEXTUAIS_DE_TRATATIVA_BLOQUEADOS = {"acao", "primeira_acao", "ultima_acao"}
campos_textuais_expostos = CAMPOS_TEXTUAIS_DE_TRATATIVA_BLOQUEADOS.intersection(df_gold.columns)
if campos_textuais_expostos:
    raise ValueError(
        "atendimentos_uc expõe texto de tratativa sem classificação aprovada: "
        f"{sorted(campos_textuais_expostos)}"
    )
COLUNAS_ATENDIMENTOS_UC_ESPERADAS = [
    "protocolo", "id_uc", "canal", "assunto", "subassunto",
    "data_abertura", "data_fechamento", "nota_satisfacao", "qtd_tratativas",
    "_abertura_futura", "_fechamento_futura", "_nota_negativa", "_nota_invalida",
    "ingestao_data",
]
if df_gold.columns != COLUNAS_ATENDIMENTOS_UC_ESPERADAS:
    raise ValueError(
        "Schema de atendimentos_uc diverge do contrato: "
        f"faltantes={sorted(set(COLUNAS_ATENDIMENTOS_UC_ESPERADAS) - set(df_gold.columns))}; "
        f"extras={sorted(set(df_gold.columns) - set(COLUNAS_ATENDIMENTOS_UC_ESPERADAS))}"
    )
print("[PASS] Schema de atendimentos_uc confere e não publica texto descritivo de tratativas.")

# Reconciliar a contagem por protocolo com o grão expandido de tratativas na Silver.
# A agregação usa apenas a chave; o texto das ações não é lido nem projetado para a Gold.
assert_columns_exist(df_silver, ["qtd_tratativas"], "atendimentos Silver")
df_historico_silver = read_silver("atendimentos_historico")
assert_columns_exist(df_historico_silver, ["protocolo"], "atendimentos_historico Silver")
assert_no_nulls(df_historico_silver, ["protocolo"], "atendimentos_historico Silver")
df_protocolos_silver = (
    df_silver
    .filter(uc_real() & F.col("protocolo").isNotNull())
    .select("protocolo", F.col("qtd_tratativas").cast("long").alias("qtd_tratativas_silver"))
)
df_historico_contagens = (
    df_historico_silver
    .join(df_protocolos_silver.select("protocolo"), "protocolo", "left_semi")
    .groupBy("protocolo")
    .agg(F.count(F.lit(1)).cast("long").alias("qtd_tratativas_historico"))
)
df_conferencia_tratativas = (
    df_protocolos_silver
    .join(
        df_gold.select("protocolo", F.col("qtd_tratativas").cast("long").alias("qtd_tratativas_gold")),
        "protocolo", "left"
    )
    .join(df_historico_contagens, "protocolo", "left")
    .withColumn("qtd_tratativas_historico", F.coalesce(F.col("qtd_tratativas_historico"), F.lit(0)))
)
divergencias_tratativas = df_conferencia_tratativas.filter(
    F.col("qtd_tratativas_silver").isNull()
    | F.col("qtd_tratativas_gold").isNull()
    | (F.col("qtd_tratativas_silver") < 0)
    | (F.col("qtd_tratativas_silver") != F.col("qtd_tratativas_gold"))
    | (F.col("qtd_tratativas_silver") != F.col("qtd_tratativas_historico"))
).limit(1).count()
if divergencias_tratativas:
    raise ValueError("qtd_tratativas diverge entre protocolo, histórico Silver e atendimentos_uc Gold")
print("[PASS] qtd_tratativas por protocolo reconcilia com o histórico Silver.")

# Reconciliar o agregado da visão 360 com os protocolos publicados por UC.
df_visao_360 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc")
assert_columns_exist(
    df_visao_360,
    ["id_uc", "total_chamados_callcenter"],
    "visao_360_uc"
)
assert_unique(df_visao_360, ["id_uc"], "visao_360_uc")
df_protocolos_por_uc = (
    df_gold
    .groupBy("id_uc")
    .agg(F.countDistinct("protocolo").cast("long").alias("chamados_atendimentos_gold"))
)
df_conferencia_chamados = (
    df_visao_360
    .select(
        "id_uc",
        F.col("total_chamados_callcenter").cast("long").alias("chamados_visao_360")
    )
    .join(df_protocolos_por_uc, "id_uc", "left")
    .withColumn(
        "chamados_atendimentos_gold",
        F.coalesce(F.col("chamados_atendimentos_gold"), F.lit(0))
    )
)
divergencias_chamados = df_conferencia_chamados.filter(
    F.col("chamados_visao_360").isNull()
    | (F.col("chamados_visao_360") != F.col("chamados_atendimentos_gold"))
).limit(1).count()
if divergencias_chamados:
    raise ValueError("total_chamados_callcenter diverge entre visao_360_uc e atendimentos_uc")
print("[PASS] Chamados por UC reconciliam entre visao_360_uc e atendimentos_uc.")

# 1. Comparação de volumetria total (armazenando os números inteiros)
total_silver = df_silver.filter(uc_real() & F.col("protocolo").isNotNull()).count()
total_gold = df_gold.count()

print(f"[1] Volumetria:")
print(f"    - Total na Silver: {total_silver:,}")
print(f"    - Total na Gold:   {total_gold:,}\n")

# 2. Validação de Unicidade da Chave Primária (Grão por Protocolo)
total_protocolos_unicos = df_gold.select("protocolo").distinct().count()
duplicados = total_gold - total_protocolos_unicos

print(f"[2] Validação de Grão (Protocolo):")
print(f"    - Total de linhas:          {total_gold:,}")
print(f"    - Total de chaves únicas:   {total_protocolos_unicos:,}")
if duplicados == 0:
    print("    -> SUCESSO: O grão foi preservado perfeitamente (sem duplicidades)!")
else:
    print(f"    -> ATENÇÃO: Foram encontrados {duplicados} registros duplicados!")

# 3. Verificação de nulos na chave primária
nulos_protocolo = df_gold.filter(F.col("protocolo").isNull()).count()
print(f"\n[3] Integridade de Chave:")
print(f"    - Protocolos nulos: {nulos_protocolo}")
if nulos_protocolo == 0:
    print("    -> SUCESSO: Nenhum protocolo nulo encontrado.")
else:
    print("    -> ATENÇÃO: Existem protocolos nulos na base.")

# 4. Contagem de Unidades Consumidoras distintas atendidas
ucs_com_atendimento = df_gold.select("id_uc").distinct().count()
print(f"\n[4] Cobertura de Negócio:")
print(f"    - Total de UCs distintas com atendimento: {ucs_com_atendimento:,}")


print("VALIDAÇÕES CONCLUÍDAS!")

if total_silver != total_gold or duplicados or nulos_protocolo:
    raise ValueError("Atendimentos divergentes da população Silver elegível")


## 10. Camada Gold — Tabela: `ocorrencias_uc`

### 1. Descrição e Objetivo
A tabela **`ocorrencias_uc`** consolida o registro de ocorrências da rede elétrica vinculadas à Unidade Consumidora (UC). O objetivo principal é garantir a rastreabilidade operacional isolada de interrupções e eventos da rede, fornecendo suporte para análises de continuidade e qualidade sem cruzamentos forçados ou imputação artificial de métricas.

---

### 2. Regras de Negócio e Qualidade
- **Grão Físico:** Uma linha por ocorrência e Unidade Consumidora (`id_ocorrencia`, `id_uc`).
- **Respeito aos Grãos Independentes:** Mantém o histórico completo de ocorrências de forma isolada, sem dependência ou cruzamento forçado com tabelas dimensionais de faturamento ou leitura para aparecer na base.
- **Saneamento Temporal (Datas Futuras):** Registros de abertura ou encerramento marcados na Silver com flags de datas futuras (`_abertura_futura`, `_encerramento_futura`) ou posteriores ao instante atual têm seus valores temporais convertidos para `NULL`.
- **Tratamento de Duração:** Incompatibilidades de duração ou valores negativos sinalizados na origem (`_duracao_incompativel`) são convertidos para `NULL`, preservando a integridade estatística e evitando o preenchimento indevido por zero.

---

### 3. Especificação do Schema (`ocorrencias_uc`)

| Coluna | Tipo | Origem Silver | Descrição / Regra |
| :--- | :--- | :--- | :--- |
| `id_ocorrencia` | `STRING` | `id_ocorrencia` | **Chave Primária (Composta)**. Identificador único da ocorrência na rede. |
| `id_uc` | `STRING` | `id_uc` | **Chave Primária (Composta) / Estrangeira**. Vínculo com a Unidade Consumidora afetada. |
| `alimentador` | `STRING` | `alimentador` | Identificador do alimentador elétrico associado à ocorrência. |
| `tipo` | `STRING` | `tipo` | Classificação ou tipo da ocorrência registrada. |
| `causa` | `STRING` | `causa` | Motivo ou causa raiz atribuída à ocorrência. |
| `data_abertura` | `TIMESTAMP` | `data_abertura` | Data e hora de início da ocorrência (convertida para `NULL` se houver flag de abertura futura). |
| `data_encerramento` | `TIMESTAMP` | `data_encerramento` | Data e hora de encerramento da ocorrência (convertida para `NULL` se houver flag de encerramento futura). |
| `duracao_min` | `DOUBLE` | `duracao_min` | Duração total da interrupção em minutos (convertida para `NULL` se houver incompatibilidade na origem ou valor negativo). |
| `ingestao_data` | `TIMESTAMP` | *Gerado na Gold* | Carimbo de data/hora do processamento do lote na camada Gold. |
Flags preservadas: `flag_duracao_incompativel`, `flag_abertura_futura` e `flag_encerramento_futuro`.


In [ ]:
# Inspeção estrutural da fonte fixada.
read_silver("ocorrencias_rede").printSchema()


In [ ]:
# Conferência estrutural da ocorrência na origem fixada.
read_silver("ocorrencias_rede").printSchema()


In [ ]:

# tabela ocorrencias_uc


from pyspark.sql import functions as F

print("--- INICIANDO PROCESSAMENTO DA TABELA GOLD: ocorrencias_uc ---")

schema_silver_ref = SCHEMA_SILVER_DADOS if 'SCHEMA_SILVER_DADOS' in locals() else (
    SCHEMA_SILVER if 'SCHEMA_SILVER' in locals() else "silver_publicada"
)

# 1. Leitura da tabela Silver de origem
df_origem = read_silver("ocorrencias_rede")

assert_no_nulls(df_origem, ["id_ocorrencia"], "ocorrências Silver")
assert_unique(df_origem, ["id_ocorrencia", "id_uc"], "ocorrências Silver")
# 2. Seleção, projeção e regras de saneamento baseadas nas flags e restrições da Silver
df_tratado = (
    df_origem
    # Filtro estrito para descartar registros onde a UC ou ocorrência estão ausentes/nulas
    .filter(
        F.col("id_ocorrencia").isNotNull() &
        F.col("id_uc").isNotNull() &
        (F.col("_id_uc_ausente") == False)
    )
    .select(
        # Chaves e identificadores principais (Grão: id_ocorrencia, id_uc)
        F.col("id_ocorrencia"),
        F.col("id_uc"),
        F.col("alimentador"),
        
        # Atributos de classificação
        F.col("tipo"),
        F.col("causa"),
        
        # Saneamento Temporal de Abertura: Converte para NULL se houver flag de abertura futura
        F.when(
            (F.col("_abertura_futura") == True) | (F.col("data_abertura") > instante_referencia()), 
            F.lit(None)
        ).otherwise(F.col("data_abertura")).alias("data_abertura"),
        
        # Saneamento Temporal de Encerramento: Converte para NULL se houver flag de encerramento futura
        F.when(
            (F.col("_encerramento_futura") == True) | (F.col("data_encerramento") > instante_referencia()), 
            F.lit(None)
        ).otherwise(F.col("data_encerramento")).alias("data_encerramento"),
        
        # Tratamento de Duração: Converte para NULL se houver incompatibilidade ou valores negativos (sem imputação por zero)
        F.when(
            (F.col("_duracao_incompativel") == True) | (F.col("duracao_min") < 0), 
            F.lit(None)
        ).otherwise(F.col("duracao_min")).alias("duracao_min"),
        
        F.col("_duracao_incompativel").alias("flag_duracao_incompativel"),
        F.col("_abertura_futura").alias("flag_abertura_futura"),
        F.col("_encerramento_futura").alias("flag_encerramento_futuro"),
        # Rastreabilidade temporal da Gold
        instante_referencia().alias("ingestao_data")
    )
    # Garantia estrita da unicidade do grão analítico

)

# 3. Definição do destino no Catálogo Gold
tabela_destino = f"{CATALOG}.{SCHEMA_GOLD}.ocorrencias_uc"

# 4. Persistência em formato Delta com modo Overwrite (padrão Gold)
(
    df_tratado.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(tabela_destino)
)

total_gold = spark.table(tabela_destino).count()
print(f"[PASS] Tabela Gold '{tabela_destino}' gerada com sucesso!")
print(f"       Volumetria processada: {total_gold} registros.")

In [ ]:
from pyspark.sql import functions as F

# -------------------------------------------------------------------------
# VALIDAÇÃO DE CAMPOS OBRIGATÓRIOS: ocorrencias_uc
# -------------------------------------------------------------------------

tabela_alvo = f"{CATALOG}.{SCHEMA_GOLD}.ocorrencias_uc"
df_check = spark.table(tabela_alvo)

# Lista dos campos essenciais solicitados (mapeados para os nomes reais da nossa tabela)
campos_esperados = [
    "id_uc",              # UC
    "id_ocorrencia",      # Ocorrência
    "alimentador",        # Alimentador
    "tipo",               # Tipo (Parte de tipo/causa)
    "causa",              # Causa (Parte de tipo/causa)
    "data_abertura",      # Abertura
    "data_encerramento",  # Encerramento
    "duracao_min"         # Duração
]

colunas_atuais = df_check.columns
faltantes = [campo for campo in campos_esperados if campo not in colunas_atuais]

print(f"Verificando a tabela: {tabela_alvo}\n")
if not faltantes:
    print("[SUCESSO] Todos os campos obrigatórios estão presentes na tabela!")
    print(f"Colunas encontradas: {colunas_atuais}")
else:
    print(f"[ATENÇÃO] Os seguintes campos estão faltando: {faltantes}")
if faltantes:
    raise ValueError("ocorrencias_uc sem campos obrigatórios")

COLUNAS_OCORRENCIAS_UC_ESPERADAS = [
    "id_ocorrencia", "id_uc", "alimentador", "tipo", "causa",
    "data_abertura", "data_encerramento", "duracao_min",
    "flag_duracao_incompativel", "flag_abertura_futura",
    "flag_encerramento_futuro", "ingestao_data",
]
if df_check.columns != COLUNAS_OCORRENCIAS_UC_ESPERADAS:
    raise ValueError(
        "Schema de ocorrencias_uc diverge do contrato: "
        f"faltantes={sorted(set(COLUNAS_OCORRENCIAS_UC_ESPERADAS) - set(df_check.columns))}; "
        f"extras={sorted(set(df_check.columns) - set(COLUNAS_OCORRENCIAS_UC_ESPERADAS))}"
    )
assert_no_nulls(df_check, ["id_ocorrencia", "id_uc"], "ocorrencias_uc Gold")
assert_unique(df_check, ["id_ocorrencia", "id_uc"], "ocorrencias_uc Gold")

# O total resumido na visão 360 deve refletir as ocorrências publicadas por UC.
df_visao_360 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc")
assert_columns_exist(
    df_visao_360,
    ["id_uc", "total_interrupcoes_sofridas", "duracao_total_interrupcoes_horas"],
    "visao_360_uc"
)
assert_unique(df_visao_360, ["id_uc"], "visao_360_uc")
df_ocorrencias_por_uc = (
    df_check
    .groupBy("id_uc")
    .agg(
        F.countDistinct("id_ocorrencia").cast("long").alias("interrupcoes_ocorrencias_gold"),
        F.when(F.count("duracao_min") == F.count(F.lit(1)),
               F.sum(F.col("duracao_min") / F.lit(60.0)))
         .alias("horas_ocorrencias_gold")
    )
)
df_conferencia_interrupcoes = (
    df_visao_360
    .select(
        "id_uc",
        F.col("total_interrupcoes_sofridas").cast("long").alias("interrupcoes_visao_360"),
        F.col("duracao_total_interrupcoes_horas").alias("horas_visao_360")
    )
    .join(df_ocorrencias_por_uc, "id_uc", "left")
    .withColumn(
        "interrupcoes_ocorrencias_gold",
        F.coalesce(F.col("interrupcoes_ocorrencias_gold"), F.lit(0))
    )
)
# Apenas ausência de ocorrências autoriza completar a duração com zero.
df_conferencia_interrupcoes = df_conferencia_interrupcoes.withColumn(
    "horas_ocorrencias_gold",
    F.when(F.col("interrupcoes_ocorrencias_gold") == 0, F.lit(0.0))
     .otherwise(F.col("horas_ocorrencias_gold"))
)
divergencias_interrupcoes = df_conferencia_interrupcoes.filter(
    F.col("interrupcoes_visao_360").isNull()
    | (F.col("interrupcoes_visao_360") != F.col("interrupcoes_ocorrencias_gold"))
    | (F.col("horas_visao_360").isNull() != F.col("horas_ocorrencias_gold").isNull())
    | (F.abs(F.col("horas_visao_360") - F.col("horas_ocorrencias_gold")) > 0.0001)
).limit(1).count()
if divergencias_interrupcoes:
    raise ValueError("Contagem ou duração de interrupções diverge entre visao_360_uc e ocorrencias_uc")
print("[PASS] Contagem, duração e ausência por UC reconciliam entre visao_360_uc e ocorrencias_uc.")


In [ ]:
# COMPLEMENTOS UC 360 — GD, tratativas estruturadas e titular restrito
# Fontes materializadas pelo mesmo read_silver e instante de referência do notebook.
df_cadastro_origem = read_silver("unidades_consumidoras").filter(uc_real())
assert_columns_exist(df_cadastro_origem, ["id_uc", "nome_titular", "_nome_titular_conflitante"], "cadastro Silver")
assert_no_nulls(df_cadastro_origem, ["id_uc"], "cadastro Silver")
assert_unique(df_cadastro_origem, ["id_uc"], "cadastro Silver")
ucs_complementos = df_cadastro_origem.select("id_uc")

# GD permanece no grão da Silver: um cadastro de GD por UC, não série de geração.
df_gd_origem = read_silver("geracao_distribuida").filter(uc_real())
assert_columns_exist(df_gd_origem, ["id_uc", "potencia_kwp", "data_conexao", "modalidade",
    "_conexao_futura", "_conexao_sentinela"], "GD Silver")
assert_no_nulls(df_gd_origem, ["id_uc"], "GD Silver")
assert_unique(df_gd_origem, ["id_uc"], "GD Silver")
if df_gd_origem.join(ucs_complementos, "id_uc", "left_anti").limit(1).count():
    raise ValueError("GD sem UC no cadastro publicado")
if df_gd_origem.filter((F.col("potencia_kwp") < 0) | F.isnan("potencia_kwp") |
    (F.abs(F.col("potencia_kwp")) == F.lit(float("inf")))).limit(1).count():
    raise ValueError("Potência de GD inválida")
df_gd_uc = df_gd_origem.select(
    "id_uc", F.col("potencia_kwp").cast("double").alias("potencia_kwp"),
    F.when(F.coalesce(F.col("_conexao_futura"), F.lit(False)) |
           F.coalesce(F.col("_conexao_sentinela"), F.lit(False)) |
           (F.col("data_conexao") > data_referencia()), F.lit(None))
     .otherwise(F.col("data_conexao")).cast("date").alias("data_conexao"),
    "modalidade",
    (F.coalesce(F.col("_conexao_futura"), F.lit(False)) |
     F.coalesce(F.col("data_conexao") > data_referencia(), F.lit(False))).alias("flag_conexao_futura"),
    F.coalesce(F.col("_conexao_sentinela"), F.lit(False)).alias("flag_conexao_sentinela"),
    instante_referencia().alias("data_processamento_gold")
)

# Não transformar texto livre em texto "anonimizado" por regex. Somente valores
# inteiros da lista explícita são classificáveis; demais conteúdos não são exportados.
ACOES_TRATATIVAS_PERMITIDAS = ["ABERTURA", "ENCERRAMENTO", "TRANSFERENCIA", "RETORNO", "ATUALIZACAO"]
CANAIS_TRATATIVAS_PERMITIDOS = ["TELEFONE", "WHATSAPP", "EMAIL", "CHAT", "PRESENCIAL", "APP", "SITE", "OUVIDORIA", "CALL CENTER"]
df_protocolos_origem = read_silver("atendimentos")
assert_no_nulls(df_protocolos_origem, ["protocolo", "id_uc"], "protocolos Silver")
assert_unique(df_protocolos_origem, ["protocolo"], "protocolos Silver")
df_hist_origem = read_silver("atendimentos_historico")
assert_columns_exist(df_hist_origem, ["protocolo", "posicao_historico", "sequencia", "data",
    "acao", "canal", "_data_invalida", "_sequencia_invalida"], "tratativas Silver")
assert_no_nulls(df_hist_origem, ["protocolo", "posicao_historico"], "tratativas Silver")
assert_unique(df_hist_origem, ["protocolo", "posicao_historico"], "tratativas Silver")
if df_hist_origem.join(df_protocolos_origem.select("protocolo"), "protocolo", "left_anti").limit(1).count():
    raise ValueError("Tratativa sem protocolo publicado")
df_protocolos_elegiveis = df_protocolos_origem.filter(uc_real()).select("protocolo", "id_uc", "qtd_tratativas")
if df_protocolos_elegiveis.join(ucs_complementos, "id_uc", "left_anti").limit(1).count():
    raise ValueError("Protocolo sem UC publicada")
df_hist_elegivel = df_hist_origem.join(df_protocolos_elegiveis.select("protocolo", "id_uc"), "protocolo", "inner")
if df_hist_elegivel.filter(F.col("posicao_historico") < 1).limit(1).count():
    raise ValueError("Posição de tratativa inválida")
acao = F.upper(F.trim(F.col("acao")))
canal = F.upper(F.trim(F.col("canal")))
df_tratativas_uc = df_hist_elegivel.select(
    "protocolo", "id_uc", F.col("posicao_historico").cast("int").alias("posicao_historico"),
    F.when(F.coalesce(F.col("_sequencia_invalida"), F.lit(False)), F.lit(None))
     .otherwise(F.col("sequencia")).cast("int").alias("sequencia_origem"),
    F.when(F.coalesce(F.col("_data_invalida"), F.lit(False)) |
           (F.col("data") > instante_referencia()), F.lit(None))
     .otherwise(F.col("data")).cast("timestamp").alias("data_tratativa"),
    F.when(acao.isNull() | (acao == ""), "AUSENTE_NA_ORIGEM")
     .when(acao.isin(ACOES_TRATATIVAS_PERMITIDAS), acao)
     .otherwise("CONTEUDO_RESTRITO").alias("acao_categoria"),
    F.when(canal.isin(CANAIS_TRATATIVAS_PERMITIDOS), canal)
     .otherwise("NAO_CLASSIFICADO").alias("canal"),
    F.coalesce(F.col("_data_invalida"), F.lit(False)).alias("flag_data_invalida"),
    F.coalesce(F.col("data") > instante_referencia(), F.lit(False)).alias("flag_data_futura"),
    F.coalesce(F.col("_sequencia_invalida"), F.lit(False)).alias("flag_sequencia_invalida"),
    instante_referencia().alias("data_processamento_gold")
)
contagens_tratativas = df_tratativas_uc.groupBy("protocolo").agg(
    F.count(F.lit(1)).alias("qtd_publicada"), F.min("posicao_historico").alias("primeira_posicao"),
    F.max("posicao_historico").alias("ultima_posicao"))
conferencia_tratativas = df_protocolos_elegiveis.join(contagens_tratativas, "protocolo", "left")
if conferencia_tratativas.filter(F.col("qtd_tratativas").isNull() |
    (F.col("qtd_tratativas") < 0) |
    (F.col("qtd_tratativas") != F.coalesce(F.col("qtd_publicada"), F.lit(0))) |
    ((F.col("qtd_publicada") > 0) & ((F.col("primeira_posicao") != 1) |
     (F.col("ultima_posicao") != F.col("qtd_publicada"))))).limit(1).count():
    raise ValueError("Tratativas divergem da contagem ou das posições por protocolo")

# Revisão pontual D20: nome somente para cadastro restrito do atendimento.
# Não usar nome como chave, não gerar customer_reference a partir de CPF/nome.
nome_titular = F.trim(F.col("nome_titular"))
conflito_titular = F.coalesce(F.col("_nome_titular_conflitante"), F.lit(False))
df_cadastro_titular_uc = df_cadastro_origem.select(
    "id_uc",
    F.when(conflito_titular | nome_titular.isNull() | (nome_titular == ""), F.lit(None))
     .otherwise(nome_titular).alias("nome_exibicao_titular"),
    F.when(conflito_titular, "CONFLITO_NA_ORIGEM")
     .when(nome_titular.isNull() | (nome_titular == ""), "AUSENTE_NA_ORIGEM")
     .otherwise("DISPONIVEL").alias("situacao_identificacao"),
    instante_referencia().alias("data_processamento_gold")
)
TIPOS_COMPLEMENTOS_UC = {
    "gd_uc": {"id_uc":"string", "potencia_kwp":"double", "data_conexao":"date", "modalidade":"string",
        "flag_conexao_futura":"boolean", "flag_conexao_sentinela":"boolean", "data_processamento_gold":"timestamp"},
    "tratativas_uc": {"protocolo":"string", "id_uc":"string", "posicao_historico":"int", "sequencia_origem":"int",
        "data_tratativa":"timestamp", "acao_categoria":"string", "canal":"string", "flag_data_invalida":"boolean",
        "flag_data_futura":"boolean", "flag_sequencia_invalida":"boolean", "data_processamento_gold":"timestamp"},
    "cadastro_titular_uc": {"id_uc":"string", "nome_exibicao_titular":"string", "situacao_identificacao":"string",
        "data_processamento_gold":"timestamp"},
}
for nome, df, chave in [
    ("gd_uc", df_gd_uc, ["id_uc"]),
    ("tratativas_uc", df_tratativas_uc, ["protocolo", "posicao_historico"]),
    ("cadastro_titular_uc", df_cadastro_titular_uc, ["id_uc"]),
]:
    tipos = {campo.name: campo.dataType.simpleString() for campo in df.schema.fields}
    if len(df.columns) != len(tipos) or tipos != TIPOS_COMPLEMENTOS_UC[nome]:
        raise ValueError(f"Schema/tipos de {nome} divergem do contrato")
    assert_no_nulls(df, chave, nome)
    assert_unique(df, chave, nome)
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(origem_gold(nome)))
print("[PASS] Complementos UC 360 validados; titular somente no preparo restrito.")


In [ ]:
# COBERTURA POR DOMINIO — observação não comprova fechamento da origem
import json as _cov_json
import re as _cov_re
from datetime import datetime as _cov_datetime, timezone as _cov_timezone

DOMINIOS_COBERTURA = {
    "consumo": ("leituras", "mes_referencia", "COMPETENCIA", []),
    "faturamento": ("faturas", "mes_referencia", "COMPETENCIA", []),
    "atendimentos": ("atendimentos", "data_abertura", "ABERTURA", ["_abertura_futura"]),
    "ocorrencias": ("ocorrencias_rede", "data_abertura", "ABERTURA", ["_abertura_futura"]),
    "medidores": ("medidores", "data_instalacao", "INSTALACAO_CADASTRAL", ["_instalacao_futura", "_instalacao_sentinela"]),
    "gd": ("geracao_distribuida", "data_conexao", "CONEXAO_CADASTRAL", ["_conexao_futura", "_conexao_sentinela"]),
}

def competencia_cobertura(valor):
    if not isinstance(valor, str) or not _cov_re.fullmatch(r"[0-9]{4}-(0[1-9]|1[0-2])", valor):
        raise ValueError("Competência de cobertura inválida; usar YYYY-MM")
    try:
        return _cov_datetime.strptime(valor, "%Y-%m").date()
    except ValueError:
        raise ValueError("Competência de cobertura inválida") from None

def ler_declaracoes_cobertura(texto, referencia):
    if not texto.strip():
        return []
    if len(texto.encode("utf-8")) > 65536:
        raise ValueError("Declarações de cobertura excedem o limite de 64 KiB")
    try:
        def unico(pares):
            objeto = {}
            for chave, valor in pares:
                if chave in objeto:
                    raise ValueError("Campo duplicado na declaração")
                objeto[chave] = valor
            return objeto
        pacote = _cov_json.loads(texto, object_pairs_hook=unico)
    except (ValueError, TypeError):
        raise ValueError("JSON de cobertura inválido") from None
    campos = {"schema_version", "run_id_silver", "lote_bronze_silver", "declaracoes"}
    if not isinstance(pacote, dict) or set(pacote) != campos:
        raise ValueError("Envelope de cobertura incompatível")
    if (pacote["schema_version"] not in {"gold-cobertura-v1", "gold-cobertura-v2"} or
        pacote["run_id_silver"] != run_id_silver or pacote["lote_bronze_silver"] != lote_bronze_silver):
        raise ValueError("Declarações de cobertura não pertencem à versão ou ao lote fixado")
    declaracoes = pacote["declaracoes"]
    if not isinstance(declaracoes, list) or len(declaracoes) > 500:
        raise ValueError("Lista de declarações de cobertura inválida")
    saida = []
    for item in declaracoes:
        esperados = {"dominio", "inicio", "fim", "estado", "recorte", "data_corte", "referencia_evidencia"}
        if pacote["schema_version"] == "gold-cobertura-v2":
            esperados = esperados | {"natureza_certificacao", "reconciliacao_extracao"}
        if not isinstance(item, dict) or set(item) != esperados or any(
            not isinstance(v, str) for k, v in item.items() if k != "reconciliacao_extracao"
        ):
            raise ValueError("Declaração de cobertura tem campos inválidos")
        item = dict(item)
        item.setdefault("natureza_certificacao", "FECHAMENTO_CONTABIL")
        item.setdefault("reconciliacao_extracao", None)
        natureza_certificacao = item["natureza_certificacao"]
        if natureza_certificacao not in {"EXTRACAO", "FECHAMENTO_CONTABIL"}:
            raise ValueError("Natureza de certificação inválida")
        rec = item["reconciliacao_extracao"]
        if natureza_certificacao == "EXTRACAO":
            campos_rec = {"registros_origem", "registros_aceitos", "registros_excluidos", "hash_inventario"}
            if not isinstance(rec, dict) or set(rec) != campos_rec:
                raise ValueError("Extração exige inventário e contagens reconciliadas")
            if any(type(rec[k]) is not int or rec[k] < 0 for k in campos_rec - {"hash_inventario"}):
                raise ValueError("Contagens da extração devem ser inteiros não negativos")
            if (not isinstance(rec["hash_inventario"], str) or
                not _cov_re.fullmatch(r"[0-9a-f]{64}", rec["hash_inventario"]) or
                rec["registros_origem"] != rec["registros_aceitos"] + rec["registros_excluidos"]):
                raise ValueError("Inventário ou reconciliação da extração inválidos")
            if item["recorte"] != "DOMINIO":
                raise ValueError("Certificação de extração exige contagem global do domínio")
            if item["estado"] == "FECHADO" and rec["registros_excluidos"]:
                raise ValueError("Extração com exclusões deve ser PARCIAL")
        elif rec is not None:
            raise ValueError("Fechamento contábil não aceita reconciliação de extração")
        if item["dominio"] not in DOMINIOS_COBERTURA or item["estado"] not in {"FECHADO", "PARCIAL", "AUSENTE"}:
            raise ValueError("Domínio ou estado de cobertura inválido")
        if item["dominio"] in {"medidores", "gd"} and item["estado"] == "FECHADO":
            raise ValueError("Cadastro atual não certifica cobertura histórica mensal")
        inicio, fim = competencia_cobertura(item["inicio"]), competencia_cobertura(item["fim"])
        if inicio > fim:
            raise ValueError("Intervalo de cobertura invertido")
        if item["recorte"] != "DOMINIO" and not (item["dominio"] == "ocorrencias" and item["recorte"].startswith("ALIMENTADOR:") and item["recorte"][12:].strip()):
            raise ValueError("Recorte de cobertura inválido")
        if not _cov_re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_./:#@-]{0,255}", item["referencia_evidencia"]):
            raise ValueError("Referência de evidência ausente ou inválida")
        try:
            corte = _cov_datetime.fromisoformat(item["data_corte"].replace("Z", "+00:00"))
        except ValueError:
            raise ValueError("Data de corte de cobertura inválida") from None
        if corte.tzinfo is None or corte.utcoffset() is None:
            raise ValueError("Data de corte de cobertura exige offset")
        corte = corte.astimezone(_cov_timezone.utc)
        if corte > referencia:
            raise ValueError("Data de corte posterior à referência da extração")
        proximo_mes = fim.replace(year=fim.year + 1, month=1) if fim.month == 12 else fim.replace(month=fim.month + 1)
        if item["estado"] == "FECHADO" and corte < _cov_datetime.combine(proximo_mes, _cov_datetime.min.time(), tzinfo=_cov_timezone.utc):
            raise ValueError("Fechamento declarado antes do término da competência")
        saida.append(dict(item, data_corte=corte.isoformat()))
    return saida

def construir_cobertura_dominios(inicio_param, fim_param, declaracoes_json):
    # Coletar ISO com offset: timestamps Python coletados podem usar o fuso do host.
    referencia_iso = spark.range(1).select(F.date_format(instante_referencia(), "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX").alias("referencia")).first()["referencia"]
    referencia = _cov_datetime.fromisoformat(referencia_iso.replace("Z", "+00:00")).astimezone(_cov_timezone.utc)
    declaracoes = ler_declaracoes_cobertura(declaracoes_json, referencia)
    if bool(inicio_param) != bool(fim_param):
        raise ValueError("Informar início e fim da janela de cobertura em conjunto")
    cadastro = read_silver("unidades_consumidoras").filter(uc_real()).select("id_uc", "alimentador")
    assert_no_nulls(cadastro, ["id_uc", "alimentador"], "cadastro da cobertura")
    assert_unique(cadastro, ["id_uc"], "cadastro da cobertura")
    if cadastro.filter(F.trim("alimentador") == "").limit(1).count():
        raise ValueError("Alimentador vazio no cadastro da cobertura")
    observacoes, escopos = [], []
    for dominio, (tabela, campo, natureza, flags) in DOMINIOS_COBERTURA.items():
        fonte = read_silver(tabela).filter(uc_real())
        assert_columns_exist(fonte, ["id_uc", campo] + flags, tabela)
        if dominio == "ocorrencias":
            fonte = fonte.filter(F.col("_id_uc_ausente") == False)
        if fonte.select("id_uc").distinct().join(cadastro.select("id_uc"), "id_uc", "left_anti").limit(1).count():
            raise ValueError("Fonte de cobertura sem UC publicada")
        if natureza == "COMPETENCIA":
            valor = F.col(campo)
            valido = valor.rlike(r"^[0-9]{4}-(0[1-9]|1[0-2])$") & (F.to_date(F.concat(valor, F.lit("-01"))) <= data_referencia())
        else:
            valor = F.date_format(campo, "yyyy-MM")
            valido = F.col(campo).isNotNull() & (F.col(campo) <= instante_referencia())
        for flag in flags:
            valido = valido & ~F.coalesce(F.col(flag), F.lit(False))
        base = fonte.select("id_uc", F.when(valido, valor).alias("competencia"))
        global_obs = base.select(F.lit(dominio).alias("dominio"), "competencia", F.lit("DOMINIO").alias("recorte"))
        observacoes.append(global_obs)
        escopos.append(spark.range(1).select(F.lit(dominio).alias("dominio"), F.lit("DOMINIO").alias("recorte"), F.lit(natureza).alias("natureza_temporal")))
        if dominio == "ocorrencias":
            observacoes.append(base.join(cadastro, "id_uc").select(F.lit(dominio).alias("dominio"), "competencia", F.concat(F.lit("ALIMENTADOR:"), F.col("alimentador")).alias("recorte")))
            escopos.append(cadastro.select(F.lit(dominio).alias("dominio"), F.concat(F.lit("ALIMENTADOR:"), F.col("alimentador")).alias("recorte"), F.lit(natureza).alias("natureza_temporal")).distinct())
    obs = observacoes[0]
    for df in observacoes[1:]:
        obs = obs.unionByName(df)
    grupos = escopos[0]
    for df in escopos[1:]:
        grupos = grupos.unionByName(df)
    if inicio_param:
        inicio, fim = competencia_cobertura(inicio_param), competencia_cobertura(fim_param)
        origem_janela = "EXPLICITA"
    else:
        limites = obs.filter(F.col("dominio").isin("consumo", "faturamento", "atendimentos", "ocorrencias") & F.col("competencia").isNotNull()).agg(F.min("competencia").alias("inicio"), F.max("competencia").alias("fim")).first()
        if limites["inicio"] is None:
            raise ValueError("Sem fatos datados: informar a janela esperada da cobertura")
        inicio, fim = competencia_cobertura(limites["inicio"]), competencia_cobertura(limites["fim"])
        origem_janela = "LIMITES_OBSERVADOS_NAO_CERTIFICADOS"
    if inicio > fim or (fim.year-inicio.year)*12 + fim.month-inicio.month > 239:
        raise ValueError("Janela de cobertura invertida ou superior a 240 meses")
    if inicio > referencia.date() or fim > referencia.date().replace(day=1):
        raise ValueError("Janela de cobertura posterior à referência da extração")
    meses = spark.range(1).select(F.explode(F.sequence(F.lit(inicio), F.lit(fim), F.expr("interval 1 month"))).alias("mes")).select(F.date_format("mes", "yyyy-MM").alias("competencia"))
    grade = grupos.crossJoin(meses)
    # Contagem do snapshot fixado; o corte externo qualifica a evidência, não filtra estes registros.
    contagens = obs.filter(F.col("competencia").isNotNull()).groupBy("dominio", "competencia", "recorte").count().withColumnRenamed("count", "qtd_registros")
    sem_competencia = obs.groupBy("dominio", "recorte").agg(F.sum(F.when(F.col("competencia").isNull(), 1).otherwise(0)).cast("long").alias("qtd_sem_competencia"))
    cobertura = grade.join(contagens, ["dominio", "competencia", "recorte"], "left").join(sem_competencia, ["dominio", "recorte"], "left").fillna(0, subset=["qtd_registros", "qtd_sem_competencia"])
    frames_declarados = []
    for item in declaracoes:
        if competencia_cobertura(item["inicio"]) < inicio or competencia_cobertura(item["fim"]) > fim:
            raise ValueError("Declaração fora da janela de cobertura")
        rec = item["reconciliacao_extracao"]
        if rec is not None:
            observados = contagens.filter((F.col("dominio") == item["dominio"]) &
                (F.col("recorte") == "DOMINIO") & F.col("competencia").between(item["inicio"], item["fim"])).agg(
                    F.coalesce(F.sum("qtd_registros"), F.lit(0)).alias("n")).first()["n"]
            if observados != rec["registros_aceitos"]:
                raise ValueError("Contagem declarada não confere com o snapshot Silver")
        frames_declarados.append(spark.range(1).select(
            F.lit(item["dominio"]).alias("dominio"), F.lit(item["recorte"]).alias("recorte"),
            F.explode(F.sequence(F.lit(competencia_cobertura(item["inicio"])), F.lit(competencia_cobertura(item["fim"])), F.expr("interval 1 month"))).alias("mes"),
            F.lit(item["estado"]).alias("estado_cobertura"), F.lit(item["data_corte"]).cast("timestamp").alias("data_corte_evidencia"),
            F.lit(item["referencia_evidencia"]).alias("referencia_evidencia"),
            F.lit(item["natureza_certificacao"]).alias("natureza_certificacao"),
            F.lit(_cov_json.dumps(rec, sort_keys=True) if rec is not None else None).cast("string").alias("reconciliacao_extracao_json")
        ).withColumn("competencia", F.date_format("mes", "yyyy-MM")).drop("mes"))
    if frames_declarados:
        declarados = frames_declarados[0]
        for df in frames_declarados[1:]:
            declarados = declarados.unionByName(df)
        assert_unique(declarados, ["dominio", "competencia", "recorte"], "declarações de cobertura")
        if declarados.join(grade, ["dominio", "competencia", "recorte"], "left_anti").limit(1).count():
            raise ValueError("Declaração fora da janela ou de recorte publicado")
        # O estado total e os recortes componentes não podem certificar afirmações opostas.
        totais = declarados.filter((F.col("dominio") == "ocorrencias") & (F.col("recorte") == "DOMINIO")).select(
            "dominio", "competencia", F.col("estado_cobertura").alias("estado_total"))
        detalhes = declarados.filter(F.col("recorte").startswith("ALIMENTADOR:")).join(totais, ["dominio", "competencia"], "inner")
        if detalhes.filter(
            ((F.col("estado_total") == "FECHADO") & F.col("estado_cobertura").isin("AUSENTE", "PARCIAL")) |
            ((F.col("estado_total") == "AUSENTE") & F.col("estado_cobertura").isin("FECHADO", "PARCIAL"))
        ).limit(1).count():
            raise ValueError("Declarações de domínio e alimentador são contraditórias")
        cobertura = cobertura.join(declarados, ["dominio", "competencia", "recorte"], "left")
    else:
        cobertura = cobertura.withColumn("natureza_certificacao", F.lit(None).cast("string")).withColumn("reconciliacao_extracao_json", F.lit(None).cast("string")).withColumn("estado_cobertura", F.lit(None).cast("string")).withColumn("data_corte_evidencia", F.lit(None).cast("timestamp")).withColumn("referencia_evidencia", F.lit(None).cast("string"))
    if cobertura.filter(((F.col("estado_cobertura") == "AUSENTE") & (F.col("qtd_registros") > 0)) |
        ((F.col("estado_cobertura") == "FECHADO") & (F.col("qtd_sem_competencia") > 0))).limit(1).count():
        raise ValueError("Declaração de cobertura contradiz as observações da origem")
    cobertura = (cobertura.withColumn("estado_cobertura", F.coalesce("estado_cobertura", F.lit("NAO_CERTIFICADA")))
        .withColumn("natureza_certificacao", F.coalesce("natureza_certificacao", F.lit("NAO_CERTIFICADA")))
        .withColumn("zero_registros_confirmado", (F.col("estado_cobertura") == "FECHADO") & (F.col("qtd_registros") == 0))
        .withColumn("motivo", F.when(F.col("estado_cobertura") == "NAO_CERTIFICADA", "SEM_EVIDENCIA_DE_COBERTURA")
            .when(F.col("zero_registros_confirmado"), "ZERO_REGISTROS_DECLARADO")
            .otherwise(F.concat(F.col("estado_cobertura"), F.lit("_DECLARADO"))))
        .withColumn("origem_janela", F.lit(origem_janela))
        .withColumn("run_id_silver", F.lit(run_id_silver)).withColumn("lote_bronze_silver", F.lit(lote_bronze_silver))
        .withColumn("data_processamento_gold", instante_referencia())
        .select("dominio", "competencia", "recorte", "natureza_temporal", "estado_cobertura", "qtd_registros",
            "qtd_sem_competencia", "zero_registros_confirmado", "data_corte_evidencia", "referencia_evidencia", "motivo",
            "origem_janela", "natureza_certificacao", "reconciliacao_extracao_json", "run_id_silver", "lote_bronze_silver", "data_processamento_gold"))
    assert_no_nulls(cobertura, ["dominio", "competencia", "recorte", "estado_cobertura", "qtd_registros", "zero_registros_confirmado"], "cobertura Gold")
    assert_unique(cobertura, ["dominio", "competencia", "recorte"], "cobertura Gold")
    return cobertura

dbutils.widgets.text("COBERTURA_INICIO", "", "Primeira competência esperada, YYYY-MM")
dbutils.widgets.text("COBERTURA_FIM", "", "Última competência esperada, YYYY-MM")
dbutils.widgets.text("COBERTURA_DECLARACOES_JSON", "", "Declarações autorizadas de cobertura do lote, opcionais")
df_cobertura_dominios = construir_cobertura_dominios(
    dbutils.widgets.get("COBERTURA_INICIO").strip(), dbutils.widgets.get("COBERTURA_FIM").strip(),
    dbutils.widgets.get("COBERTURA_DECLARACOES_JSON"))
TIPOS_COBERTURA_DOMINIOS = {
    "dominio":"string", "competencia":"string", "recorte":"string", "natureza_temporal":"string",
    "estado_cobertura":"string", "qtd_registros":"bigint", "qtd_sem_competencia":"bigint",
    "zero_registros_confirmado":"boolean", "data_corte_evidencia":"timestamp", "referencia_evidencia":"string",
    "motivo":"string", "origem_janela":"string", "run_id_silver":"string", "lote_bronze_silver":"string",
    "data_processamento_gold":"timestamp", "natureza_certificacao":"string", "reconciliacao_extracao_json":"string",
}
if {campo.name:campo.dataType.simpleString() for campo in df_cobertura_dominios.schema.fields} != TIPOS_COBERTURA_DOMINIOS:
    raise ValueError("Schema/tipos da cobertura divergem do contrato")
(df_cobertura_dominios.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.cobertura_dominios"))
print("[PASS] Cobertura mensal observada; certificação da extração separada do fechamento contábil.")


## Diagnóstico candidato de preço observado e volume medido

A rotina abaixo compara as mesmas UCs em cada recorte nas duas janelas. Uma UC só integra a base comum quando tem faturamento e todas as leituras observadas válidas em todos os meses solicitados, e o consumo acumulado é positivo nas duas janelas. Ausência de um mês ou mudança do recorte não é completada com média ou zero.

Para cada UC comum, R é o faturamento e Q o consumo medido comparável de cada janela, P=R/Q. Calculam-se P0ΔQ (volume), Q0ΔP (preço observado) e ΔPΔQ (interação) antes da soma por município/classe ou outras dimensões autorizadas. A mudança do peso entre UCs não vira efeito de preço da UC. Mudanças de bandeira, composição interna e demais componentes da fatura ainda podem alterar P: não é reajuste homologado nem prova de causalidade.

A variação total é reconciliada com as contribuições de entrada e saída de faturamento no recorte, e as parcelas sem leitura e demais lacunas fora da base comum. Essas entradas/saídas não provam ativação/desativação de cliente. Sem UC comum, os efeitos de preço/volume permanecem nulos, enquanto as contribuições de composição podem ser informadas sob cobertura certificada.

Exige cobertura com run Silver e lote Bronze iguais aos esperados da execução e fechamento declarado de consumo/faturamento em todos os meses das duas janelas. Sem cobertura, os valores derivados são nulos e o estado é `COBERTURA_NAO_CERTIFICADA`. Chave UC/mês duplicada, valor não finito, flags/medidas inconsistentes e falha de reconciliação interrompem o cálculo. O vínculo entre a série e as dimensões é validado antes do join para impedir multiplicação de fatos.

A análise candidata é materializada em `decomposicao_receita` no preparo, promovida e exportada com as demais projeções. Sem configuração, publica tabela vazia tipada e estado `NAO_CONFIGURADA` no controle/manifesto. Sua aprovação e integração ao Backend/diagnóstico permanecem pendentes. A fonte não contém energia faturada nem valor efetivamente recebido: `valor_energia` é monetário, `consumo_kwh` é medido e status/data não comprovam valor pago. O estudo candidato de recuperação só libera médias históricas por inspeção após validação temporal, com intervalos e limitação à população elegível; não fabrica horizonte mensal nem comprova resultado futuro. As declarações de cobertura também precisam de evidência externa, sem fechamento automático.


In [ ]:
# PREÇO-VOLUME OBSERVADO — mesmas UCs e janelas completas no recorte
def decompor_preco_volume(base, cobertura, inicio_p0, fim_p0, inicio_p1, fim_p1,
                         dimensoes=("municipio", "bairro", "classe_consumo"), *,
                         run_id_silver_esperado, lote_bronze_silver_esperado):
    """Compara a mesma UC no recorte; retorna o DataFrame para validação e publicação pelo notebook.

    A entrada tem grão UC/competência, com as medidas de consumo_mensal_uc e
    dimensões da mesma execução. Preço observado usa consumo medido, não kWh
    faturado nem tarifa homologada. Entradas/saídas são do faturamento no recorte.
    """
    import re
    from datetime import date

    def mes(valor):
        if not isinstance(valor, str) or not re.fullmatch(r"[0-9]{4}-(0[1-9]|1[0-2])", valor):
            raise ValueError("Período preço-volume exige YYYY-MM")
        return date.fromisoformat(valor + "-01")

    janelas = [(mes(inicio_p0), mes(fim_p0)), (mes(inicio_p1), mes(fim_p1))]
    if any(a > b for a, b in janelas) or janelas[0][1] >= janelas[1][0]:
        raise ValueError("Janelas devem ser ordenadas, sem sobreposição")
    permitidas = {"municipio", "bairro", "classe_consumo", "subgrupo_tarifario", "bandeira_tarifaria", "possui_gd"}
    if (not isinstance(dimensoes, (tuple, list)) or not dimensoes or
        any(not isinstance(d, str) or d not in permitidas for d in dimensoes) or
        len(dimensoes) != len(set(dimensoes))):
        raise ValueError("Dimensões preço-volume inválidas")
    dims = list(dimensoes)
    medidas = ["receita_reais", "receita_sem_leitura_reais",
               "receita_base_comparavel_reais", "consumo_base_comparavel_kwh"]
    assert_columns_exist(base, ["id_uc", "competencia", "tem_fatura"] + dims + medidas, "base preço-volume")
    assert_no_nulls(base, ["id_uc", "competencia", "tem_fatura"] + dims, "base preço-volume")
    assert_unique(base, ["id_uc", "competencia"], "base preço-volume")
    invalida = F.lit(False)
    for medida in medidas:
        numero = F.col(medida).cast("double")
        invalida = invalida | F.isnan(numero) | (F.abs(numero) == F.lit(float("inf")))
    inconsistente = (F.col("tem_fatura") & (F.col("receita_reais").isNull() | F.col("receita_sem_leitura_reais").isNull()))
    inconsistente = inconsistente | (~F.col("tem_fatura") & (F.col("receita_reais").isNotNull() | F.col("receita_sem_leitura_reais").isNotNull()))
    r, q = F.col("receita_base_comparavel_reais"), F.col("consumo_base_comparavel_kwh")
    inconsistente = inconsistente | (r.isNull() != q.isNull()) | (q < 0)
    inconsistente = inconsistente | (r.isNotNull() & (~F.col("tem_fatura") |
        (F.abs(r.cast("double")-F.col("receita_reais").cast("double")) > 0.000001)))
    if base.filter(invalida | inconsistente).limit(1).count():
        raise ValueError("Medida não finita ou presença/valor inconsistente na base preço-volume")
    assert_columns_exist(cobertura, ["dominio", "competencia", "recorte", "estado_cobertura",
                                    "qtd_sem_competencia", "run_id_silver", "lote_bronze_silver"], "cobertura preço-volume")
    assert_unique(cobertura, ["dominio", "competencia", "recorte"], "cobertura preço-volume")
    for identidade in (run_id_silver_esperado, lote_bronze_silver_esperado):
        if not isinstance(identidade, str) or not identidade.strip():
            raise ValueError("Identidade esperada da cobertura ausente")
    divergente = (F.col("run_id_silver").isNull() | F.col("lote_bronze_silver").isNull() |
        (F.col("run_id_silver") != F.lit(run_id_silver_esperado)) |
        (F.col("lote_bronze_silver") != F.lit(lote_bronze_silver_esperado)))
    if cobertura.filter(divergente).limit(1).count():
        raise ValueError("Cobertura não pertence ao run/lote esperado")
    certificados = True
    meses_esperados = []
    for a, b in janelas:
        esperados = (b.year-a.year)*12 + b.month-a.month + 1
        meses_esperados.append(esperados)
        for dominio in ["consumo", "faturamento"]:
            completos = cobertura.filter((F.col("dominio") == dominio) & (F.col("recorte") == "DOMINIO") &
                F.col("competencia").between(a.strftime("%Y-%m"), b.strftime("%Y-%m")) &
                (F.col("estado_cobertura") == "FECHADO") & (F.col("qtd_sem_competencia") == 0))
            certificados = certificados and completos.count() == esperados

    por_uc = []
    for idx, (inicio, fim) in enumerate([(inicio_p0, fim_p0), (inicio_p1, fim_p1)]):
        por_uc.append(base.filter(F.col("competencia").between(inicio, fim)).groupBy("id_uc", *dims).agg(
            *[F.sum(c).alias(f"{c}_p{idx}") for c in medidas],
            F.sum(F.when(r.isNotNull() & q.isNotNull(), 1).otherwise(0)).alias(f"meses_comparaveis_p{idx}"),
            F.sum(F.when(F.col("tem_fatura"), 1).otherwise(0)).alias(f"meses_faturados_p{idx}")))
    pares = por_uc[0].join(por_uc[1], ["id_uc"] + dims, "full")
    r0, r1 = (F.col(f"receita_base_comparavel_reais_p{i}").cast("double") for i in (0, 1))
    q0, q1 = (F.col(f"consumo_base_comparavel_kwh_p{i}").cast("double") for i in (0, 1))
    comum = ((F.col("meses_comparaveis_p0") == meses_esperados[0]) &
             (F.col("meses_comparaveis_p1") == meses_esperados[1]) & (q0 > 0) & (q1 > 0))
    comum = F.coalesce(comum, F.lit(False))
    faturada0, faturada1 = (F.coalesce(F.col(f"meses_faturados_p{i}"), F.lit(0)) > 0 for i in (0, 1))
    entrada, saida = ~faturada0 & faturada1, faturada0 & ~faturada1
    fora = ~comum & ~entrada & ~saida
    # Zero aqui representa ausência de fatura na partição, somente sob cobertura certificada.
    total0, total1 = (F.coalesce(F.col(f"receita_reais_p{i}").cast("double"), F.lit(0.)) for i in (0, 1))
    sem0, sem1 = (F.coalesce(F.col(f"receita_sem_leitura_reais_p{i}").cast("double"), F.lit(0.)) for i in (0, 1))
    variacao, delta_sem = total1-total0, sem1-sem0
    p0, p1 = F.when(comum, r0/q0), F.when(comum, r1/q1)
    # Calcular os efeitos por UC antes da soma evita atribuir troca de UCs a preço.
    pares = (pares.withColumn("_comum", comum).withColumn("_entrada", entrada).withColumn("_saida", saida)
        .withColumn("_volume", F.when(comum, p0*(q1-q0)))
        .withColumn("_preco", F.when(comum, q0*(p1-p0)))
        .withColumn("_interacao", F.when(comum, (p1-p0)*(q1-q0)))
        .withColumn("_entradas", F.when(entrada, variacao).otherwise(0.))
        .withColumn("_saidas", F.when(saida, variacao).otherwise(0.))
        .withColumn("_sem", F.when(fora, delta_sem).otherwise(0.))
        .withColumn("_outros", F.when(fora, variacao-delta_sem).otherwise(0.))
        .withColumn("_total0", total0).withColumn("_total1", total1))
    efeito_map = {"efeito_volume_reais":"_volume", "efeito_preco_observado_reais":"_preco",
        "efeito_interacao_reais":"_interacao", "variacao_entradas_reais":"_entradas",
        "variacao_saidas_reais":"_saidas", "variacao_sem_leitura_fora_base_reais":"_sem",
        "variacao_outros_fora_base_reais":"_outros"}
    resultado = pares.groupBy(*dims).agg(
        F.sum(F.when(F.col("_comum"), 1).otherwise(0)).cast("long").alias("qtd_ucs_base_comum"),
        F.sum(F.when(F.col("_entrada"), 1).otherwise(0)).cast("long").alias("qtd_ucs_entrantes"),
        F.sum(F.when(F.col("_saida"), 1).otherwise(0)).cast("long").alias("qtd_ucs_saintes"),
        F.count(F.lit(1)).alias("qtd_ucs_observadas"),
        F.sum("_total0").alias("faturamento_p0_reais"), F.sum("_total1").alias("faturamento_p1_reais"),
        *[F.sum(origem).alias(destino) for destino, origem in efeito_map.items()])
    resultado = resultado.withColumn("estado_decomposicao",
        F.when(~F.lit(certificados), "COBERTURA_NAO_CERTIFICADA")
         .when(F.col("qtd_ucs_base_comum") > 0, "CALCULADA_BASE_COMUM").otherwise("SEM_BASE_COMUM"))
    resultado = resultado.withColumn("variacao_faturamento_total_reais",
        F.col("faturamento_p1_reais")-F.col("faturamento_p0_reais"))
    soma = F.lit(0.)
    for nome in efeito_map:
        soma = soma + F.coalesce(F.col(nome), F.lit(0.))
    resultado = resultado.withColumn("residuo_reconciliacao_reais", F.col("variacao_faturamento_total_reais")-soma)
    if resultado.filter(F.col("residuo_reconciliacao_reais").isNull() |
        F.isnan("residuo_reconciliacao_reais") |
        (F.abs("residuo_reconciliacao_reais") > F.greatest(F.lit(0.000001),
            F.abs("variacao_faturamento_total_reais")*F.lit(1e-9)))).limit(1).count():
        raise ValueError("Decomposição por UC não reconcilia o faturamento total")
    # Sem certificação, zero de linhas não autoriza zero financeiro ou efeitos.
    for nome in list(efeito_map) + ["faturamento_p0_reais", "faturamento_p1_reais",
                                   "variacao_faturamento_total_reais", "residuo_reconciliacao_reais"]:
        resultado = resultado.withColumn(nome, F.when(F.lit(certificados), F.col(nome)))
    for nome, valor in [("inicio_p0", inicio_p0), ("fim_p0", fim_p0), ("inicio_p1", inicio_p1), ("fim_p1", fim_p1)]:
        resultado = resultado.withColumn(nome, F.lit(valor))
    return resultado.withColumn("metodo", F.lit("COORTE_UC_PRECO_OBSERVADO_VOLUME_INTERACAO_V2"))


def preparar_base_preco_volume(mensal, cadastro_uc_mes):
    """Associa dimensões do mesmo snapshot sem multiplicar os fatos UC/mês."""
    dims = cadastro_uc_mes.select("id_uc", F.col("mes_referencia").alias("competencia"), "municipio", "bairro",
        F.col("classe").alias("classe_consumo"), "subgrupo_tarifario", "bandeira_tarifaria", "possui_gd")
    assert_unique(mensal, ["id_uc", "competencia"], "mensal preço-volume")
    assert_unique(dims, ["id_uc", "competencia"], "dimensões preço-volume")
    if mensal.select("id_uc", "competencia").join(dims.select("id_uc", "competencia"),
        ["id_uc", "competencia"], "left_anti").limit(1).count():
        raise ValueError("UC/competência sem dimensões no snapshot preço-volume")
    return mensal.join(dims, ["id_uc", "competencia"], "inner")


# Contrato físico da análise de duas janelas, separado dos fatos mensais.
TIPOS_DECOMPOSICAO_RECEITA = {
    "municipio": "string", "bairro": "string", "classe_consumo": "string",
    "qtd_ucs_base_comum": "bigint", "qtd_ucs_entrantes": "bigint",
    "qtd_ucs_saintes": "bigint", "qtd_ucs_observadas": "bigint",
    "faturamento_p0_reais": "double", "faturamento_p1_reais": "double",
    "efeito_volume_reais": "double", "efeito_preco_observado_reais": "double",
    "efeito_interacao_reais": "double", "variacao_entradas_reais": "double",
    "variacao_saidas_reais": "double", "variacao_sem_leitura_fora_base_reais": "double",
    "variacao_outros_fora_base_reais": "double", "estado_decomposicao": "string",
    "variacao_faturamento_total_reais": "double", "residuo_reconciliacao_reais": "double",
    "inicio_p0": "string", "fim_p0": "string", "inicio_p1": "string", "fim_p1": "string",
    "metodo": "string", "run_id_silver": "string", "lote_bronze_silver": "string",
    "data_processamento_gold": "timestamp",
}
METODO_PRECO_VOLUME = "COORTE_UC_PRECO_OBSERVADO_VOLUME_INTERACAO_V2"
LIMITACOES_MEDIDAS_GOLD = {
    "energia_faturada_kwh": "SEM_MEDIDA_COMPROVADA",
    "valor_recebido_reais": "SEM_MEDIDA_COMPROVADA",
    "energia_recuperavel_kwh": "ESTIMATIVA_HISTORICA_CONDICIONAL",
    "fechamento_mensal": "EXIGE_DECLARACAO_VINCULADA",
}


def configurar_preco_volume(janelas):
    import re
    from datetime import date
    if not isinstance(janelas, (tuple, list)) or len(janelas) != 4 or any(not isinstance(v, str) for v in janelas):
        raise ValueError("Configuração preço-volume exige quatro competências textuais")
    if not any(janelas):
        return {"estado": "NAO_CONFIGURADA", "metodo": METODO_PRECO_VOLUME, "janelas": None}
    if not all(janelas):
        raise ValueError("Preço-volume exige as quatro competências das duas janelas")
    if any(not re.fullmatch(r"[0-9]{4}-(0[1-9]|1[0-2])", v) for v in janelas):
        raise ValueError("Configuração preço-volume exige YYYY-MM")
    datas = [date.fromisoformat(v + "-01") for v in janelas]
    if datas[0] > datas[1] or datas[2] > datas[3] or datas[1] >= datas[2]:
        raise ValueError("Janelas preço-volume invertidas ou sobrepostas")
    return {"estado": "CONFIGURADA", "metodo": METODO_PRECO_VOLUME,
            "janelas": dict(zip(["inicio_p0", "fim_p0", "inicio_p1", "fim_p1"], janelas))}


def validar_configuracao_preco_volume(config):
    if not isinstance(config, dict) or set(config) != {"estado", "metodo", "janelas"}:
        raise ValueError("Contrato de configuração preço-volume inválido")
    if config["estado"] == "NAO_CONFIGURADA":
        esperado = configurar_preco_volume(("", "", "", ""))
    elif config["estado"] == "CONFIGURADA":
        janelas = config["janelas"]
        nomes = ["inicio_p0", "fim_p0", "inicio_p1", "fim_p1"]
        if not isinstance(janelas, dict) or set(janelas) != set(nomes):
            raise ValueError("Janelas preço-volume incompatíveis")
        esperado = configurar_preco_volume(tuple(janelas[k] for k in nomes))
    else:
        raise ValueError("Estado da configuração preço-volume inválido")
    if config != esperado:
        raise ValueError("Método ou configuração preço-volume incompatível")


def validar_decomposicao_receita(df, config, run_esperado, lote_esperado, instante_esperado):
    validar_configuracao_preco_volume(config)
    tipos = {c.name: c.dataType.simpleString() for c in df.schema.fields}
    if tipos != TIPOS_DECOMPOSICAO_RECEITA or len(df.columns) != len(tipos):
        raise ValueError("Schema/tipos de decomposicao_receita divergem do contrato")
    if config["estado"] == "NAO_CONFIGURADA":
        if df.limit(1).count():
            raise ValueError("Análise não configurada não pode reutilizar resultado anterior")
        return
    chaves = ["municipio", "bairro", "classe_consumo", "inicio_p0", "fim_p0", "inicio_p1", "fim_p1"]
    obrigatorios = chaves + ["estado_decomposicao", "metodo", "run_id_silver", "lote_bronze_silver",
                            "data_processamento_gold"] + [k for k in tipos if k.startswith("qtd_")]
    invalidos = F.lit(False)
    for nome in obrigatorios:
        invalidos = invalidos | F.col(nome).isNull()
    invalidos = invalidos | (F.col("run_id_silver") != F.lit(run_esperado)) | (F.col("lote_bronze_silver") != F.lit(lote_esperado))
    invalidos = invalidos | (F.col("data_processamento_gold") != instante_esperado) | (F.col("metodo") != METODO_PRECO_VOLUME)
    for nome, valor in config["janelas"].items():
        invalidos = invalidos | (F.col(nome) != valor)
    estados = ["CALCULADA_BASE_COMUM", "SEM_BASE_COMUM", "COBERTURA_NAO_CERTIFICADA"]
    invalidos = invalidos | ~F.col("estado_decomposicao").isin(estados)
    comum = F.col("estado_decomposicao") == "CALCULADA_BASE_COMUM"
    sem = F.col("estado_decomposicao") == "SEM_BASE_COMUM"
    nao_certificada = F.col("estado_decomposicao") == "COBERTURA_NAO_CERTIFICADA"
    invalidos = invalidos | (comum & (F.col("qtd_ucs_base_comum") <= 0)) | (sem & (F.col("qtd_ucs_base_comum") != 0))
    invalidos = invalidos | (F.col("qtd_ucs_observadas") <= 0) | (
        F.col("qtd_ucs_base_comum") + F.col("qtd_ucs_entrantes") + F.col("qtd_ucs_saintes") > F.col("qtd_ucs_observadas"))
    for nome, tipo in tipos.items():
        if tipo == "bigint":
            invalidos = invalidos | (F.col(nome) < 0)
        if tipo == "double":
            invalidos = invalidos | F.isnan(nome) | (F.abs(F.col(nome)) == float("inf"))
            invalidos = invalidos | (nao_certificada & F.col(nome).isNotNull())
            if nome.startswith("efeito_"):
                invalidos = invalidos | (comum & F.col(nome).isNull()) | (sem & F.col(nome).isNotNull())
            else:
                invalidos = invalidos | (~nao_certificada & F.col(nome).isNull())
    termos = ["efeito_volume_reais", "efeito_preco_observado_reais", "efeito_interacao_reais",
              "variacao_entradas_reais", "variacao_saidas_reais", "variacao_sem_leitura_fora_base_reais",
              "variacao_outros_fora_base_reais"]
    soma = F.lit(0.)
    for nome in termos:
        soma = soma + F.coalesce(F.col(nome), F.lit(0.))
    tolerancia = F.greatest(F.lit(0.000001), F.abs("variacao_faturamento_total_reais") * F.lit(1e-9))
    residuo = F.col("variacao_faturamento_total_reais") - soma
    invalidos = invalidos | (~nao_certificada & ((F.abs(residuo) > tolerancia) |
        (F.abs(residuo-F.col("residuo_reconciliacao_reais")) > tolerancia) |
        (F.abs(F.col("faturamento_p1_reais")-F.col("faturamento_p0_reais")-F.col("variacao_faturamento_total_reais")) > tolerancia)))
    if df.filter(invalidos).limit(1).count() or df.groupBy(*chaves).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError("Análise preço-volume com identidade, valores ou reconciliação inválidos")


_pv_nomes = ("PRECO_VOLUME_INICIO_P0", "PRECO_VOLUME_FIM_P0", "PRECO_VOLUME_INICIO_P1", "PRECO_VOLUME_FIM_P1")
for _nome in _pv_nomes:
    dbutils.widgets.text(_nome, "", "Competência preço-volume YYYY-MM; opcional")
_pv_janelas = tuple(dbutils.widgets.get(nome).strip() for nome in _pv_nomes)
preco_volume_configuracao = configurar_preco_volume(_pv_janelas)
if any(_pv_janelas):
    _pv_base = preparar_base_preco_volume(gold_consumo_mensal_uc, df_uc_mes)
    decomposicao_preco_volume = decompor_preco_volume(_pv_base, df_cobertura_dominios, *_pv_janelas,
        run_id_silver_esperado=run_id_silver, lote_bronze_silver_esperado=lote_bronze_silver)
    decomposicao_preco_volume.printSchema()
    print("Diagnóstico candidato por UC disponível; conferir cobertura, entradas/saídas e base comum.")

# Materializar sempre: sem configuração a tabela vazia substitui a análise anterior.
if preco_volume_configuracao["estado"] == "NAO_CONFIGURADA":
    df_decomposicao_gold = spark.createDataFrame([], schema=", ".join(
        f"{nome} {tipo}" for nome, tipo in TIPOS_DECOMPOSICAO_RECEITA.items()))
else:
    df_decomposicao_gold = (decomposicao_preco_volume
        .withColumn("run_id_silver", F.lit(run_id_silver))
        .withColumn("lote_bronze_silver", F.lit(lote_bronze_silver))
        .withColumn("data_processamento_gold", instante_referencia())
        .select(*TIPOS_DECOMPOSICAO_RECEITA))
validar_decomposicao_receita(df_decomposicao_gold, preco_volume_configuracao,
    run_id_silver, lote_bronze_silver, instante_referencia())
(df_decomposicao_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.decomposicao_receita"))


## Validação do conjunto em preparo


## 7. Testes de Qualidade Automatizados e Validação do Pipeline
### Objetivo:
Garantir que os dados gerados na Gold atendem às restrições de integridade, contagem e consistência lógica antes de serem exportados para o banco de dados local da aplicação.

### 7.1 Suíte de Testes Automatizados de Qualidade e Integridade da Camada Gold

**Objetivo:** Executar a validação sistêmica de qualidade, unicidade de chaves primárias (PKs), não-nulidade e consistência de domínios em todas as **quinze projeções analíticas e um cadastro restrito** consolidadas na camada Gold, atuando como a primeira trava do *Quality Gate*.

---

#### Escopo de Validação e Asserções Aplicadas:

1. **Mapeamento de Chaves Primárias (PKs Simples e Compostas):**
   * `diagnostico_receita`: `['competencia', 'municipio', 'classe_consumo', 'subgrupo_tarifario', 'bandeira_tarifaria', 'possui_gd']`
   * `indicadores_dec_fec`: `['id_alimentador', 'competencia']`
   * `fila_inspecao`: `['id_uc']`
   * `visao_360_uc`: `['id_uc']`
   * `faturas_uc`: `['id_fatura']`
   * `medidor_uc`: `['id_medidor']`
   * `atendimentos_uc`: `['protocolo']`
   * `ocorrencias_uc`: `['id_ocorrencia', 'id_uc']`
   * `consumo_mensal_uc`: `['id_uc', 'competencia']`
   * `consumo_receita_bairro`: `['competencia', 'municipio', 'bairro', 'classe_consumo']`
   * `variacao_consumo_uc`: `['id_uc']`

2. **Verificações Estruturais e Integridade de Registros:**
   * **Presença de Dados:** Garantia de que nenhuma tabela da camada Gold está vazia (`count > 0`).
   * **Validação de Nulos em Chaves (`assert_no_nulls`):** Asserção estrita de que nenhum atributo formador de PK contenha valores `NULL`.
   * **Unicidade de Chaves Primárias (`assert_unique`):** Garantia de inexistência de duplicatas no grão definido de cada tabela.

3. **Regras de Negócio e Consistência de Domínio:**
   * **Integridade das Faturas (`faturas_uc`):** Obrigatoriedade de valores válidos nos campos vitais de negócio (`id_uc`, `mes_referencia`, `valor_total`) e validação estrita do formato de competência (`YYYY-MM`).
   * **Ativos de Medição (`medidor_uc`):** Validação de integridade nos atributos obrigatórios (`id_medidor`, `id_uc`) e reconciliação volumétrica de 100% dos registros preservados da camada Silver (`medidores`).
   * **Histórico de Atendimentos (`atendimentos_uc`):** Validação da não-nulidade nas chaves obrigatórias (`protocolo`, `id_uc`), garantia de unicidade estrita do grão por protocolo e reconciliação volumétrica de 100% com a tabela de origem da Silver (`atendimentos`).
   * **Histórico de Ocorrências (`ocorrencias_uc`):** Validação da não-nulidade na chave composta (`id_ocorrencia`, `id_uc`), garantia de unicidade estrita do grão e reconciliação volumétrica ajustada com os registros válidos da tabela de origem da Silver (`ocorrencias_rede`).
   * **Indicadores Operacionais (`indicadores_dec_fec`):** Bloqueio de valores negativos em DEC, FEC, durações e frequências de interrupção, além de garantia de denominadores válidos (`total_ucs_alimentador > 0`).
   * **Métricas Financeiras (`diagnostico_receita`):** Impedimento de receita ou consumo negativos e validação da tarifa média efetiva para consumos positivos.
   * **Conformidade Temporal:** Rigor no padrão regex para competências/meses de referência no formato `YYYY-MM` (`^[0-9]{4}-(0[1-9]|1[0-2])$`).
   * **Grão Único por UC:** Confirmação de que as visões por cliente (`fila_inspecao`, `visao_360_uc` e `variacao_consumo_uc`) mantêm rigorosamente a relação de 1 linha por `id_uc`.

In [ ]:
# SEÇÃO 7.1 : Suíte de Testes Automatizados de Qualidade e Integridade

from pyspark.sql import functions as F

# Fallback seguro para referência ao schema Silver
schema_silver_ref = SCHEMA_SILVER_DADOS if 'SCHEMA_SILVER_DADOS' in locals() else (
    SCHEMA_SILVER if 'SCHEMA_SILVER' in locals() else "silver_publicada"
)

# Dicionário atualizado cobrindo TODAS as tabelas da camada Gold (incluindo ocorrencias_uc)
tables_gold_pk = {
    "diagnostico_receita": [
        "competencia",
        "municipio",
        "classe_consumo",
        "subgrupo_tarifario",
        "bandeira_tarifaria",
        "possui_gd"
    ],
    "indicadores_dec_fec": [
        "id_alimentador",
        "competencia"
    ],
    "fila_inspecao": ["id_uc"],
    "visao_360_uc": ["id_uc"],
    "faturas_uc": ["id_fatura"],
    "medidor_uc": ["id_medidor"],  
    "atendimentos_uc": ["protocolo"],
    "ocorrencias_uc": ["id_ocorrencia", "id_uc"],
    "consumo_mensal_uc": ["id_uc", "competencia"],
    "consumo_receita_bairro": ["competencia", "municipio", "bairro", "classe_consumo"],
    "variacao_consumo_uc": ["id_uc"],
    "gd_uc": ["id_uc"],
    "tratativas_uc": ["protocolo", "posicao_historico"],
    "cobertura_dominios": ["dominio", "competencia", "recorte"],
    "decomposicao_receita": ["municipio", "bairro", "classe_consumo", "inicio_p0", "fim_p0", "inicio_p1", "fim_p1"]
}

# 1. Confere que todas as tabelas existem e têm registros.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_columns_exist(df, pk_cols, nome_completo)

    if df.limit(1).count() == 0 and tabela not in {"indicadores_dec_fec", "medidor_uc", "atendimentos_uc", "ocorrencias_uc", "gd_uc", "tratativas_uc", "decomposicao_receita"}:
        raise ValueError(f"{nome_completo} está vazia.")

    print(f"[PASS] {nome_completo}: possui registros.")

# 2. Chaves compostas e primárias não podem conter nulos.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_no_nulls(df, pk_cols, nome_completo)
    print(f"[PASS] {nome_completo}: chave sem nulos.")

# 3. A chave declarada deve ser única.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_unique(df, pk_cols, nome_completo)
    print(f"[PASS] {nome_completo}: chave única.")


# --- VALIDAÇÕES ESPECÍFICAS: faturas_uc ---
df_faturas = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc")

# Atributos vitais não podem conter nulos
assert_no_nulls(df_faturas, ["id_uc", "mes_referencia", "valor_total"], f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc")
print("[PASS] faturas_uc: id_uc, mes_referencia e valor_total sem nulos.")

# Mês de referência no formato estrito YYYY-MM
mes_ref_invalido = (
    df_faturas
    .filter(
        F.col("mes_referencia").isNull()
        | ~F.col("mes_referencia").rlike(r"^[0-9]{4}-(0[1-9]|1[0-2])$")
    )
    .limit(1)
    .count()
)

if mes_ref_invalido > 0:
    raise ValueError("faturas_uc contém mes_referencia fora do formato YYYY-MM.")

print("[PASS] faturas_uc: mes_referencia no formato YYYY-MM.")


# --- VALIDAÇÕES ESPECÍFICAS: medidor_uc ---
df_medidor = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.medidor_uc")

# Atributos vitais obrigatoriamente não nulos
assert_no_nulls(df_medidor, ["id_medidor", "id_uc"], f"{CATALOG}.{SCHEMA_GOLD}.medidor_uc")
print("[PASS] medidor_uc: id_medidor e id_uc sem nulos.")

# Reconciliação Volumétrica com a Silver (deve manter 100% dos registros de medidores)
total_silver_medidores = read_silver("medidores").filter(uc_real()).count()
total_gold_medidores = df_medidor.count()

if total_gold_medidores != total_silver_medidores:
    raise ValueError(
        f"Divergência volumétrica em medidor_uc! "
        f"Silver: {total_silver_medidores} | Gold: {total_gold_medidores}"
    )

print(f"[PASS] medidor_uc: reconciliação volumétrica de 100% com a Silver ({total_gold_medidores} registros).")


# --- VALIDAÇÕES ESPECÍFICAS: atendimentos_uc ---
df_atendimentos = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.atendimentos_uc")

# Atributos vitais obrigatoriamente não nulos (protocolo e id_uc)
assert_no_nulls(df_atendimentos, ["protocolo", "id_uc"], f"{CATALOG}.{SCHEMA_GOLD}.atendimentos_uc")
print("[PASS] atendimentos_uc: protocolo e id_uc sem nulos.")

# Reconciliação Volumétrica com a Silver (deve manter 100% dos registros de atendimentos válidos)
total_silver_atendimentos = read_silver("atendimentos").filter(uc_real() & F.col("protocolo").isNotNull()).count()
total_gold_atendimentos = df_atendimentos.count()

if total_gold_atendimentos != total_silver_atendimentos:
    raise ValueError(
        f"Divergência volumétrica em atendimentos_uc! "
        f"Silver: {total_silver_atendimentos} | Gold: {total_gold_atendimentos}"
    )

print(f"[PASS] atendimentos_uc: reconciliação volumétrica de 100% com a Silver ({total_gold_atendimentos} registros).")


# --- VALIDAÇÕES ESPECÍFICAS: ocorrencias_uc ---
df_ocorrencias = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.ocorrencias_uc")

# Atributos vitais da chave composta obrigatoriamente não nulos
assert_no_nulls(df_ocorrencias, ["id_ocorrencia", "id_uc"], f"{CATALOG}.{SCHEMA_GOLD}.ocorrencias_uc")
print("[PASS] ocorrencias_uc: id_ocorrencia e id_uc sem nulos.")

# Reconciliação Volumétrica com a Silver (considerando o filtro de UCs ausentes aplicado no ETL)
total_silver_ocorrencias = read_silver("ocorrencias_rede").filter(F.col("_id_uc_ausente") == False).count()
total_gold_ocorrencias = df_ocorrencias.count()

if total_gold_ocorrencias != total_silver_ocorrencias:
    raise ValueError(
        f"Divergência volumétrica em ocorrencias_uc! "
        f"Silver (válidos): {total_silver_ocorrencias} | Gold: {total_gold_ocorrencias}"
    )

print(f"[PASS] ocorrencias_uc: reconciliação volumétrica de 100% com a Silver ajustada ({total_gold_ocorrencias} registros).")


# --- DEMAIS TESTES DE MÉTRICAS DAS OUTRAS TABELAS ---

# Verificar métricas negativas em DEC/FEC.
df_dec = spark.table(
    f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec"
)

if (
    df_dec
    .filter(
        (F.col("DEC") < 0)
        | (F.col("FEC") < 0)
        | (F.col("duracao_total_interrupcoes_horas") < 0)
        | (F.col("frequencia_total_interrupcoes") < 0)
        | (F.col("total_ucs_alimentador") <= 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "indicadores_dec_fec contém valores negativos "
        "ou denominadores inválidos."
    )

# FEC exige contagem e denominador; DEC permanece desconhecido se faltar duração.
if (
    df_dec
    .filter(
        F.col("FEC").isNull()
        | (F.col("DEC").isNull() != F.col("duracao_total_interrupcoes_horas").isNull())
    )
    .limit(1)
    .count() > 0
):
    raise ValueError("indicadores_dec_fec contém FEC ausente ou DEC incompatível com a duração disponível.")

print("[PASS] DEC/FEC: ausência de duração preservada, sem negativos ou denominadores inválidos.")

# Verificar métricas de receita e consumo.
df_receita = spark.table(
    f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita"
)

if (
    df_receita
    .filter(
        (F.col("receita_total_reais") < 0)
        | (F.col("consumo_total_kwh") < 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "diagnostico_receita contém receita ou consumo negativos. "
        "Confirme se estornos e ajustes devem ser permitidos."
    )

# A tarifa efetiva pode ser nula quando consumo total é zero ou nulo.
if (
    df_receita
    .filter(
        (F.col("consumo_base_comparavel_kwh") > 0)
        & F.col("receita_base_comparavel_reais").isNotNull()
        & F.col("tarifa_media_efetiva").isNull()
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "diagnostico_receita tem tarifa efetiva nula "
        "apesar de consumo positivo."
    )

print("[PASS] Receita e consumo dentro dos limites definidos.")

# Formato estrito YYYY-MM para diagnostico_receita.
competencia_invalida = (
    df_receita
    .filter(
        F.col("competencia").isNull()
        | ~F.col("competencia").rlike(
            r"^[0-9]{4}-(0[1-9]|1[0-2])$"
        )
    )
    .limit(1)
    .count()
)

if competencia_invalida > 0:
    raise ValueError(
        "diagnostico_receita contém competência fora do formato YYYY-MM."
    )

print("[PASS] Competências de receita no formato YYYY-MM.")

# Checagem equivalente para DEC/FEC.
competencia_dec_invalida = (
    df_dec
    .filter(
        F.col("competencia").isNull()
        | ~F.col("competencia").rlike(
            r"^[0-9]{4}-(0[1-9]|1[0-2])$"
        )
    )
    .limit(1)
    .count()
)

if competencia_dec_invalida > 0:
    raise ValueError(
        "indicadores_dec_fec contém competência fora do formato YYYY-MM."
    )

print("[PASS] Competências de DEC/FEC no formato YYYY-MM.")

# Garantir que nenhuma UC foi multiplicada nas tabelas de visão/fila/variação.
for tabela in ["fila_inspecao", "visao_360_uc", "variacao_consumo_uc"]:
    df = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.{tabela}")

    assert_unique(df, ["id_uc"], f"{CATALOG}.{SCHEMA_GOLD}.{tabela}")

print("[PASS] Uma linha por UC na fila, visão 360 e variação de consumo.")
print("Todos os testes de qualidade da Gold foram concluídos.")
# Conferir novamente a análise materializada antes de permitir sua promoção.
df_decomposicao_test = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.decomposicao_receita")
validar_decomposicao_receita(df_decomposicao_test, preco_volume_configuracao,
    run_id_silver, lote_bronze_silver, instante_referencia())
print("[PASS] Análise materializada confere com configuração, origem e identidade financeira.")

validar_estimativas_recuperacao(spark.table(f"{CATALOG}.{SCHEMA_GOLD}.fila_inspecao"))


### 7.2 Testes de Qualidade e Integridade Cruzada 

**Objetivo:** Validar a consistência física, financeira, cadastral, relacional e o saneamento temporal entre as tabelas analíticas complementares (`consumo_receita_bairro`, `variacao_consumo_uc`, `consumo_mensal_uc`, `faturas_uc`, `medidor_uc`) e a tabela mestra de receitas (`diagnostico_receita`).

---

#### Escopo de Testes e Validações Automatizadas:

1. **Validação Estrutural e Integridade de Schemas:**
   * Confirmar a presença das colunas obrigatórias em `faturas_uc`, incluindo situação/saldo indicativos e flags de qualidade, e em `medidor_uc`, incluindo datas de instalação/retirada, flags e situação candidata no corte.
   * Assegurar a não-nulidade e a unicidade das chaves primárias e dimensões de agrupamento (`bairro`, `id_uc`, `competencia`, `id_fatura`, `id_medidor`).

2. **Auditoria de Volumetria Cadastral e Saneamento Temporal:**
   * **Volumetria de UCs:** Garantir que a contagem distinta de Unidades Consumidoras na tabela de variação temporal (`variacao_consumo_uc`) seja 100% equivalente ao total de UCs operacionais retidas na `visao_360_uc` (`EXPECTED_UC_COUNT`).
   * **Saneamento de Datas em `faturas_uc`:** Emissão e pagamento seguem as flags Silver. Vencimento futuro é preservado; flags de pagamento inconclusivo impedem apresentar a data como liquidação comprovada.
   * **Saneamento e vigência em `medidor_uc`:** Datas sinalizadas como sentinelas ou futuras pela Silver ficam nulas e mantêm flags. Não há corte arbitrário por ano; os rótulos temporais são candidatos, não selecionam um medidor único por UC e precisam de validação conjunta.

3. **Integridade Referencial Cruzada (Chaves Órfãs):**
   * **Validação `medidor_uc` vs. `visao_360_uc`:** Asserção via *Left Anti Join* garantindo que 100% dos identificadores de UC (`id_uc`) presentes no histórico de medidores existam na dimensão mestre cadastral (`visao_360_uc`), assegurando ausência total de chaves órfãs.

4. **Conciliação Físico-Financeira Cruzada (Tolerância: R$ 0,01 e 0,01 kWh):**
   * **Tabela Territorial (`consumo_receita_bairro`) vs. Diagnóstico:** A soma total da receita e do consumo por bairro deve bater exatamente com os totais consolidados no diagnóstico de receita.
   * **Tabela Mensal (`consumo_mensal_uc`) vs. Diagnóstico:** A consolidação do consumo e da receita individualizada por UC em nível mensal deve resultar no mesmo valor global do diagnóstico.

5. **Validação de Regra de Negócio Dinâmica (Geração Distribuída / Solar):**
   * Teste de asserção para comprovar a identificação das UCs equipadas com micro/minigeração distribuída (`possui_gd = 'Sim'`) que registraram redução expressiva de consumo (`queda_consumo_superior_30_pct = True`).

In [ ]:

# SEÇÃO 7.2 : Testes de Qualidade e Integridade Cruzada


from pyspark.sql import functions as F

print("=== TESTES DE QUALIDADE E INTEGRIDADE CRUZADA ===")

# Nome das tabelas gravadas no catálogo Delta
TABELA_BAIRRO = f"{CATALOG}.{SCHEMA_GOLD}.consumo_receita_bairro"
TABELA_VARIACAO = f"{CATALOG}.{SCHEMA_GOLD}.variacao_consumo_uc"
TABELA_CONSUMO_MENSAL = f"{CATALOG}.{SCHEMA_GOLD}.consumo_mensal_uc"
TABELA_DIAGNOSTICO = f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita"
TABELA_VISAO_360 = f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc"
TABELA_FATURAS = f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc"
TABELA_MEDIDORES = f"{CATALOG}.{SCHEMA_GOLD}.medidor_uc"
TABELA_FILA = f"{CATALOG}.{SCHEMA_GOLD}.fila_inspecao"
TABELA_DEC = f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec"
TABELA_ATENDIMENTOS = f"{CATALOG}.{SCHEMA_GOLD}.atendimentos_uc"
TABELA_OCORRENCIAS = f"{CATALOG}.{SCHEMA_GOLD}.ocorrencias_uc"

# Tolerâncias financeiras e operacionais
TOLERANCIA_REAIS = Decimal("0.01")
TOLERANCIA_KWH = 0.01

# Carregar tabelas do catálogo Delta
df_bairro_test = spark.table(TABELA_BAIRRO)
df_variacao_test = spark.table(TABELA_VARIACAO)
df_mensal_test = spark.table(TABELA_CONSUMO_MENSAL)
df_diagnostico_test = spark.table(TABELA_DIAGNOSTICO)
df_v360_test = spark.table(TABELA_VISAO_360)
df_faturas_test = spark.table(TABELA_FATURAS)
df_medidores_test = spark.table(TABELA_MEDIDORES)
df_fila_test = spark.table(TABELA_FILA)
df_dec_test = spark.table(TABELA_DEC)
df_atendimentos_test = spark.table(TABELA_ATENDIMENTOS)
df_ocorrencias_test = spark.table(TABELA_OCORRENCIAS)

# 1. Volumetria cadastral esperada via contagem distinta na Visão 360
EXPECTED_UC_COUNT = df_v360_test.select("id_uc").distinct().count()

# 2. Validação exata dos schemas candidatos publicados.
def assert_schema_gold(df, esperadas, nome):
    recebidas = list(df.columns)
    faltantes = sorted(set(esperadas) - set(recebidas))
    extras = sorted(set(recebidas) - set(esperadas))
    duplicadas = sorted({coluna for coluna in recebidas if recebidas.count(coluna) > 1})
    if len(recebidas) != len(esperadas) or faltantes or extras or duplicadas:
        raise ValueError(
            f"Schema de {nome} diverge do contrato candidato: "
            f"faltantes={faltantes}; extras={extras}; colunas_duplicadas={duplicadas}"
        )

cols_esperadas_diagnostico = [
    "competencia", "municipio", "classe_consumo", "subgrupo_tarifario", "bandeira_tarifaria", "possui_gd",
    "receita_total_reais", "receita_com_leitura_reais", "receita_sem_leitura_reais", "consumo_total_kwh",
    "receita_base_comparavel_reais", "consumo_base_comparavel_kwh", "qtd_ucs_distintas_faturadas_no_mes",
    "qtd_ucs_com_leitura", "soma_faturamentos_mensais", "qtd_leituras_observadas", "qtd_leituras_validas",
    "ticket_medio_reais_por_uc", "consumo_medio_kwh_por_uc", "tarifa_media_efetiva",
]
cols_esperadas_dec = [
    "id_alimentador", "municipio", "competencia", "total_ucs_alimentador", "ucs_interrompidas",
    "frequencia_total_interrupcoes", "duracao_total_interrupcoes_horas", "DEC", "FEC",
]
cols_esperadas_bairro = [
    "competencia", "municipio", "bairro", "classe_consumo", "receita_total_reais",
    "receita_com_leitura_reais", "receita_sem_leitura_reais", "consumo_total_kwh",
    "receita_base_comparavel_reais", "consumo_base_comparavel_kwh", "qtd_ucs_distintas_faturadas_no_mes",
    "qtd_ucs_com_leitura", "soma_faturamentos_mensais", "qtd_leituras_observadas", "qtd_leituras_validas",
]
cols_esperadas_variacao = [
    "id_uc", "municipio", "bairro", "possui_gd", "consumo_medio_inicial", "meses_com_leitura_p1",
    "consumo_medio_final", "meses_com_leitura_p2", "variacao_kwh", "variacao_pct", "queda_consumo_superior_30_pct",
]
cols_esperadas_mensal = [
    "id_uc", "competencia", "consumo_kwh", "receita_reais", "receita_com_leitura_reais",
    "receita_sem_leitura_reais", "receita_base_comparavel_reais", "consumo_base_comparavel_kwh",
    "tem_fatura", "tem_leitura", "qtd_faturas", "qtd_leituras_observadas", "qtd_leituras_validas",
]
cols_esperadas_faturas = [
    "id_fatura", "id_uc", "mes_referencia", "data_emissao", "data_vencimento", "data_pagamento",
    "valor_energia", "tributos", "valor_total", "bandeira_tarifaria", "status", "valor_fatura_declarada_em_aberto_reais", "valor_em_aberto_reais",
    "situacao_pagamento_analitica", "flag_paga", "flag_tem_pagamento", "flag_valor_inconsistente",
    "flag_data_pagamento_inconclusiva", "flag_paga_sem_data_valida", "flag_pagamento_antes_emissao",
    "flag_pagamento_sentinela", "flag_pagamento_futuro", "flag_emissao_sentinela", "flag_emissao_futura",
    "flag_vencimento_sentinela", "flag_valor_negativo", "data_processamento_gold",
]
cols_esperadas_medidores = [
    "id_medidor", "id_uc", "modelo", "tipo", "data_instalacao", "data_retirada", "status",
    "flag_instalacao_sentinela", "flag_instalacao_futura", "flag_retirada_sentinela", "flag_retirada_futura",
    "ingestao_data", "situacao_na_referencia",
]
cols_esperadas_fila = [
    "id_uc", "municipio", "bairro", "uf", "classe", "subgrupo", "alimentador", "data_ligacao",
    "status", "possui_geracao_solar", "total_inspecoes_realizadas", "total_inspecoes_irregulares",
    "media_consumo_historico", "prioridade_inspecao", "motivo_prioridade",
    "media_historica_recuperacao_kwh_por_inspecao", "intervalo_historico_recuperacao_inferior_kwh", "intervalo_historico_recuperacao_superior_kwh",
    "referencia_media_historica_recuperacao", "estado_media_historica_recuperacao",
    "energia_estimada_recuperavel_kwh", "intervalo_recuperacao_inferior_kwh", "intervalo_recuperacao_superior_kwh",
    "estado_estimativa_recuperacao", "metodo_estimativa_recuperacao", "validacao_recuperacao_json",
]
cols_esperadas_v360 = [
    "id_uc", "municipio", "bairro", "uf", "classe", "subgrupo", "alimentador", "data_ligacao",
    "status", "possui_gd", "total_chamados_callcenter", "media_satisfacao_atendimento",
    "total_interrupcoes_sofridas", "duracao_total_interrupcoes_horas", "media_consumo_historico",
]
cols_esperadas_atendimentos = [
    "protocolo", "id_uc", "canal", "assunto", "subassunto", "data_abertura", "data_fechamento",
    "nota_satisfacao", "qtd_tratativas", "_abertura_futura", "_fechamento_futura", "_nota_negativa",
    "_nota_invalida", "ingestao_data",
]
cols_esperadas_ocorrencias = [
    "id_ocorrencia", "id_uc", "alimentador", "tipo", "causa", "data_abertura", "data_encerramento",
    "duracao_min", "flag_duracao_incompativel", "flag_abertura_futura", "flag_encerramento_futuro",
    "ingestao_data",
]

for df, colunas, tabela in [
    (df_diagnostico_test, cols_esperadas_diagnostico, TABELA_DIAGNOSTICO),
    (df_dec_test, cols_esperadas_dec, TABELA_DEC),
    (df_fila_test, cols_esperadas_fila, TABELA_FILA),
    (df_v360_test, cols_esperadas_v360, TABELA_VISAO_360),
    (df_bairro_test, cols_esperadas_bairro, TABELA_BAIRRO),
    (df_variacao_test, cols_esperadas_variacao, TABELA_VARIACAO),
    (df_mensal_test, cols_esperadas_mensal, TABELA_CONSUMO_MENSAL),
    (df_faturas_test, cols_esperadas_faturas, TABELA_FATURAS),
    (df_medidores_test, cols_esperadas_medidores, TABELA_MEDIDORES),
    (df_atendimentos_test, cols_esperadas_atendimentos, TABELA_ATENDIMENTOS),
    (df_ocorrencias_test, cols_esperadas_ocorrencias, TABELA_OCORRENCIAS),
]:
    assert_schema_gold(df, colunas, tabela)

for nome in ["gd_uc", "tratativas_uc", "cadastro_titular_uc"]:
    df = spark.table(origem_gold(nome))
    assert_schema_gold(df, list(TIPOS_COMPLEMENTOS_UC[nome]), nome)
    if {campo.name: campo.dataType.simpleString() for campo in df.schema.fields} != TIPOS_COMPLEMENTOS_UC[nome]:
        raise ValueError(f"Tipos de {nome} divergem do contrato")
df_cobertura_test = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.cobertura_dominios")
assert_schema_gold(df_cobertura_test, list(TIPOS_COBERTURA_DOMINIOS), "cobertura_dominios")
if {campo.name:campo.dataType.simpleString() for campo in df_cobertura_test.schema.fields} != TIPOS_COBERTURA_DOMINIOS:
    raise ValueError("Tipos de cobertura_dominios divergem do contrato")
print("[PASS] Schemas analíticos e cadastro restrito conferem por nome, quantidade e tipos dos complementos.")

# 3. Nulos e Unicidade
chave_bairro = ["competencia", "municipio", "bairro", "classe_consumo"]
chave_mensal = ["id_uc", "competencia"]

assert_no_nulls(df_bairro_test, chave_bairro, TABELA_BAIRRO)
assert_no_nulls(df_variacao_test, ["id_uc", "municipio", "bairro", "possui_gd"], TABELA_VARIACAO)
assert_no_nulls(df_mensal_test, chave_mensal, TABELA_CONSUMO_MENSAL)
assert_no_nulls(df_faturas_test, ["id_fatura", "id_uc", "mes_referencia", "valor_total"], TABELA_FATURAS)
assert_no_nulls(df_medidores_test, ["id_medidor", "id_uc"], TABELA_MEDIDORES)
assert_no_nulls(df_faturas_test, ["situacao_pagamento_analitica"], TABELA_FATURAS)
assert_no_nulls(df_medidores_test, ["situacao_na_referencia"], TABELA_MEDIDORES)
assert_no_nulls(df_fila_test, ["prioridade_inspecao", "motivo_prioridade"], TABELA_FILA)
situacoes_fatura_invalidas = df_faturas_test.filter(~F.col("situacao_pagamento_analitica").isin("EM_ABERTO_DECLARADO", "EM_ABERTO_COM_SINAL_DE_PAGAMENTO", "PAGA_DECLARADA_COM_DATA_VALIDA", "PAGA_DECLARADA_SEM_DATA_VALIDA", "PAGAMENTO_INCONCLUSIVO", "SITUACAO_INDETERMINADA")).limit(1).count()
assert situacoes_fatura_invalidas == 0, "faturas_uc contém situação analítica fora do domínio."
situacoes_medidor_invalidas = df_medidores_test.filter(~F.col("situacao_na_referencia").isin("INSTALADO", "RETIRADO", "INDETERMINADO")).limit(1).count()
assert situacoes_medidor_invalidas == 0, "medidor_uc contém situação temporal fora do domínio."

assert_unique(df_bairro_test, chave_bairro, TABELA_BAIRRO)
assert_unique(df_variacao_test, ["id_uc"], TABELA_VARIACAO)
assert_unique(df_mensal_test, chave_mensal, TABELA_CONSUMO_MENSAL)
assert_unique(df_faturas_test, ["id_fatura"], TABELA_FATURAS)
assert_unique(df_medidores_test, ["id_medidor"], TABELA_MEDIDORES)
assert_unique(df_fila_test, ["id_uc"], TABELA_FILA)
print("[PASS] Chaves e dimensões obrigatórias validadas sem nulos e únicas em todas as tabelas.")

# 4. Volumetria Cadastral em Variacao UC
total_ucs_variacao = df_variacao_test.select("id_uc").distinct().count()
assert total_ucs_variacao == EXPECTED_UC_COUNT, f"Volumetria divergente: esperado {EXPECTED_UC_COUNT:,}, encontrado {total_ucs_variacao:,}."
print(f"[PASS] Volumetria cadastral cravada em {total_ucs_variacao:,} UCs distintas.")

# 5. Validação de Saneamento de Datas em faturas_uc e medidor_uc
# 5.1 Saneamento de faturas_uc
datas_invalidas_faturas = df_faturas_test.filter(fatura_datas_invalidas()).count()
assert datas_invalidas_faturas == 0, f"Encontradas {datas_invalidas_faturas} datas sentinelas/futuras não tratadas em faturas_uc!"
print("[PASS] Saneamento de datas em faturas_uc verificado: zero datas incompatíveis; vencimento futuro preservado.")

# 5.2 Saneamento de medidor_uc (cronologia e sentinela sinalizada pela Silver)
datas_invalidas_medidores = (
    df_medidores_test
    .filter(
        (F.col("data_instalacao") >= "9999-01-01") | (F.col("data_instalacao") > data_referencia())
        | ((F.col("data_retirada").isNotNull()) & (
            (F.col("data_retirada") >= "9999-01-01") | (F.col("data_retirada") > data_referencia())
        ))
    )
    .count()
)
assert datas_invalidas_medidores == 0, f"Encontradas {datas_invalidas_medidores} datas sentinelas/inválidas em medidor_uc!"
print("[PASS] Saneamento de datas em medidor_uc verificado: zero datas sentinelas ou futuras.")

# 6. Validação de Integridade Referencial Cruzada (Chaves Órfãs em medidor_uc vs visao_360_uc)
ucs_orfas_medidor = (
    df_medidores_test
    .select("id_uc")
    .distinct()
    .join(df_v360_test.select("id_uc").distinct(), on="id_uc", how="left_anti")
    .count()
)

assert ucs_orfas_medidor == 0, f"FALHA DE INTEGRIDADE: Encontradas {ucs_orfas_medidor} UCs em medidor_uc que não existem na visao_360_uc!"
print("[PASS] Integridade Referencial verificada: 0 UCs órfãs em medidor_uc (100% pertencem à visao_360_uc).")

# 7. Conciliação Físico-Financeira
totais_bairro = df_bairro_test.agg(
    F.sum("receita_total_reais").alias("receita_bairro"),
    F.sum("consumo_total_kwh").alias("consumo_bairro")
)

totais_mensal = df_mensal_test.agg(
    F.sum("receita_reais").alias("receita_mensal"),
    F.sum("consumo_kwh").alias("consumo_mensal")
)

totais_diagnostico = df_diagnostico_test.agg(
    F.sum("receita_total_reais").alias("receita_diagnostico"),
    F.sum("consumo_total_kwh").alias("consumo_diagnostico")
)

conciliacao_geral = (
    totais_bairro
    .crossJoin(totais_mensal)
    .crossJoin(totais_diagnostico)
    .withColumn("diff_receita_bairro", F.round(F.col("receita_bairro") - F.col("receita_diagnostico"), 4))
    .withColumn("diff_consumo_bairro", F.round(F.col("consumo_bairro") - F.col("consumo_diagnostico"), 4))
    .withColumn("diff_receita_mensal", F.round(F.col("receita_mensal") - F.col("receita_diagnostico"), 4))
    .withColumn("diff_consumo_mensal", F.round(F.col("consumo_mensal") - F.col("consumo_diagnostico"), 4))
)

display(conciliacao_geral)

totais_conciliacao = conciliacao_geral.first()
conferir_totais(
    totais_conciliacao["receita_bairro"], totais_conciliacao["receita_diagnostico"],
    TOLERANCIA_REAIS, "receita territorial"
)
conferir_totais(
    totais_conciliacao["consumo_bairro"], totais_conciliacao["consumo_diagnostico"],
    TOLERANCIA_KWH, "consumo territorial"
)
conferir_totais(
    totais_conciliacao["receita_mensal"], totais_conciliacao["receita_diagnostico"],
    TOLERANCIA_REAIS, "receita mensal"
)
conferir_totais(
    totais_conciliacao["consumo_mensal"], totais_conciliacao["consumo_diagnostico"],
    TOLERANCIA_KWH, "consumo mensal"
)

print("[PASS] Conciliação Físico-Financeira ok para Tabelas Territorial e Mensal (diferenças zeradas em R$ e kWh).")

# 8. Validação Dinâmica de Regras de Negócio
queda_gd = (
    df_variacao_test
    .filter(
        (F.col("possui_gd") == "Sim")
        & (F.col("queda_consumo_superior_30_pct") == True)
    )
    .count()
)

print(f"UCs com GD e queda >30%: {queda_gd:,}")
# Zero UCs nesse recorte é um resultado legítimo, não uma falha do lote.

print(f"[PASS] Regra de Negócio validada dinamicamente: {queda_gd:,} UCs com GD e queda >30% identificadas.")
print("\nTodos os testes de qualidade e integridade cruzada foram concluídos com 100% de sucesso!")

In [ ]:
# DECOMPOSIÇÃO DE FATURAS DA SILVER 

from pyspark.sql import functions as F

faturas = read_silver("faturas")
leituras = read_silver("leituras")
ucs = read_silver("unidades_consumidoras")

# Normalização de colunas
col_comp_fat = "mes_referencia" if "mes_referencia" in faturas.columns else "competencia"
col_comp_lei = "mes_referencia" if "mes_referencia" in leituras.columns else "competencia"
col_val_fat = "valor_total" if "valor_total" in faturas.columns else "valor_total_reais"

# 1. Chaves de existência de Leituras
chaves_leituras = (
    leituras
    .filter(F.col("id_uc").isNotNull() & F.col(col_comp_lei).isNotNull())
    .select(
        F.col("id_uc"),
        F.col(col_comp_lei).alias("comp_leitura")
    )
    .dropDuplicates(["id_uc", "comp_leitura"])
)

# 2. Chaves de existência de Cadastro de UCs
chaves_ucs = (
    ucs
    .filter(F.col("id_uc").isNotNull())
    .select("id_uc")
    .dropDuplicates(["id_uc"])
)

# 3. Classificação Mutuamente Exclusiva das Faturas
faturas_classificadas = (
    faturas
    .join(
        chaves_leituras,
        (faturas["id_uc"] == chaves_leituras["id_uc"]) & 
        (F.col(col_comp_fat) == F.col("comp_leitura")),
        "left"
    )
    .select(faturas["*"], F.col("comp_leitura"))
    .withColumn("tem_leitura", F.col("comp_leitura").isNotNull())
    .join(chaves_ucs.withColumn("tem_cadastro", F.lit(True)), "id_uc", "left")
    .withColumn("tem_cadastro", F.coalesce(F.col("tem_cadastro"), F.lit(False)))
    .withColumn(
        "grupo",
        F.when(F.col("id_uc").isNull() | F.col(col_comp_fat).isNull(), "0. identificacao_ausente")
         .when(F.col("id_uc").startswith("TESTE"), "0. uc_teste")
         .when(~F.col("tem_leitura"), "1. sem_leitura_correspondente")
         .when(~F.col("tem_cadastro"), "2. com_leitura_sem_cadastro")
         .otherwise("3. com_leitura_e_cadastro")
    )
)

print("--- DECOMPOSIÇÃO DE FATURAS DA SILVER ---")
display(
    faturas_classificadas
    .groupBy("grupo")
    .agg(
        F.count("*").alias("qtd_linhas_fatura"),
        F.sum(col_val_fat).alias("valor_total_reais")
    )
    .orderBy("grupo")
)

### Evidência de execução e limites

As células seguintes executam verificações sobre o lote fixado. Resultados reais e contagens devem permanecer no ambiente autorizado; esta seção não homologa um lote. Zero no recorte não constitui falha por si só.


In [ ]:
# Reconciliação independente: total financeiro e parcelas com/sem leitura.
from decimal import Decimal
from pyspark.sql import functions as F

faturas_silver = read_silver("faturas")
faturas_elegiveis = faturas_silver.filter(uc_real() & F.col("mes_referencia").isNotNull())
chaves = (read_silver("leituras").filter(uc_real() & F.col("mes_referencia").isNotNull())
          .select("id_uc", "mes_referencia").distinct())
com_leitura = faturas_elegiveis.join(chaves, ["id_uc", "mes_referencia"], "left_semi")
sem_leitura = faturas_elegiveis.join(chaves, ["id_uc", "mes_referencia"], "left_anti")

def total_reais(df, coluna):
    return df.agg(F.sum(coluna)).first()[0]

def conferir_reais(esperado, observado, nome):
    if esperado is None or observado is None:
        if esperado is not None or observado is not None:
            raise ValueError(f"Ausência financeira divergente: {nome}")
    elif abs(esperado - observado) > Decimal("0.01"):
        raise ValueError(f"Reconciliação financeira divergente: {nome}")

historico = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc")
if historico.count() != faturas_silver.count():
    raise ValueError("Histórico de faturas perdeu ou duplicou registros")
conferir_reais(total_reais(faturas_silver, "valor_total"), total_reais(historico, "valor_total"), "histórico")
for tabela in ["diagnostico_receita", "consumo_receita_bairro"]:
    observado = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.{tabela}")
    conferir_reais(total_reais(faturas_elegiveis, "valor_total"), total_reais(observado, "receita_total_reais"), tabela)
    # Parcela vazia vale zero apenas quando há faturamento no agregado; grupos só de leitura permanecem nulos.
    existe_faturamento = faturas_elegiveis.limit(1).count() > 0
    for fonte, coluna in [(com_leitura, "receita_com_leitura_reais"), (sem_leitura, "receita_sem_leitura_reais")]:
        esperado = total_reais(fonte, "valor_total")
        if esperado is None and existe_faturamento:
            esperado = Decimal("0.00")
        conferir_reais(esperado, total_reais(observado, coluna), coluna)
    divergencias = observado.filter(
        (F.col("receita_total_reais").isNull() != F.col("receita_com_leitura_reais").isNull()) |
        (F.col("receita_total_reais").isNull() != F.col("receita_sem_leitura_reais").isNull()) |
        (F.abs(F.col("receita_total_reais") - F.col("receita_com_leitura_reais") - F.col("receita_sem_leitura_reais")) > F.lit(Decimal("0.01"))))
    if divergencias.limit(1).count():
        raise ValueError("Partições financeiras não fecham por recorte")
print("[PASS] Faturamento total e partições reconciliados com fatos independentes da Silver")


### Evidência de execução e limites

As células seguintes executam verificações sobre o lote fixado. Resultados reais e contagens devem permanecer no ambiente autorizado; esta seção não homologa um lote. Zero no recorte não constitui falha por si só.


In [ ]:
# Reconciliação independente de todos os grupos de continuidade.
cadastro_rede = read_silver("unidades_consumidoras").filter(uc_real()).select("id_uc", "alimentador")
assert_unique(cadastro_rede, ["id_uc"], "cadastro de rede")
eventos_rede = (read_silver("ocorrencias_rede").drop("alimentador")
    .filter(F.col("id_uc").isNotNull() & F.col("id_ocorrencia").isNotNull() & F.col("data_abertura").isNotNull())
    .join(cadastro_rede, "id_uc", "inner")
    .withColumn("competencia", F.date_format("data_abertura", "yyyy-MM")))
esperado = (eventos_rede.groupBy("alimentador", "competencia")
    .agg(F.when(F.count(duracao_valida()) == F.count(F.lit(1)),
                F.sum(duracao_valida() / F.lit(60.0))).alias("horas_esperadas"),
         F.count(F.lit(1)).alias("eventos_esperados"))
    .join(cadastro_rede.groupBy("alimentador").agg(F.countDistinct("id_uc").alias("ucs_esperadas")), "alimentador")
    .withColumnRenamed("alimentador", "id_alimentador"))
observado = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec")
comparacao = esperado.join(observado, ["id_alimentador", "competencia"], "full")
if comparacao.filter(
    F.col("ucs_esperadas").isNull() | F.col("total_ucs_alimentador").isNull() |
    (F.col("total_ucs_alimentador") != F.col("ucs_esperadas")) |
    (F.col("duracao_total_interrupcoes_horas").isNull() != F.col("horas_esperadas").isNull()) |
    (F.abs(F.col("duracao_total_interrupcoes_horas") - F.col("horas_esperadas")) > 0.0001) |
    (F.col("frequencia_total_interrupcoes") != F.col("eventos_esperados")) |
    (F.col("DEC").isNull() != F.col("horas_esperadas").isNull()) |
    F.col("frequencia_total_interrupcoes").isNull() | F.col("FEC").isNull() |
    (F.abs(F.col("DEC") - F.col("horas_esperadas") / F.col("ucs_esperadas")) > 0.0001) |
    (F.abs(F.col("FEC") - F.col("eventos_esperados") / F.col("ucs_esperadas")) > 0.0001)
).limit(1).count():
    raise ValueError("Reconciliação DEC/FEC divergente da Silver fixada")
print("[PASS] Grupos DEC/FEC reconciliados com a Silver")


### Evidência de execução e limites

As células seguintes executam verificações sobre o lote fixado. Resultados reais e contagens devem permanecer no ambiente autorizado; esta seção não homologa um lote. Zero no recorte não constitui falha por si só.


In [ ]:
# 7.5 TESTES COMPLEMENTARES: NULOS, TEMPORAL E VISÃO 360 (INCLUINDO faturas_uc, medidor_uc, atendimentos_uc E ocorrencias_uc)

from pyspark.sql import functions as F

print("--- RUNNING SANITY CHECKS ---")

schema_silver_ref = SCHEMA_SILVER_DADOS if 'SCHEMA_SILVER_DADOS' in locals() else (
    SCHEMA_SILVER if 'SCHEMA_SILVER' in locals() else "silver_publicada"
)

# Carregamento das tabelas Gold e Silver para auditoria
gold_receita = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita")
gold_faturas = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc")
gold_medidor = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.medidor_uc")
gold_atendimentos = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.atendimentos_uc")
gold_ocorrencias = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.ocorrencias_uc")
silver_faturas = read_silver("faturas")

# 3.1 Teste de Nulos ou Métricas Inválidas (diagnostico_receita + faturas_uc + medidor_uc + atendimentos_uc + ocorrencias_uc)
invalidos_diagnostico = gold_receita.filter(
    ((F.col("soma_faturamentos_mensais") > 0) & F.col("receita_total_reais").isNull()) |
    ((F.col("qtd_leituras_validas") > 0) & F.col("consumo_total_kwh").isNull()) |
    (F.col("receita_total_reais") < 0) |
    (F.col("consumo_total_kwh") < 0)
).count()

invalidos_faturas_uc = gold_faturas.filter(
    F.col("id_fatura").isNull() |
    F.col("id_uc").isNull() |
    F.col("mes_referencia").isNull() |
    F.col("valor_total").isNull() |
    (F.col("valor_total") < 0)
).count()

invalidos_medidor_uc = gold_medidor.filter(
    F.col("id_medidor").isNull() |
    F.col("id_uc").isNull()
).count()

invalidos_atendimentos_uc = gold_atendimentos.filter(
    F.col("protocolo").isNull() |
    F.col("id_uc").isNull()
).count()

invalidos_ocorrencias_uc = gold_ocorrencias.filter(
    F.col("id_ocorrencia").isNull() |
    F.col("id_uc").isNull()
).count()

# 3.2 Datas históricas sem flag são preservadas; vencimento futuro é legítimo.
# O corte de 1950 proposto para data_ligacao não é uma regra para faturas.
datas_invalidas_faturas = gold_faturas.filter(fatura_datas_invalidas()).count()

# 3.3 Trava Sanitária Fail-Fast em medidor_uc (cronologia e sentinela sinalizada pela Silver)
datas_invalidas_medidor = gold_medidor.filter(
    # Datas Sentinelas ou Impossíveis na Instalação
    (F.col("data_instalacao") >= "9999-01-01") |
    (F.col("data_instalacao") > data_referencia()) |
    
    # Datas Sentinelas ou de Equipamentos Ativos na Retirada (cronologia e sentinela sinalizada pela Silver)
    ((F.col("data_retirada").isNotNull()) & (
        (F.col("data_retirada") >= "9999-01-01") | 
        (F.col("data_retirada") > data_referencia())
    ))
).count()

# 3.4 Teste de Saneamento Temporal e de Notas em atendimentos_uc
atendimentos_nao_sanados = gold_atendimentos.filter(
    ((F.col("_abertura_futura") == True) & (F.col("data_abertura").isNotNull())) |
    ((F.col("_fechamento_futura") == True) & (F.col("data_fechamento").isNotNull())) |
    ((F.col("_nota_invalida") == True) & (F.col("nota_satisfacao").isNotNull())) |
    ((F.col("nota_satisfacao").isNotNull()) & (F.col("nota_satisfacao") < 0))
).count()

# 3.5 Teste de Saneamento Temporal e de Duração em ocorrencias_uc (Direto na tabela Gold)
# Valida se restou alguma data futura ou duração negativa após o saneamento do ETL
ocorrencias_nao_sanadas = gold_ocorrencias.filter(
    ((F.col("data_abertura").isNotNull()) & (F.col("data_abertura") > instante_referencia())) |
    ((F.col("data_encerramento").isNotNull()) & (F.col("data_encerramento") > instante_referencia())) |
    ((F.col("duracao_min").isNotNull()) & (F.col("duracao_min") < 0))
).count()

total_invalidos = (
    invalidos_diagnostico + 
    invalidos_faturas_uc + 
    invalidos_medidor_uc + 
    invalidos_atendimentos_uc + 
    invalidos_ocorrencias_uc +
    datas_invalidas_faturas + 
    datas_invalidas_medidor + 
    atendimentos_nao_sanados +
    ocorrencias_nao_sanadas
)

print(f"[TESTE 1.1] Métricas nulas ou inválidas no Diagnóstico: {invalidos_diagnostico} (Esperado: 0)")
print(f"[TESTE 1.2] Métricas nulas ou inválidas em faturas_uc: {invalidos_faturas_uc} (Esperado: 0)")
print(f"[TESTE 1.3] Chaves nulas em medidor_uc: {invalidos_medidor_uc} (Esperado: 0)")
print(f"[TESTE 1.4] Chaves nulas em atendimentos_uc: {invalidos_atendimentos_uc} (Esperado: 0)")
print(f"[TESTE 1.5] Chaves nulas em ocorrencias_uc: {invalidos_ocorrencias_uc} (Esperado: 0)")
print(f"[TESTE 1.6] Datas sentinelas/futuras não tratadas em faturas_uc: {datas_invalidas_faturas} (Esperado: 0)")
print(f"[TESTE 1.7] Datas sentinelas/futuras não tratadas em medidor_uc: {datas_invalidas_medidor} (Esperado: 0)")
print(f"[TESTE 1.8] Saneamento temporal/notas falho em atendimentos_uc: {atendimentos_nao_sanados} (Esperado: 0)")
print(f"[TESTE 1.9] Saneamento temporal/duração falho em ocorrencias_uc: {ocorrencias_nao_sanadas} (Esperado: 0)")

# 3.6 Cobertura Temporal de Competências / Meses de Referência
col_silver = "competencia" if "competencia" in silver_faturas.columns else "mes_referencia"
col_gold_rec = "competencia" if "competencia" in gold_receita.columns else "mes_referencia"

comp_silver = set(r[col_silver] for r in silver_faturas.select(col_silver).distinct().collect() if r[col_silver])
comp_gold_rec = set(r[col_gold_rec] for r in gold_receita.select(col_gold_rec).distinct().collect() if r[col_gold_rec])
comp_gold_fat = set(r["mes_referencia"] for r in gold_faturas.select("mes_referencia").distinct().collect() if r["mes_referencia"])

diff_meses_diagnostico = comp_silver - comp_gold_rec
diff_meses_faturas_uc = comp_silver - comp_gold_fat

print(f"[TESTE 2.1] Competências da Silver não representadas no Diagnóstico: {diff_meses_diagnostico}")
print(f"[TESTE 2.2] Meses de referência ausentes em faturas_uc: {diff_meses_faturas_uc} (Esperado: set())")

# 3.7 Cobertura de UCs na Visão 360 (Validação Estrita na Tabela Delta Persistida)
df_v360_prep = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc").select("id_uc").distinct()

try:
    total_v360 = df_v360_prep.count()
    print(f"[TESTE 3] Total de UCs únicas na Visão 360 (Tabela Persistida): {total_v360}")
except Exception as e:
    raise RuntimeError(
        f"FALHA CRÍTICA: Não foi possível ler a tabela 'visao_360_uc' diretamente do catálogo Gold. "
        f"Verifique se a persistência física ocorreu corretamente. Erro: {str(e)}"
    )

# INTERRUPÇÃO EXPLÍCITA EM CASO DE FALHA NO PIPELINE (FAIL-FAST)
total_divergencias = total_invalidos + len(diff_meses_faturas_uc)

if total_divergencias > 0:
    raise ValueError(
        f"FALHA NO PIPELINE DE DADOS: Foram encontradas {total_divergencias} divergências nos testes complementares "
        f"({total_invalidos} registros inválidos/nulos/datas não tratadas e {len(diff_meses_faturas_uc)} meses ausentes em faturas_uc). "
        f"O Job foi interrompido para evitar a publicação de dados inconsistentes na Camada Gold."
    )

print("\n Verificações complementares passaram; o aceite depende dos controles e da reconciliação do lote final.")


## 8. Linhagem e Registro de Execução (`gold.controle_gold`)

**Objetivo:** Persistir os metadados de execução da camada Gold, assegurando a rastreabilidade completa e o vínculo estrito com o `run_id` e lote da camada Silver de origem.

---

### Diretrizes de Governança e Operação do Job:

1. **Rastreabilidade Histórica (`append`):**
   * A tabela `gold.controle_gold` acumula o histórico de execuções via `append`, registrando a chave `run_id_silver`, lote de origem, status de processamento, lista de tabelas geradas e o ambiente de execução (`prod` / `dev`).

2. **Diferenciação de Registros (Desenvolvimento vs. Produção):**
   * Execuções manuais do notebook em ambiente de desenvolvimento (`dev`) registram eventos locais para rastreio de testes.
   * **Registro Oficial de Auditoria:** Apenas as execuções orquestradas e automatizadas pelo **Job/Workflow oficial em ambiente de produção (`prod`)** são consideradas carimbos válidos de publicação e homologação dos dados para consumo.

3. **Status de Homologação:**
   * Grava o estado `"CONCLUIDO"` indicando o processamento e a aprovação bem-sucedida de todas as **treze projeções analíticas e um cadastro restrito Gold** na suíte de testes de qualidade.

4. **Tabelas Registradas no Lote (11 Tabelas):**
   * `diagnostico_receita`, `indicadores_dec_fec`, `fila_inspecao`, `visao_360_uc`, `faturas_uc`, `medidor_uc`, `atendimentos_uc`, `ocorrencias_uc`, `consumo_mensal_uc`, `consumo_receita_bairro` e `variacao_consumo_uc`.

### Limite da publicação
Os testes são executados em schema de preparo por execução. Ambientes dev/test promovem apenas para gold_dev/gold_test, sem sobrescrever gold de produção. Só candidatos aprovados são promovidos. Falhas capturadas durante a promoção tentam restaurar as versões Delta anteriores. Não existe transação atômica entre treze projeções analíticas e um cadastro restrito: impedir Jobs concorrentes e aguardar sucesso antes de exportar/consumir. Interrupções abruptas e falhas de recuperação exigem intervenção operacional.


In [0]:
# REGISTRO DE EXECUÇÃO E LINHAGEM DA CAMADA GOLD 

import datetime
import json as _gold_json
from uuid import uuid4
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

tb_controle_gold = f"{CATALOG}.{SCHEMA_GOLD_PUBLICADO}.controle_gold"

# Não usar fallback silencioso para ambiente ou metadados de linhagem.
if AMBIENTE not in {"dev", "test", "prod"}:
    raise RuntimeError(f"AMBIENTE não configurado corretamente: {AMBIENTE!r}")

if "run_id_silver" not in globals() or not run_id_silver:
    raise RuntimeError("run_id_silver ausente; registro Gold interrompido.")

if "lote_bronze_silver" not in globals() or not lote_bronze_silver:
    raise RuntimeError("lote_bronze_silver ausente; registro Gold interrompido.")

if "silver_lineage_status" not in globals() or not silver_lineage_status:
    raise RuntimeError(
        "silver_lineage_status ausente; registro Gold interrompido."
    )

# Em produção, só registrar conclusão se a Silver foi validada fisicamente.
if AMBIENTE == "prod":
    if silver_lineage_status != "CONTROLE_FISICO":
        raise RuntimeError(
            "Produção exige linhagem física da Silver. "
            f"Status recebido: {silver_lineage_status}"
        )

    if "status_exec" not in globals() or str(status_exec).strip().upper() != "CONCLUIDO":
        raise RuntimeError(
            "Produção exige status Silver CONCLUIDO."
        )

# Inventário analítico separado do cadastro restrito, sob a mesma execução Gold.
tabelas_geradas = [
    "diagnostico_receita",
    "indicadores_dec_fec",
    "fila_inspecao",
    "visao_360_uc",
    "faturas_uc",
    "medidor_uc",  
    "atendimentos_uc", 
    "ocorrencias_uc",          
    "consumo_mensal_uc",
    "consumo_receita_bairro",
    "variacao_consumo_uc", "gd_uc", "tratativas_uc", "cobertura_dominios", "decomposicao_receita"
]
tabelas_restritas_geradas = ["cadastro_titular_uc"]

status_silver_val = (
    str(status_exec).strip().upper()
    if "status_exec" in globals() and status_exec is not None
    else "NAO_VALIDADO"
)

run_id_gold = (
    f"GOLD_{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_"
    f"{uuid4().hex[:8]}"
)

preco_volume_referencia = spark.range(1).select(F.date_format(
    instante_referencia(), "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX").alias("referencia")).first()["referencia"]
validar_configuracao_preco_volume(preco_volume_configuracao)
dados_controle = [(
    run_id_gold,
    str(run_id_silver),
    str(lote_bronze_silver),
    status_silver_val,
    str(silver_lineage_status),
    "CONCLUIDO",
    ", ".join(tabelas_geradas),
    ", ".join(tabelas_restritas_geradas),
    AMBIENTE,
    _gold_json.dumps(preco_volume_configuracao, sort_keys=True),
    preco_volume_referencia,
    _gold_json.dumps(LIMITACOES_MEDIDAS_GOLD, sort_keys=True),
    datetime.datetime.now(datetime.timezone.utc)
)]

schema_controle = StructType([
    StructField("run_id_gold", StringType(), False),
    StructField("run_id_silver", StringType(), False),
    StructField("lote_bronze_silver", StringType(), False),
    StructField("status_silver", StringType(), False),
    StructField("silver_lineage_status", StringType(), False),
    StructField("status_gold", StringType(), False),
    StructField("tabelas_geradas", StringType(), False),
    StructField("tabelas_restritas_geradas", StringType(), False),
    StructField("ambiente", StringType(), False),
    StructField("preco_volume_configuracao_json", StringType(), False),
    StructField("preco_volume_referencia", StringType(), False),
    StructField("limitacoes_medidas_json", StringType(), False),
    StructField("processado_em", TimestampType(), False)
])

df_novo_registro = spark.createDataFrame(
    dados_controle,
    schema=schema_controle
)

def registrar_controle():
    try:
        if spark.catalog.tableExists(tb_controle_gold):
            (
                df_novo_registro.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(tb_controle_gold)
            )
        else:
            (
                df_novo_registro.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(tb_controle_gold)
            )
    except Exception:
        # Não restaurar dados se o controle desta execução já foi efetivamente confirmado.
        if spark.catalog.tableExists(tb_controle_gold):
            confirmado = spark.table(tb_controle_gold).filter(F.col("run_id_gold") == run_id_gold).count()
            if confirmado == 1:
                return
        raise

tabelas_ausentes = [
    origem_gold(nome)
    for nome in tabelas_geradas + tabelas_restritas_geradas
    if not spark.catalog.tableExists(origem_gold(nome))
]

if tabelas_ausentes:
    raise RuntimeError(
        "Não foi possível registrar a Gold como concluída. "
        f"Tabelas ausentes: {tabelas_ausentes}"
    )

promover_tabelas(tabelas_geradas + tabelas_restritas_geradas, registrar_controle, tb_controle_gold)

print(
    f"[PASS] Execução Gold registrada em {tb_controle_gold}: "
    f"run_id_gold={run_id_gold}, ambiente={AMBIENTE}, "
    f"status_silver={status_silver_val}, "
    f"linhagem={silver_lineage_status}."
)


## Revisão de semântica do dicionário — 28/10/2026

A média histórica usa leituras não nulas; variações são calculadas antes do arredondamento a duas casas e a flag usa variação bruta < −30%. DEC/FEC conta eventos por UC afetada e publica somente grupos com ocorrência. As categorias e a ausência de casos em ALTA - CONSUMO ZERO são observações de lote. Consulte docs/dicionario_gold.md e o registro de progresso do PRD. Esta revisão preserva todas as células de código e não reproduz a bateria externa de 122 verificações.

### Verificação local das correções — 30/10/2026

- Alterações restritas a este notebook; dicionário e exportador serão alinhados separadamente.
- Sintaxe: 31 células Python válidas.
- Spark local com fontes sintéticas e persistência substituída por armazenamento em memória: 29 células computacionais executadas, 11 candidatos gerados e 14 casos de borda aprovados (vencimento futuro, pagamento inconclusivo, sentinelas, notas, duração ausente, zero legítimo, duplicidade, evento sem UC e universo elegível).
- Coordenação da promoção: 6 cenários aprovados com transporte controlado, incluindo falha de gravação, falha do controle, recuperação e mudança da origem. Isso não comprova a execução real de comandos Delta/Unity Catalog.
- Os roteiros sintéticos temporários foram descartados após o uso, conforme solicitado. Nenhum lote real foi acessado nesses testes.
- Pendente: executar o Job completo no ambiente autorizado, validar permissões/histórico Delta, promoção/RESTORE e reconciliação do lote real. Os consumidores devem aguardar a conclusão; não há atomicidade entre tabelas.


### Complemento da fatia Dados — 30/10/2026

A leitura foi alinhada às vistas `silver_publicada`, com materialização e checagem de identidade antes/depois. Sete cenários locais de fonte passaram. No exportador candidato v2, oito cenários passaram com codec Spark e transporte Delta/arquivos controlado; quatro cenários adicionais validaram a publicação do manifesto sem arquivo final parcial. Esses testes usam apenas dados inventados e não comprovam o transporte UC/Delta real. Os roteiros temporários foram removidos; detalhes e limites estão no Registro de progresso do PRD. Migração/carga v2 e jornada dos consumidores permanecem pendentes.
